# Sierra Leone Disaster Risk Reduction & Preparedness
## Quantitative Analysis

This notebook develops a quantitative evidence base for Sierra Leone's disaster risk reduction and preparedness context.

The analysis combines country-level socioeconomic, governance, health, disaster-impact and INFORM risk data. It uses descriptive analysis, transformation, correlation analysis, regression, diagnostics, sensitivity analysis, international benchmarking, structural similarity methods and machine-learning techniques to support the wider S-DRIF decision-support framework.

> **Analytical note:** This is observational country-level analysis. Regression and correlation results describe associations and should not be interpreted as causal effects without an appropriate causal design.


In [ ]:
import sys
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels
import sklearn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Statsmodels:", statsmodels.__version__)
print("Scikit learn:", sklearn.__version__)

## 1. Environment & Project Configuration

The notebook uses a portable project structure so it can be run from GitHub or another machine without hard-coded personal file paths.

Expected local structure:

```text
project/
├── data/
│   ├── V2 DISASTER PREPAREDNESS DATA.xlsx
│   └── INFORM_Risk_2026_v072.xlsx
├── outputs/
└── analysis/
    └── Sierra_Leone_DRR_Quantitative_Analysis.ipynb
```

The source workbooks are kept outside the public repository unless their redistribution rights are confirmed.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_DATA_FILE = DATA_DIR / "V2 DISASTER PREPAREDNESS DATA.xlsx"
INFORM_FILE = DATA_DIR / "INFORM_Risk_2026_v072.xlsx"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

if not MASTER_DATA_FILE.exists():
    print(
        f"Master dataset not found. Place '{MASTER_DATA_FILE.name}' in the data/ folder before running the analysis."
    )

if not INFORM_FILE.exists():
    print(
        f"INFORM workbook not found. Place '{INFORM_FILE.name}' in the data/ folder before running the INFORM sections."
    )


In [ ]:
# Load the master country-level dataset
file = MASTER_DATA_FILE

df = pd.read_excel(
    file,
    sheet_name="AFRICA MASTER SHEET"
)

df.head()


In [ ]:
df_raw = df.copy()
df = df_raw.copy()

In [ ]:
df.index = range(1, len(df) + 1)

df.head()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.columns.tolist())
print(df.info())
print(df.head())

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
    .str.upper()
)

print(df.columns.tolist())

In [ ]:
print("Duplicate ISO3:", df["ISO3"].duplicated().sum())
print("Duplicate country names:", df["COUNTRY_NAME"].duplicated().sum())

df[df["ISO3"].duplicated(keep=False)].sort_values("ISO3")


In [ ]:
print(df["REGION"].value_counts(dropna=False))
print("Countries:", df["COUNTRY_NAME"].nunique())

df[["ISO3", "COUNTRY_NAME", "REGION"]].sort_values("COUNTRY_NAME")

In [ ]:
numeric_cols = [
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "WGI_SCORE",
    "GHSI",
    "INFORM_RISK_SCORE",
    "GOV_HEALTH_EXPENDITURE",
    "PRIVATE_HEALTH_EXPENDITURE",
    "GINI",
    "DISASTER_COUNT",
    "DEATHS_FROM_DISASTER_COUNT",
    "TOTAL_PEOPLE_AFFECTED_BY_DISASTER"
]

print(df[numeric_cols].dtypes)

for col in numeric_cols:
    print(col, "negative values:", (df[col] < 0).sum())


In [ ]:
missing = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Percent_Missing": df.isna().mean().mul(100).round(2)
}).sort_values("Percent_Missing", ascending=False)

In [ ]:
desc = df[numeric_cols].describe().T
desc["missing"] = df[numeric_cols].isna().sum()
desc

## 2. Data Preparation & Variable Transformation

The first analytical stage establishes the country-level dataset, audits its structure and missingness, and creates log-transformed versions of positively skewed variables for subsequent modelling.

Original-scale variables are retained so descriptive results remain interpretable.


In [ ]:
# Create a separate dataframe for analysis
df_analysis = df.copy()

# Log transformations for positively skewed variables
df_analysis["LOG_GDP"] = np.log(df_analysis["GDP_PER_CAPITA"])
df_analysis["LOG_POPULATION"] = np.log(df_analysis["TOTAL_POPULATION"])
df_analysis["LOG_GOV_HEALTH"] = np.log(df_analysis["GOV_HEALTH_EXPENDITURE"])
df_analysis["LOG_PRIVATE_HEALTH"] = np.log(df_analysis["PRIVATE_HEALTH_EXPENDITURE"])

# log1p allows genuine zero values
df_analysis["LOG_DISASTER_COUNT"] = np.log1p(df_analysis["DISASTER_COUNT"])
df_analysis["LOG_DISASTER_DEATHS"] = np.log1p(df_analysis["DEATHS_FROM_DISASTER_COUNT"])
df_analysis["LOG_PEOPLE_AFFECTED"] = np.log1p(
    df_analysis["TOTAL_PEOPLE_AFFECTED_BY_DISASTER"]
)

In [ ]:
df_analysis[
    [
        "GDP_PER_CAPITA",
        "LOG_GDP",
        "TOTAL_POPULATION",
        "LOG_POPULATION",
        "GOV_HEALTH_EXPENDITURE",
        "LOG_GOV_HEALTH",
        "PRIVATE_HEALTH_EXPENDITURE",
        "LOG_PRIVATE_HEALTH",
        "DISASTER_COUNT",
        "LOG_DISASTER_COUNT",
        "DEATHS_FROM_DISASTER_COUNT",
        "LOG_DISASTER_DEATHS",
        "TOTAL_PEOPLE_AFFECTED_BY_DISASTER",
        "LOG_PEOPLE_AFFECTED"
    ]
].describe().T

In [ ]:
original_cols = [
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "GOV_HEALTH_EXPENDITURE",
    "PRIVATE_HEALTH_EXPENDITURE",
    "DISASTER_COUNT",
    "DEATHS_FROM_DISASTER_COUNT",
    "TOTAL_PEOPLE_AFFECTED_BY_DISASTER"
]

log_cols = [
    "LOG_GDP",
    "LOG_POPULATION",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "LOG_DISASTER_COUNT",
    "LOG_DISASTER_DEATHS",
    "LOG_PEOPLE_AFFECTED"
]

skew_comparison = pd.DataFrame({
    "Original_Skewness": df_analysis[original_cols].skew().values,
    "Log_Skewness": df_analysis[log_cols].skew().values
}, index=original_cols)

skew_comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ============================================================
# VARIABLES
# ============================================================

pairs = [
    ("GDP_PER_CAPITA", "LOG_GDP"),
    ("TOTAL_POPULATION", "LOG_POPULATION"),
    ("GOV_HEALTH_EXPENDITURE", "LOG_GOV_HEALTH"),
    ("PRIVATE_HEALTH_EXPENDITURE", "LOG_PRIVATE_HEALTH"),
    ("DISASTER_COUNT", "LOG_DISASTER_COUNT"),
    ("DEATHS_FROM_DISASTER_COUNT", "LOG_DISASTER_DEATHS"),
    ("TOTAL_PEOPLE_AFFECTED_BY_DISASTER", "LOG_PEOPLE_AFFECTED")
]

# More readable display names
labels = {
    "GDP_PER_CAPITA": "GDP per Capita",
    "LOG_GDP": "Log GDP per Capita",
    
    "TOTAL_POPULATION": "Total Population",
    "LOG_POPULATION": "Log Total Population",
    
    "GOV_HEALTH_EXPENDITURE": "Government Health Expenditure",
    "LOG_GOV_HEALTH": "Log Government Health Expenditure",
    
    "PRIVATE_HEALTH_EXPENDITURE": "Private Health Expenditure",
    "LOG_PRIVATE_HEALTH": "Log Private Health Expenditure",
    
    "DISASTER_COUNT": "Disaster Count",
    "LOG_DISASTER_COUNT": "Log Disaster Count",
    
    "DEATHS_FROM_DISASTER_COUNT": "Disaster Deaths",
    "LOG_DISASTER_DEATHS": "Log Disaster Deaths",
    
    "TOTAL_PEOPLE_AFFECTED_BY_DISASTER": "People Affected by Disasters",
    "LOG_PEOPLE_AFFECTED": "Log People Affected"
}

# ============================================================
# CREATE FIGURE
# ============================================================

fig, axes = plt.subplots(
    nrows=7,
    ncols=2,
    figsize=(15, 25)
)

fig.patch.set_facecolor("white")

# ============================================================
# PLOT EACH VARIABLE
# ============================================================

for i, (original, transformed) in enumerate(pairs):

    # -----------------------------
    # ORIGINAL DISTRIBUTION
    # -----------------------------

    ax_original = axes[i, 0]

    sns.histplot(
        df_analysis[original].dropna(),
        kde=True,
        bins=12,
        ax=ax_original
    )

    ax_original.set_title(
        f"{labels[original]}",
        fontsize=12,
        fontweight="bold",
        pad=10
    )

    ax_original.set_xlabel(
        "Original scale",
        fontsize=10
    )

    ax_original.set_ylabel(
        "Number of countries",
        fontsize=10
    )

    ax_original.spines["top"].set_visible(False)
    ax_original.spines["right"].set_visible(False)

    # -----------------------------
    # TRANSFORMED DISTRIBUTION
    # -----------------------------

    ax_log = axes[i, 1]

    sns.histplot(
        df_analysis[transformed].dropna(),
        kde=True,
        bins=12,
        ax=ax_log
    )

    ax_log.set_title(
        f"{labels[transformed]}",
        fontsize=12,
        fontweight="bold",
        pad=10
    )

    ax_log.set_xlabel(
        "Log transformed scale",
        fontsize=10
    )

    ax_log.set_ylabel(
        "Number of countries",
        fontsize=10
    )

    ax_log.spines["top"].set_visible(False)
    ax_log.spines["right"].set_visible(False)


# ============================================================
# MAIN TITLE
# ============================================================

fig.suptitle(
    "Distributional Assessment Before and After Log Transformation",
    fontsize=20,
    fontweight="bold",
    y=0.995
)

# Subtitle
fig.text(
    0.5,
    0.988,
    "African country level analysis | n = 54",
    ha="center",
    fontsize=11
)

# ============================================================
# SECTION LABELS
# ============================================================

fig.text(
    0.25,
    0.978,
    "ORIGINAL VARIABLES",
    ha="center",
    fontsize=11,
    fontweight="bold"
)

fig.text(
    0.75,
    0.978,
    "LOG TRANSFORMED VARIABLES",
    ha="center",
    fontsize=11,
    fontweight="bold"
)

# ============================================================
# SPACING
# ============================================================

plt.subplots_adjust(
    top=0.965,
    bottom=0.025,
    left=0.08,
    right=0.97,
    hspace=0.65,
    wspace=0.25
)

# ============================================================
# SAVE HIGH RESOLUTION
# ============================================================

output_file = OUTPUT_DIR / "Variable_Distributions_Before_After_Log_Transformation.png"

plt.savefig(
    output_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved to:\n{output_file}")

In [ ]:
print(np.isfinite(
    df_analysis.select_dtypes(include=np.number)
).all().all())

print(df_analysis.isna().sum().sort_values(ascending=False))


In [ ]:
def iqr_flags(data, column):
    x = data[column].dropna()
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return data[(data[column] < lower) | (data[column] > upper)][
        ["ISO3", "COUNTRY_NAME", column]
    ]

for col in ["LOG_GDP", "LOG_GOV_HEALTH", "LOG_PRIVATE_HEALTH",
            "LOG_DISASTER_COUNT", "LOG_DISASTER_DEATHS",
            "LOG_PEOPLE_AFFECTED"]:
    print("\n", col)
    display(iqr_flags(df_analysis, col))


In [ ]:
x = df_analysis["LOG_PEOPLE_AFFECTED"].dropna()

q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower)
print("Upper bound:", upper)

In [ ]:
df_analysis[
    df_analysis["LOG_PEOPLE_AFFECTED"].isin(
        iqr_flags(df_analysis, "LOG_PEOPLE_AFFECTED")["LOG_PEOPLE_AFFECTED"]
    )
][[
    "ISO3",
    "COUNTRY_NAME",
    "TOTAL_PEOPLE_AFFECTED_BY_DISASTER",
    "LOG_PEOPLE_AFFECTED"
]]

In [ ]:
# ============================================================
# 18. STEP 15: AFRICAN DESCRIPTIVE BASELINE
# ============================================================

# Original scale variables retained for interpretation
original_summary_cols = [
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "WGI_SCORE",
    "GHSI",
    "INFORM_RISK_SCORE",
    "GOV_HEALTH_EXPENDITURE",
    "PRIVATE_HEALTH_EXPENDITURE",
    "GINI",
    "DISASTER_COUNT",
    "DEATHS_FROM_DISASTER_COUNT",
    "TOTAL_PEOPLE_AFFECTED_BY_DISASTER"
]

# Transformed variables used where skewness required it
analysis_summary_cols = [
    "LOG_GDP",
    "LOG_POPULATION",
    "WGI_SCORE",
    "GHSI",
    "INFORM_RISK_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "GINI",
    "LOG_DISASTER_COUNT",
    "LOG_DISASTER_DEATHS",
    "LOG_PEOPLE_AFFECTED"
]

# Descriptive statistics on the original scale
africa_summary = (
    df_analysis[original_summary_cols]
    .describe()
    .T
)

africa_summary

In [ ]:
# ============================================================
# AFRICAN MEAN VS MEDIAN
# ============================================================

africa_mean = (
    df_analysis[original_summary_cols]
    .mean()
)

africa_median = (
    df_analysis[original_summary_cols]
    .median()
)

africa_mean_median = pd.DataFrame({
    "Mean": africa_mean,
    "Median": africa_median
})

africa_mean_median["Difference"] = (
    africa_mean_median["Mean"]
    - africa_mean_median["Median"]
)

africa_mean_median["Mean_Median_Ratio"] = (
    africa_mean_median["Mean"]
    / africa_mean_median["Median"]
)

africa_mean_median.round(2)

In [ ]:
# ============================================================
# DESCRIPTIVE STATISTICS FOR ANALYTICAL VARIABLES
# ============================================================

analysis_summary = (
    df_analysis[analysis_summary_cols]
    .describe()
    .T
)

analysis_summary["Missing"] = (
    df_analysis[analysis_summary_cols]
    .isna()
    .sum()
)

analysis_summary

In [ ]:
# ============================================================
# 19. STEP 16: SIERRA LEONE BASELINE
# ============================================================

sl = df_analysis[
    df_analysis["COUNTRY_NAME"].eq("Sierra Leone")
].copy()

sl

In [ ]:
relationships = [
    ("WGI_SCORE", "GHSI", "Government Effectiveness vs Preparedness"),
    ("LOG_GDP", "GHSI", "Log GDP per Capita vs Preparedness"),
    ("LOG_GOV_HEALTH", "GHSI", "Government Health Expenditure vs Preparedness"),
    ("LOG_PRIVATE_HEALTH", "GHSI", "Private Health Expenditure vs Preparedness"),
    ("INFORM_RISK_SCORE", "GHSI", "INFORM Risk vs Preparedness")
]

# Sierra Leone
sl = df_analysis[
    df_analysis["COUNTRY_NAME"].eq("Sierra Leone")
].iloc[0]

for x, y, title in relationships:

    fig, ax = plt.subplots(figsize=(9, 6))

    # Main regression plot
    sns.regplot(
        data=df_analysis,
        x=x,
        y=y,
        ax=ax,
        scatter_kws={
            "s": 55,
            "alpha": 0.65
        },
        line_kws={
            "linewidth": 2
        }
    )

    # Sierra Leone marker
    ax.scatter(
        sl[x],
        sl[y],
        s=180,
        color="green",
        marker="o",
        edgecolor="white",
        linewidth=2,
        zorder=10,
        label="Sierra Leone"
    )

    # Sierra Leone label
    ax.annotate(
        "Sierra Leone",
        xy=(sl[x], sl[y]),
        xytext=(10, 10),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color="green",
        zorder=11
    )

    # Titles
    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=15
    )

    # Axis labels
    ax.set_xlabel(
        x.replace("_", " ").title(),
        fontsize=11
    )

    ax.set_ylabel(
        y.replace("_", " ").title(),
        fontsize=11
    )

    # Clean appearance
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        alpha=0.2,
        linewidth=0.8
    )

    ax.legend(
        frameon=False,
        loc="best"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
output_folder = str(OUTPUT_DIR)

## 3. Correlation Analysis

Pearson and Spearman correlations assess the direction and strength of pairwise associations among the main analytical variables.

Using both measures provides a robustness check because Spearman correlation is rank-based and less sensitive to distributional assumptions.


In [ ]:
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ============================================================
# CORRELATION ANALYSIS
# ============================================================

corr_cols = [
    "LOG_GDP",
    "LOG_POPULATION",
    "WGI_SCORE",
    "GHSI",
    "INFORM_RISK_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "GINI",
    "LOG_DISASTER_COUNT",
    "LOG_DISASTER_DEATHS",
    "LOG_PEOPLE_AFFECTED"
]

# Pearson correlation
pearson_corr = df_analysis[corr_cols].corr(
    method="pearson"
)

# Spearman rank correlation
spearman_corr = df_analysis[corr_cols].corr(
    method="spearman"
)

# ============================================================
# SAVE CORRELATION TABLES
# ============================================================

output_folder = str(OUTPUT_DIR)

pearson_corr.to_csv(
    os.path.join(
        output_folder,
        "Pearson_Correlation_Matrix.csv"
    )
)

spearman_corr.to_csv(
    os.path.join(
        output_folder,
        "Spearman_Correlation_Matrix.csv"
    )
)

print("Correlation matrices saved.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 7))

sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True
)

plt.title("Pearson Correlation Matrix")

plt.show()

In [ ]:
df_analysis["DEATHS_PER_100K"] = (
    df_analysis["DEATHS_FROM_DISASTER_COUNT"]
    / df_analysis["TOTAL_POPULATION"]
) * 100000

df_analysis["AFFECTED_PER_100K"] = (
    df_analysis["TOTAL_PEOPLE_AFFECTED_BY_DISASTER"]
    / df_analysis["TOTAL_POPULATION"]
) * 100000

In [ ]:
df_analysis[
    [
        "COUNTRY_NAME",
        "DEATHS_PER_100K",
        "AFFECTED_PER_100K"
    ]
].sort_values(
    "AFFECTED_PER_100K",
    ascending=False
)

In [ ]:
df_analysis[
    ["DEATHS_PER_100K", "AFFECTED_PER_100K"]
].describe().T

In [ ]:
df_analysis[
    ["DEATHS_PER_100K", "AFFECTED_PER_100K"]
].skew()

In [ ]:
vif_cols = [
    "LOG_GDP",
    "LOG_POPULATION",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "GINI"
]

In [ ]:
vif_data = df_analysis[vif_cols].dropna().copy()

print("Observations used for VIF:", len(vif_data))
print("Variables:", vif_data.columns.tolist())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

vif_results = pd.DataFrame()

vif_results["Variable"] = vif_data.columns

vif_results["VIF"] = [
    variance_inflation_factor(
        vif_data.values,
        i
    )
    for i in range(vif_data.shape[1])
]

vif_results = vif_results.sort_values(
    "VIF",
    ascending=False
)

display(vif_results.round(2))

In [ ]:
import os

output_folder = str(OUTPUT_DIR)

vif_results.to_csv(
    os.path.join(
        output_folder,
        "VIF_Analysis.csv"
    ),
    index=False
)

print("Saved:", os.path.join(output_folder, "VIF_Analysis.csv"))

In [ ]:
original_vif_cols = [
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "GOV_HEALTH_EXPENDITURE",
    "PRIVATE_HEALTH_EXPENDITURE",
    "GINI"
]

log_vif_cols = [
    "LOG_GDP",
    "LOG_POPULATION",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "GINI"
]

In [ ]:
def calculate_vif(data, columns):

    temp = data[columns].dropna()

    results = pd.DataFrame({
        "Variable": temp.columns,
        "VIF": [
            variance_inflation_factor(
                temp.values,
                i
            )
            for i in range(temp.shape[1])
        ]
    })

    return results.sort_values(
        "VIF",
        ascending=False
    )


original_vif = calculate_vif(
    df_analysis,
    original_vif_cols
)

log_vif = calculate_vif(
    df_analysis,
    log_vif_cols
)

print("ORIGINAL VARIABLES")
display(original_vif.round(2))

print("LOG TRANSFORMED VARIABLES")
display(log_vif.round(2))

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

vif_cols_no_gini = [
    "LOG_GDP",
    "LOG_POPULATION",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH"
]

vif_data_no_gini = (
    df_analysis[vif_cols_no_gini]
    .dropna()
    .copy()
)

vif_results_no_gini = pd.DataFrame({
    "Variable": vif_data_no_gini.columns,
    "VIF": [
        variance_inflation_factor(
            vif_data_no_gini.values,
            i
        )
        for i in range(vif_data_no_gini.shape[1])
    ]
})

display(
    vif_results_no_gini
    .sort_values("VIF", ascending=False)
    .round(2)
)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

candidate_models = {
    "Governance": [
        "WGI_SCORE"
    ],
    
    "Economic": [
        "LOG_GDP"
    ],
    
    "Government_Health": [
        "LOG_GOV_HEALTH"
    ],
    
    "Economic_Governance": [
        "LOG_GDP",
        "WGI_SCORE"
    ],
    
    "Economic_Government_Health": [
        "LOG_GDP",
        "LOG_GOV_HEALTH"
    ],
    
    "Governance_Government_Health": [
        "WGI_SCORE",
        "LOG_GOV_HEALTH"
    ]
}

In [ ]:
for model_name, variables in candidate_models.items():
    
    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)
    
    display(
        calculate_vif(
            df_analysis,
            variables
        )
    )

In [ ]:
for name, variables in candidate_models.items():
    temp = df_analysis[variables].dropna()
    
    print(
        f"{name}: {len(temp)} complete observations | "
        f"Variables: {variables}"
    )

In [ ]:
def calculate_vif(data, variables):
    
    temp = data[variables].dropna().copy()
    
    # Single predictor = no multicollinearity possible
    if len(variables) == 1:
        return pd.DataFrame({
            "Variable": variables,
            "VIF": [1.00],
            "N": [len(temp)]
        })
    
    results = pd.DataFrame({
        "Variable": temp.columns,
        "VIF": [
            variance_inflation_factor(
                temp.values,
                i
            )
            for i in range(temp.shape[1])
        ]
    })
    
    results["N"] = len(temp)
    
    return results.sort_values(
        "VIF",
        ascending=False
    ).round(2)

## 4. Regression Analysis

The regression stage evaluates bivariate and multivariable associations with the Global Health Security Index (GHSI).

Models are followed by influence diagnostics and sensitivity analyses where appropriate. The objective is to assess statistical relationships and robustness, not to establish causality.


In [ ]:
import statsmodels.api as sm

# ==========================================
# MODEL 1: GOVERNMENT EFFECTIVENESS → GHSI
# ==========================================

model_1_data = df_analysis[
    ["GHSI", "WGI_SCORE"]
].dropna().copy()

X = sm.add_constant(
    model_1_data["WGI_SCORE"]
)

y = model_1_data["GHSI"]

model_1 = sm.OLS(
    y,
    X
).fit()

print(model_1.summary())

In [ ]:
model_1_results = pd.DataFrame({
    "Coefficient": model_1.params,
    "Std_Error": model_1.bse,
    "t_value": model_1.tvalues,
    "p_value": model_1.pvalues,
    "CI_Lower": model_1.conf_int()[0],
    "CI_Upper": model_1.conf_int()[1]
})

display(model_1_results.round(4))

In [ ]:
print("R-squared:", round(model_1.rsquared, 4))
print("Adjusted R-squared:", round(model_1.rsquared_adj, 4))
print("Observations:", int(model_1.nobs))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(
    x=model_1.fittedvalues,
    y=model_1.resid,
    s=70,
    ax=ax
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Fitted GHSI")
ax.set_ylabel("Residuals")
ax.set_title(
    "Model 1: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_1.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 1: Q-Q Plot of OLS Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(
    model_1.resid,
    model_1.model.exog
)

bp_results = pd.Series(
    bp_test,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results.round(4))

In [ ]:
influence = model_1.get_influence()

cooks_d = influence.cooks_distance[0]

cook_results = pd.DataFrame({
    "COUNTRY_NAME": model_1_data.index.map(
        df_analysis["COUNTRY_NAME"]
    ),
    "Cooks_Distance": cooks_d
}).sort_values(
    "Cooks_Distance",
    ascending=False
)

display(cook_results.head(10))

In [ ]:
cook_results = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_1_data.index, "COUNTRY_NAME"
    ].values,
    "Cooks_Distance": cooks_d
}).sort_values(
    "Cooks_Distance",
    ascending=False
)

display(cook_results)

In [ ]:
n = int(model_1.nobs)
k = int(model_1.df_model) + 1

cook_threshold = 4 / (n - k - 1)

print("Threshold:", cook_threshold)

In [ ]:
influential_countries = cook_results[
    cook_results["Cooks_Distance"] > cook_threshold
]

display(influential_countries)

In [ ]:
exclude = influential_countries["COUNTRY_NAME"].tolist()

sensitivity_data = model_1_data[
    ~model_1_data.index.isin(
        df_analysis[
            df_analysis["COUNTRY_NAME"].isin(exclude)
        ].index
    )
].copy()

X_sensitivity = sm.add_constant(
    sensitivity_data["WGI_SCORE"]
)

y_sensitivity = sensitivity_data["GHSI"]

model_1_sensitivity = sm.OLS(
    y_sensitivity,
    X_sensitivity
).fit()

print(model_1_sensitivity.summary())

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(
    model_1.resid,
    model_1.model.exog
)

bp_results = pd.Series(
    bp_test,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results.round(4))

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_1.fittedvalues,
    y=model_1.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")
plt.title(
    "Model 1: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as stats

fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_1.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 1: Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
influence = model_1.get_influence()

leverage = influence.hat_matrix_diag

leverage_results = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_1_data.index,
        "COUNTRY_NAME"
    ].values,
    "Leverage": leverage
}).sort_values(
    "Leverage",
    ascending=False
)

display(leverage_results.head(54))

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_1.fittedvalues,
    y=model_1.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")
plt.title(
    "Model 1: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_1.fittedvalues,
    y=model_1.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")
plt.title(
    "Model 1: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as stats

fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_1.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 1: Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
residual_results = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_1_data.index,
        "COUNTRY_NAME"
    ].values,
    "Fitted_GHSI": model_1.fittedvalues.values,
    "Residual": model_1.resid.values,
    "Abs_Residual": abs(model_1.resid.values)
}).sort_values(
    "Abs_Residual",
    ascending=False
)

display(residual_results.head(54))

In [ ]:
# ==========================================
# MODEL 2: GDP PER CAPITA → GHSI
# ==========================================

model_2_data = df_analysis[
    ["GHSI", "LOG_GDP"]
].dropna().copy()

print("Observations:", len(model_2_data))

In [ ]:
X = sm.add_constant(
    model_2_data["LOG_GDP"]
)

y = model_2_data["GHSI"]

model_2 = sm.OLS(
    y,
    X
).fit()

print(model_2.summary())

In [ ]:
# ==========================================
# MODEL 2: COOK'S DISTANCE
# ==========================================

influence_2 = model_2.get_influence()

cooks_d_2 = influence_2.cooks_distance[0]

cook_results_2 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_2_data.index,
        "COUNTRY_NAME"
    ].values,
    "Cooks_Distance": cooks_d_2
}).sort_values(
    "Cooks_Distance",
    ascending=False
)

display(cook_results_2)

In [ ]:
# Identify observations above the Model 2 Cook's distance threshold

n_2 = int(model_2.nobs)
k_2 = int(model_2.df_model) + 1

cook_threshold_2 = 4 / (n_2 - k_2 - 1)

influential_2 = cook_results_2[
    cook_results_2["Cooks_Distance"] > cook_threshold_2
].copy()

print("Cook's distance threshold:", round(cook_threshold_2, 4))
print("Number of flagged observations:", len(influential_2))

display(influential_2)

In [ ]:
exclude_2 = influential_2["COUNTRY_NAME"].tolist()

sensitivity_data_2 = model_2_data[
    ~model_2_data.index.isin(
        df_analysis[
            df_analysis["COUNTRY_NAME"].isin(exclude_2)
        ].index
    )
].copy()

X_sensitivity_2 = sm.add_constant(
    sensitivity_data_2["LOG_GDP"]
)

y_sensitivity_2 = sensitivity_data_2["GHSI"]

model_2_sensitivity = sm.OLS(
    y_sensitivity_2,
    X_sensitivity_2
).fit()

print(model_2_sensitivity.summary())

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test_2 = het_breuschpagan(
    model_2.resid,
    model_2.model.exog
)

bp_results_2 = pd.Series(
    bp_test_2,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results_2.round(4))

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test_2 = het_breuschpagan(
    model_2.resid,
    model_2.model.exog
)

bp_results_2 = pd.Series(
    bp_test_2,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results_2.round(4))

In [ ]:
influence_2 = model_2.get_influence()

leverage_2 = influence_2.hat_matrix_diag

leverage_results_2 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_2_data.index,
        "COUNTRY_NAME"
    ].values,
    "Leverage": leverage_2
}).sort_values(
    "Leverage",
    ascending=False
)

display(leverage_results_2)

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_2.fittedvalues,
    y=model_2.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")

plt.title(
    "Model 2: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_2.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 2: Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# MODEL 3: GOVERNMENT HEALTH EXPENDITURE → GHSI
# ==========================================

model_3_data = df_analysis[
    ["GHSI", "LOG_GOV_HEALTH"]
].dropna().copy()

print("Observations:", len(model_3_data))

X = sm.add_constant(
    model_3_data["LOG_GOV_HEALTH"]
)

y = model_3_data["GHSI"]

model_3 = sm.OLS(
    y,
    X
).fit()

print(model_3.summary())

In [ ]:
# ==========================================
# MODEL 3: COOK'S DISTANCE
# ==========================================

influence_3 = model_3.get_influence()

cooks_d_3 = influence_3.cooks_distance[0]

cook_results_3 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_3_data.index,
        "COUNTRY_NAME"
    ].values,
    "Cooks_Distance": cooks_d_3
}).sort_values(
    "Cooks_Distance",
    ascending=False
)

display(cook_results_3)

In [ ]:
# Identify Model 3 observations exceeding the Cook's distance threshold

n_3 = int(model_3.nobs)
k_3 = int(model_3.df_model) + 1

cook_threshold_3 = 4 / (n_3 - k_3 - 1)

influential_3 = cook_results_3[
    cook_results_3["Cooks_Distance"] > cook_threshold_3
].copy()

print("Cook's distance threshold:", round(cook_threshold_3, 4))
print("Number of flagged observations:", len(influential_3))

display(influential_3)

In [ ]:
exclude_3 = influential_3["COUNTRY_NAME"].tolist()

sensitivity_data_3 = model_3_data[
    ~model_3_data.index.isin(
        df_analysis[
            df_analysis["COUNTRY_NAME"].isin(exclude_3)
        ].index
    )
].copy()

X_sensitivity_3 = sm.add_constant(
    sensitivity_data_3["LOG_GOV_HEALTH"]
)

y_sensitivity_3 = sensitivity_data_3["GHSI"]

model_3_sensitivity = sm.OLS(
    y_sensitivity_3,
    X_sensitivity_3
).fit()

print(model_3_sensitivity.summary())

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test_3 = het_breuschpagan(
    model_3.resid,
    model_3.model.exog
)

bp_results_3 = pd.Series(
    bp_test_3,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results_3.round(4))

In [ ]:
influence_3 = model_3.get_influence()

leverage_3 = influence_3.hat_matrix_diag

leverage_results_3 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_3_data.index,
        "COUNTRY_NAME"
    ].values,
    "Leverage": leverage_3
}).sort_values(
    "Leverage",
    ascending=False
)

display(leverage_results_3)

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_3.fittedvalues,
    y=model_3.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")

plt.title(
    "Model 3: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_3.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 3: Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### Model 4 — Private Health Expenditure → GHSI


In [ ]:
# ==========================================
# MODEL 4: PRIVATE HEALTH EXPENDITURE → GHSI
# ==========================================

model_4_data = df_analysis[
    ["GHSI", "LOG_PRIVATE_HEALTH"]
].dropna().copy()

print("Observations:", len(model_4_data))

X = sm.add_constant(
    model_4_data["LOG_PRIVATE_HEALTH"]
)

y = model_4_data["GHSI"]

model_4 = sm.OLS(
    y,
    X
).fit()

print(model_4.summary())

In [ ]:
# ==========================================
# MODEL 4: COOK'S DISTANCE
# ==========================================

influence_4 = model_4.get_influence()

cooks_d_4 = influence_4.cooks_distance[0]

cook_results_4 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_4_data.index,
        "COUNTRY_NAME"
    ].values,
    "Cooks_Distance": cooks_d_4
}).sort_values(
    "Cooks_Distance",
    ascending=False
)

display(cook_results_4)

In [ ]:
# ==========================================
# MODEL 4: RUN SENSITIVITY REGRESSION
# ==========================================

X_sensitivity_4 = sm.add_constant(
    model_4_sensitivity_data["LOG_PRIVATE_HEALTH"]
)

y_sensitivity_4 = model_4_sensitivity_data["GHSI"]

model_4_sensitivity = sm.OLS(
    y_sensitivity_4,
    X_sensitivity_4
).fit()

print(model_4_sensitivity.summary())

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test_4 = het_breuschpagan(
    model_4.resid,
    model_4.model.exog
)

bp_results_4 = pd.Series(
    bp_test_4,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results_4.round(4))

In [ ]:
influence_4 = model_4.get_influence()

leverage_4 = influence_4.hat_matrix_diag

leverage_results_4 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_4_data.index,
        "COUNTRY_NAME"
    ].values,
    "Leverage": leverage_4
}).sort_values(
    "Leverage",
    ascending=False
)

display(leverage_results_4)

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_4.fittedvalues,
    y=model_4.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")

plt.title(
    "Model 4: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_4.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 4: Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# MODEL 5: INFORM RISK → GHSI
# ==========================================

model_5_data = df_analysis[
    ["GHSI", "INFORM_RISK_SCORE"]
].dropna().copy()

print("Observations:", len(model_5_data))

X = sm.add_constant(
    model_5_data["INFORM_RISK_SCORE"]
)

y = model_5_data["GHSI"]

model_5 = sm.OLS(
    y,
    X
).fit()

print(model_5.summary())

In [ ]:
# ==========================================
# MODEL 5: COOK'S DISTANCE
# ==========================================

influence_5 = model_5.get_influence()

cooks_d_5 = influence_5.cooks_distance[0]

cook_results_5 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_5_data.index,
        "COUNTRY_NAME"
    ].values,
    "Cooks_Distance": cooks_d_5
}).sort_values(
    "Cooks_Distance",
    ascending=False
)

display(cook_results_5)

In [ ]:
# ==========================================
# MODEL 5: SENSITIVITY ANALYSIS
# ==========================================

exclude_5 = [
    "Somalia, Fed. Rep.",
    "Equatorial Guinea",
    "Mauritius",
    "South Africa"
]

model_5_sensitivity_data = df_analysis[
    ~df_analysis["COUNTRY_NAME"].isin(exclude_5)
][
    ["GHSI", "INFORM_RISK_SCORE"]
].dropna().copy()

print("Original Model 5 N:", len(model_5_data))
print("Sensitivity Model 5 N:", len(model_5_sensitivity_data))

print("\nCountries excluded:")
for country in exclude_5:
    print("-", country)

In [ ]:
# ==========================================
# RUN SENSITIVITY OLS
# ==========================================

X_sensitivity_5 = sm.add_constant(
    model_5_sensitivity_data["INFORM_RISK_SCORE"]
)

y_sensitivity_5 = model_5_sensitivity_data["GHSI"]

model_5_sensitivity = sm.OLS(
    y_sensitivity_5,
    X_sensitivity_5
).fit()

print(model_5_sensitivity.summary())

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test_5 = het_breuschpagan(
    model_5.resid,
    model_5.model.exog
)

bp_results_5 = pd.Series(
    bp_test_5,
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

display(bp_results_5.round(4))

In [ ]:
influence_5 = model_5.get_influence()

leverage_5 = influence_5.hat_matrix_diag

leverage_results_5 = pd.DataFrame({
    "COUNTRY_NAME": df_analysis.loc[
        model_5_data.index,
        "COUNTRY_NAME"
    ].values,
    "Leverage": leverage_5
}).sort_values(
    "Leverage",
    ascending=False
)

display(leverage_results_5)

In [ ]:
# ==========================================
# MODEL 5: RESIDUALS VS FITTED
# ==========================================

plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=model_5.fittedvalues,
    y=model_5.resid,
    s=75,
    alpha=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.xlabel("Fitted GHSI")
plt.ylabel("Residuals")

plt.title(
    "Model 5: Residuals vs Fitted Values",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# MODEL 5: Q-Q PLOT
# ==========================================

fig, ax = plt.subplots(figsize=(8, 6))

stats.probplot(
    model_5.resid,
    dist="norm",
    plot=ax
)

ax.set_title(
    "Model 5: Normal Q-Q Plot of Residuals",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### Multivariable Candidate Models


## 5. Multivariable Model Development

Candidate models combine economic, governance, health-expenditure and risk variables. Complete-case sample sizes and multicollinearity are assessed before comparing specifications.


In [ ]:
# ==========================================
# MULTIVARIABLE MODEL: DATASET CHECK
# ==========================================

candidate_vars = [
    "GHSI",
    "LOG_GDP",
    "WGI_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "INFORM_RISK_SCORE",
    "GINI"
]

model_multi_data = df_analysis[
    candidate_vars
].copy()

print("Missing values:")
display(
    model_multi_data.isna().sum()
)

print("\nComplete cases:")
print(
    model_multi_data.dropna().shape
)

In [ ]:
# ==========================================
# COMPARE AVAILABLE SAMPLE SIZES
# ==========================================

model_without_gini_vars = [
    "GHSI",
    "LOG_GDP",
    "WGI_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "INFORM_RISK_SCORE"
]

model_with_gini_vars = model_without_gini_vars + ["GINI"]

n_without_gini = df_analysis[
    model_without_gini_vars
].dropna().shape[0]

n_with_gini = df_analysis[
    model_with_gini_vars
].dropna().shape[0]

print("Complete cases WITHOUT GINI:", n_without_gini)
print("Complete cases WITH GINI:", n_with_gini)
print("Countries lost by including GINI:", n_without_gini - n_with_gini)

In [ ]:
primary_vars = [
    "GHSI",
    "LOG_GDP",
    "WGI_SCORE",
    "LOG_GOV_HEALTH",
    "LOG_PRIVATE_HEALTH",
    "INFORM_RISK_SCORE"
]

primary_data = df_analysis[
    primary_vars
].dropna().copy()

print("Primary model N:", len(primary_data))

In [ ]:
model_A_vars = [
    "GHSI",
    "LOG_GDP",
    "WGI_SCORE"
]

model_A_data = df_analysis[
    model_A_vars
].dropna().copy()

print("Model A N:", len(model_A_data))

In [ ]:
model_B_vars = [
    "GHSI",
    "LOG_GDP",
    "LOG_GOV_HEALTH"
]

model_B_data = df_analysis[
    model_B_vars
].dropna().copy()

print("Model B N:", len(model_B_data))

In [ ]:
model_C_vars = [
    "GHSI",
    "WGI_SCORE",
    "LOG_GOV_HEALTH"
]

model_C_data = df_analysis[
    model_C_vars
].dropna().copy()

print("Model C N:", len(model_C_data))

In [ ]:
# ==========================================
# MULTIVARIABLE CANDIDATE MODELS
# ==========================================

# ------------------------------------------
# MODEL A: GDP + GOVERNANCE
# ------------------------------------------

model_A_data = df_analysis[
    ["GHSI", "LOG_GDP", "WGI_SCORE"]
].dropna().copy()

X_A = sm.add_constant(
    model_A_data[["LOG_GDP", "WGI_SCORE"]]
)

y_A = model_A_data["GHSI"]

model_A = sm.OLS(
    y_A,
    X_A
).fit()

print("=" * 70)
print("MODEL A: LOG GDP + WGI")
print("=" * 70)
print(model_A.summary())


# ------------------------------------------
# MODEL B: GDP + GOVERNMENT HEALTH
# ------------------------------------------

model_B_data = df_analysis[
    ["GHSI", "LOG_GDP", "LOG_GOV_HEALTH"]
].dropna().copy()

X_B = sm.add_constant(
    model_B_data[["LOG_GDP", "LOG_GOV_HEALTH"]]
)

y_B = model_B_data["GHSI"]

model_B = sm.OLS(
    y_B,
    X_B
).fit()

print("=" * 70)
print("MODEL B: LOG GDP + LOG GOVERNMENT HEALTH")
print("=" * 70)
print(model_B.summary())


# ------------------------------------------
# MODEL C: GOVERNANCE + GOVERNMENT HEALTH
# ------------------------------------------

model_C_data = df_analysis[
    ["GHSI", "WGI_SCORE", "LOG_GOV_HEALTH"]
].dropna().copy()

X_C = sm.add_constant(
    model_C_data[["WGI_SCORE", "LOG_GOV_HEALTH"]]
)

y_C = model_C_data["GHSI"]

model_C = sm.OLS(
    y_C,
    X_C
).fit()

print("=" * 70)
print("MODEL C: WGI + LOG GOVERNMENT HEALTH")
print("=" * 70)
print(model_C.summary())

In [ ]:
# ==========================================
# VIF FOR CANDIDATE MULTIVARIABLE MODELS
# ==========================================

print("=" * 70)
print("MODEL A: LOG GDP + WGI")
print("=" * 70)

display(
    calculate_vif(
        model_A_data,
        ["LOG_GDP", "WGI_SCORE"]
    )
)

print("=" * 70)
print("MODEL B: LOG GDP + LOG GOVERNMENT HEALTH")
print("=" * 70)

display(
    calculate_vif(
        model_B_data,
        ["LOG_GDP", "LOG_GOV_HEALTH"]
    )
)

print("=" * 70)
print("MODEL C: WGI + LOG GOVERNMENT HEALTH")
print("=" * 70)

display(
    calculate_vif(
        model_C_data,
        ["WGI_SCORE", "LOG_GOV_HEALTH"]
    )
)

In [ ]:
# ==========================================
# WGI ONLY vs WGI + GOVERNMENT HEALTH
# ==========================================

# WGI-only model
X_restricted = sm.add_constant(
    model_C_data["WGI_SCORE"]
)

y_restricted = model_C_data["GHSI"]

model_restricted = sm.OLS(
    y_restricted,
    X_restricted
).fit()


# WGI + Government Health model
X_full = sm.add_constant(
    model_C_data[
        ["WGI_SCORE", "LOG_GOV_HEALTH"]
    ]
)

model_full = sm.OLS(
    y_restricted,
    X_full
).fit()


# Nested F-test
f_test = model_full.compare_f_test(
    model_restricted
)

print("=" * 70)
print("WGI ONLY")
print("=" * 70)
print(model_restricted.summary())

print("=" * 70)
print("WGI + GOVERNMENT HEALTH")
print("=" * 70)
print(model_full.summary())

print("=" * 70)
print("NESTED F-TEST")
print("=" * 70)

print("F statistic:", round(f_test[0], 4))
print("p-value:", round(f_test[1], 4))
print("df difference:", f_test[2])

In [ ]:
# ==========================================
# WGI ONLY vs WGI + GDP
# ==========================================

# WGI-only model using the same 52-country sample
model_A_restricted_data = model_A_data[
    ["GHSI", "WGI_SCORE"]
].dropna()

X_restricted_A = sm.add_constant(
    model_A_restricted_data["WGI_SCORE"]
)

y_restricted_A = model_A_restricted_data["GHSI"]

model_A_restricted = sm.OLS(
    y_restricted_A,
    X_restricted_A
).fit()


# Full model: WGI + GDP
X_full_A = sm.add_constant(
    model_A_data[
        ["WGI_SCORE", "LOG_GDP"]
    ]
)

model_A_full = sm.OLS(
    model_A_data["GHSI"],
    X_full_A
).fit()


# Nested F-test
f_test_A = model_A_full.compare_f_test(
    model_A_restricted
)

print("=" * 70)
print("WGI ONLY")
print("=" * 70)
print(model_A_restricted.summary())

print("=" * 70)
print("WGI + GDP")
print("=" * 70)
print(model_A_full.summary())

print("=" * 70)
print("NESTED F-TEST")
print("=" * 70)

print("F statistic:", round(f_test_A[0], 4))
print("p-value:", round(f_test_A[1], 4))
print("df difference:", f_test_A[2])

In [ ]:
# ==========================================
# WGI ONLY vs WGI + INFORM RISK
# ==========================================

model_D_data = df_analysis[
    ["GHSI", "WGI_SCORE", "INFORM_RISK_SCORE"]
].dropna().copy()

# WGI-only model
X_restricted_D = sm.add_constant(
    model_D_data["WGI_SCORE"]
)

y_D = model_D_data["GHSI"]

model_D_restricted = sm.OLS(
    y_D,
    X_restricted_D
).fit()


# WGI + INFORM Risk
X_full_D = sm.add_constant(
    model_D_data[
        ["WGI_SCORE", "INFORM_RISK_SCORE"]
    ]
)

model_D_full = sm.OLS(
    y_D,
    X_full_D
).fit()


# Nested F-test
f_test_D = model_D_full.compare_f_test(
    model_D_restricted
)

print("=" * 70)
print("WGI ONLY")
print("=" * 70)
print(model_D_restricted.summary())

print("=" * 70)
print("WGI + INFORM RISK")
print("=" * 70)
print(model_D_full.summary())

print("=" * 70)
print("NESTED F-TEST")
print("=" * 70)

print("F statistic:", round(f_test_D[0], 4))
print("p-value:", round(f_test_D[1], 4))
print("df difference:", f_test_D[2])

In [ ]:
# ==========================================
# VIF: WGI + INFORM RISK
# ==========================================

display(
    calculate_vif(
        model_D_data,
        ["WGI_SCORE", "INFORM_RISK_SCORE"]
    )
)

### Why Governance and INFORM Risk Are Assessed Together


## 6. INFORM 2026 Data Integration


In [ ]:
# ==========================================
# LOAD INFORM 2026 WORKBOOK
# ==========================================

inform_path = INFORM_FILE

inform_excel = pd.ExcelFile(inform_path)

print("Sheets in INFORM workbook:")
print(inform_excel.sheet_names)

In [ ]:
# ==========================================
# PREVIEW ALL SHEETS
# ==========================================

for sheet in inform_excel.sheet_names:
    temp = pd.read_excel(inform_path, sheet_name=sheet)

    print("\n" + "=" * 70)
    print("SHEET:", sheet)
    print("Shape:", temp.shape)
    print("Columns:")
    
    for col in temp.columns:
        print(" -", col)

In [ ]:
# ==========================================
# INSPECT INFORM DIMENSION SHEETS
# ==========================================

inform_sheets = [
    "INFORM Risk 2026 (a-z)",
    "Hazard & Exposure",
    "Vulnerability",
    "Lack of Coping Capacity"
]

for sheet in inform_sheets:
    
    temp = pd.read_excel(
        inform_path,
        sheet_name=sheet
    )
    
    print("\n" + "=" * 80)
    print("SHEET:", sheet)
    print("SHAPE:", temp.shape)
    print("=" * 80)
    
    print("\nCOLUMNS:")
    for i, col in enumerate(temp.columns):
        print(i, ":", col)

In [ ]:
# ==========================================
# PREVIEW INFORM DIMENSION SHEETS
# ==========================================

for sheet in inform_sheets:
    
    temp = pd.read_excel(
        inform_path,
        sheet_name=sheet
    )
    
    print("\n" + "=" * 80)
    print("SHEET:", sheet)
    print("=" * 80)
    
    display(temp.head(3))

In [ ]:
# ==========================================
# INSPECT INFORM RISK DATA
# ==========================================

inform_risk_raw = pd.read_excel(
    inform_path,
    sheet_name="INFORM Risk 2026 (a-z)",
    header=None
)

print("Shape:", inform_risk_raw.shape)

display(inform_risk_raw.head(10))

In [ ]:
# ==========================================
# EXTRACT INFORM DIMENSION DATA
# ==========================================

inform_raw = pd.read_excel(
    inform_path,
    sheet_name="INFORM Risk 2026 (a-z)",
    header=1
)

# Remove completely empty columns
inform_raw = inform_raw.dropna(axis=1, how="all")

print("Columns:")
for col in inform_raw.columns:
    print("-", col)

print("\nShape:", inform_raw.shape)

display(inform_raw.head())

In [ ]:
# ==========================================
# CHECK INFORM DIMENSIONS
# ==========================================

inform_dimensions = [
    "COUNTRY",
    "ISO3",
    "INFORM RISK",
    "HAZARD & EXPOSURE",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY"
]

display(
    inform_raw[inform_dimensions].head(10)
)

print("\nMissing values:")
print(
    inform_raw[inform_dimensions].isna().sum()
)

In [ ]:
# ==========================================
# FIND COPING CAPACITY COLUMNS
# ==========================================

for col in inform_raw.columns:
    if any(
        term in str(col).upper()
        for term in [
            "COPING",
            "INSTITUTIONAL",
            "DRR",
            "GOVERNANCE",
            "INFRASTRUCTURE",
            "COMMUNICATION",
            "HEALTH CARE"
        ]
    ):
        print(col)
        

In [ ]:
# ==========================================
# INFORM LACK OF COPING COMPONENTS
# ==========================================

coping_components = [
    "Institutional",
    "DRR",
    "Governance",
    "Infrastructure",
    "Communication",
    "Physical infrastructure",
    "Access to health care"
]

for col in coping_components:
    print(col, "->", col in inform_raw.columns)

In [ ]:
# ==========================================
# CHECK COMPONENT DATA
# ==========================================

display(
    inform_raw[
        ["ISO3"] + coping_components
    ].head(10)
)

print("\nMissing values:")
print(
    inform_raw[
        coping_components
    ].isna().sum()
)

In [ ]:
# ==========================================
# WGI + INFORM COPING COMPONENTS
# Individual nested models
# ==========================================

# Merge the seven components into df_analysis
coping_data = inform_raw[
    ["ISO3"] + coping_components
].copy()

df_analysis_inform = df_analysis.merge(
    coping_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

# Store results
coping_results = []

for component in coping_components:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].dropna().copy()

    # WGI-only model on the same sample
    X_restricted = sm.add_constant(
        model_data["WGI_SCORE"]
    )

    y = model_data["GHSI"]

    restricted_model = sm.OLS(
        y,
        X_restricted
    ).fit()

    # WGI + component
    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ]
    )

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # Nested F-test
    f_test = full_model.compare_f_test(
        restricted_model
    )

    # VIF
    vif_table = calculate_vif(
        model_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table[
        vif_table["Variable"] == component
    ]["VIF"].iloc[0]

    coping_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "Component_coef": full_model.params[component],
        "Component_p": full_model.pvalues[component],
        "WGI_p": full_model.pvalues["WGI_SCORE"],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })

coping_results_df = pd.DataFrame(
    coping_results
)

display(
    coping_results_df.round(4)
)

In [ ]:
# ==========================================
# CLEAN INFORM COPING COMPONENTS
# ==========================================

coping_components = [
    "Institutional",
    "DRR",
    "Governance",
    "Infrastructure",
    "Communication",
    "Physical infrastructure",
    "Access to health care"
]

# Create merged dataset
coping_data = inform_raw[
    ["ISO3"] + coping_components
].copy()

df_analysis_inform = df_analysis.merge(
    coping_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

# Convert INFORM components to numeric
for component in coping_components:
    df_analysis_inform[component] = pd.to_numeric(
        df_analysis_inform[component],
        errors="coerce"
    )

# Check data types
print(df_analysis_inform[coping_components].dtypes)

# Check missing values after conversion
print("\nMissing values:")
print(
    df_analysis_inform[coping_components].isna().sum()
)

In [ ]:
# ==========================================
# RESTRICT INFORM DATA TO AFRICA
# ==========================================

# African ISO3 codes represented in your analysis dataset
africa_iso3 = df_analysis["ISO3"].dropna().unique()

print("African countries in df_analysis:", len(africa_iso3))

# Keep only African countries from INFORM
inform_africa = inform_raw[
    inform_raw["ISO3"].isin(africa_iso3)
].copy()

print("African countries in INFORM:", len(inform_africa))

# Check whether all df_analysis countries were found
missing_from_inform = set(africa_iso3) - set(inform_africa["ISO3"])

print("African countries missing from INFORM:", len(missing_from_inform))

if missing_from_inform:
    print(sorted(missing_from_inform))

In [ ]:
# ==========================================
# CHECK INFORM AFRICA COMPONENTS
# ==========================================

coping_components = [
    "Institutional",
    "DRR",
    "Governance",
    "Infrastructure",
    "Communication",
    "Physical infrastructure",
    "Access to health care"
]

print("\nMissing values among African countries:")
print(
    inform_africa[coping_components].isna().sum()
)

In [ ]:
# ==========================================
# DRR MISSINGNESS — AFRICA ONLY
# ==========================================

drr_missing_africa = inform_africa[
    inform_africa["DRR"].isna()
][
    ["COUNTRY", "ISO3", "DRR"]
]

print(
    "African countries missing DRR:",
    len(drr_missing_africa)
)

display(drr_missing_africa)

In [ ]:
# ==========================================
# MERGE AFRICAN INFORM DATA
# ==========================================

coping_data = inform_africa[
    ["ISO3"] + coping_components
].copy()

df_analysis_inform = df_analysis.merge(
    coping_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

print("df_analysis N:", len(df_analysis))
print("Merged N:", len(df_analysis_inform))

print("\nMissing after Africa-only merge:")
print(
    df_analysis_inform[coping_components].isna().sum()
)

In [ ]:
# ==========================================
# WGI + INFORM COPING COMPONENTS
# ROBUST NUMERIC VERSION
# ==========================================

coping_results = []

for component in coping_components:

    # Select variables
    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    # Force everything to numeric
    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    # Remove missing/non-numeric observations
    model_data = model_data.dropna().copy()

    # Explicitly convert to float
    y = model_data["GHSI"].astype(float)

    X_restricted = sm.add_constant(
        model_data[["WGI_SCORE"]].astype(float)
    )

    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].astype(float)
    )

    # ------------------------------------------
    # WGI ONLY
    # ------------------------------------------

    restricted_model = sm.OLS(
        y,
        X_restricted
    ).fit()

    # ------------------------------------------
    # WGI + COMPONENT
    # ------------------------------------------

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # ------------------------------------------
    # NESTED F-TEST
    # ------------------------------------------

    f_test = full_model.compare_f_test(
        restricted_model
    )

    # ------------------------------------------
    # VIF
    # ------------------------------------------

    vif_table = calculate_vif(
        model_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    # ------------------------------------------
    # STORE RESULTS
    # ------------------------------------------

    coping_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "Component_coef": full_model.params[component],
        "Component_p": full_model.pvalues[component],
        "WGI_coef": full_model.params["WGI_SCORE"],
        "WGI_p": full_model.pvalues["WGI_SCORE"],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })


# ==========================================
# RESULTS TABLE
# ==========================================

coping_results_df = pd.DataFrame(
    coping_results
)

display(
    coping_results_df.round(4)
)

In [ ]:
# ==========================================
# DIAGNOSE DTYPE PROBLEM
# ==========================================

test_data = df_analysis_inform[
    ["GHSI", "WGI_SCORE", "Institutional"]
].copy()

for col in test_data.columns:
    test_data[col] = pd.to_numeric(
        test_data[col],
        errors="coerce"
    )

test_data = test_data.dropna()

print(test_data.dtypes)
print()
print(test_data.head())
print()
print("Shape:", test_data.shape)

# Force NumPy arrays
y_test = test_data["GHSI"].to_numpy(dtype=float)

X_test = sm.add_constant(
    test_data[["WGI_SCORE", "Institutional"]].to_numpy(dtype=float)
)

print("\ny dtype:", y_test.dtype)
print("X dtype:", X_test.dtype)

# Test OLS
test_model = sm.OLS(
    y_test,
    X_test
).fit()

print(test_model.summary())

In [ ]:
# ==========================================
# INFORM COPING COMPONENT ANALYSIS
# WGI + EACH COMPONENT
# AFRICA ONLY
# ==========================================

coping_results = []

for component in coping_components:

    # --------------------------------------
    # Prepare clean data
    # --------------------------------------

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    # --------------------------------------
    # Convert to NumPy
    # --------------------------------------

    y = model_data["GHSI"].to_numpy(dtype=float)

    X_wgi = sm.add_constant(
        model_data[["WGI_SCORE"]].to_numpy(dtype=float)
    )

    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].to_numpy(dtype=float)
    )

    # --------------------------------------
    # Restricted: WGI only
    # --------------------------------------

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # --------------------------------------
    # Full: WGI + INFORM component
    # --------------------------------------

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # --------------------------------------
    # Nested F-test
    # --------------------------------------

    f_test = full_model.compare_f_test(
        restricted_model
    )

    # --------------------------------------
    # VIF
    # --------------------------------------

    vif_data = model_data[
        ["WGI_SCORE", component]
    ].copy()

    vif_data = vif_data.astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    # --------------------------------------
    # Store results
    # --------------------------------------

    coping_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Component_coef": full_model.params[2],
        "Component_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })


# ==========================================
# RESULTS
# ==========================================

coping_results_df = pd.DataFrame(
    coping_results
)

display(
    coping_results_df.round(4)
)

In [ ]:
# ==========================================
# RANK COMPONENTS BY NESTED F-TEST
# ==========================================

coping_results_ranked = (
    coping_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

# Rank from 1
coping_results_ranked.index = coping_results_ranked.index + 1
coping_results_ranked.index.name = "Rank"

display(
    coping_results_ranked.round(4)
)

In [ ]:
# ==========================================
# INFORM DIMENSION ANALYSIS
# WGI + INFORM MAJOR DIMENSIONS
# AFRICA ONLY
# ==========================================

inform_dimensions = [
    "HAZARD & EXPOSURE",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY"
]

dimension_results = []

for dimension in inform_dimensions:

    # --------------------------------------
    # Prepare data
    # --------------------------------------

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", dimension]
    ].copy()

    for col in ["GHSI", "WGI_SCORE", dimension]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    # --------------------------------------
    # NumPy arrays
    # --------------------------------------

    y = model_data["GHSI"].to_numpy(dtype=float)

    X_wgi = sm.add_constant(
        model_data[
            ["WGI_SCORE"]
        ].to_numpy(dtype=float)
    )

    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", dimension]
        ].to_numpy(dtype=float)
    )

    # --------------------------------------
    # WGI-only model
    # --------------------------------------

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # --------------------------------------
    # WGI + INFORM dimension
    # --------------------------------------

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # --------------------------------------
    # Nested F-test
    # --------------------------------------

    f_test = full_model.compare_f_test(
        restricted_model
    )

    # --------------------------------------
    # VIF
    # --------------------------------------

    vif_data = model_data[
        ["WGI_SCORE", dimension]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", dimension]
    )

    dimension_vif = vif_table.loc[
        vif_table["Variable"] == dimension,
        "VIF"
    ].iloc[0]

    # --------------------------------------
    # Store results
    # --------------------------------------

    dimension_results.append({
        "Dimension": dimension,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Dimension_coef": full_model.params[2],
        "Dimension_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Dimension_VIF": dimension_vif
    })


# ==========================================
# RESULTS TABLE
# ==========================================

dimension_results_df = pd.DataFrame(
    dimension_results
)

dimension_results_df.index = (
    dimension_results_df.index + 1
)

dimension_results_df.index.name = "Rank"

display(
    dimension_results_df.round(4)
)

In [ ]:
# ==========================================
# ADD INFORM MAJOR DIMENSIONS
# AFRICA ONLY
# ==========================================

inform_dimensions = [
    "HAZARD & EXPOSURE",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY"
]

# Extract dimensions from Africa-only INFORM data
dimension_data = inform_africa[
    ["ISO3"] + inform_dimensions
].copy()

# Convert to numeric
for dimension in inform_dimensions:
    dimension_data[dimension] = pd.to_numeric(
        dimension_data[dimension],
        errors="coerce"
    )

# Add dimensions to existing analysis dataset
df_analysis_inform = df_analysis_inform.drop(
    columns=inform_dimensions,
    errors="ignore"
)

df_analysis_inform = df_analysis_inform.merge(
    dimension_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

print("Merged N:", len(df_analysis_inform))

print("\nMissing values:")
print(
    df_analysis_inform[inform_dimensions].isna().sum()
)

In [ ]:
# ==========================================
# INFORM DIMENSION ANALYSIS
# WGI + INFORM MAJOR DIMENSIONS
# ==========================================

dimension_results = []

for dimension in inform_dimensions:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", dimension]
    ].copy()

    for col in ["GHSI", "WGI_SCORE", dimension]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    y = model_data["GHSI"].to_numpy(dtype=float)

    X_wgi = sm.add_constant(
        model_data[["WGI_SCORE"]].to_numpy(dtype=float)
    )

    X_full = sm.add_constant(
        model_data[["WGI_SCORE", dimension]].to_numpy(dtype=float)
    )

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    f_test = full_model.compare_f_test(
        restricted_model
    )

    vif_data = model_data[
        ["WGI_SCORE", dimension]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", dimension]
    )

    dimension_vif = vif_table.loc[
        vif_table["Variable"] == dimension,
        "VIF"
    ].iloc[0]

    dimension_results.append({
        "Dimension": dimension,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Dimension_coef": full_model.params[2],
        "Dimension_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Dimension_VIF": dimension_vif
    })

dimension_results_df = pd.DataFrame(
    dimension_results
)

dimension_results_df = (
    dimension_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

dimension_results_df.index = (
    dimension_results_df.index + 1
)

dimension_results_df.index.name = "Rank"

display(
    dimension_results_df.round(4)
)

In [ ]:
# ==========================================
# ADD INFORM HAZARD COMPONENTS
# ==========================================

hazard_components = [
    "Natural",
    "Human"
]

hazard_data = inform_africa[
    ["ISO3"] + hazard_components
].copy()

# Convert to numeric
for component in hazard_components:
    hazard_data[component] = pd.to_numeric(
        hazard_data[component],
        errors="coerce"
    )

# Remove if already present, then merge
df_analysis_inform = df_analysis_inform.drop(
    columns=hazard_components,
    errors="ignore"
)

df_analysis_inform = df_analysis_inform.merge(
    hazard_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

print("Merged N:", len(df_analysis_inform))

print("\nMissing values:")
print(
    df_analysis_inform[hazard_components].isna().sum()
)

In [ ]:
# ==========================================
# HAZARD & EXPOSURE DECOMPOSITION
# WGI + NATURAL / HUMAN HAZARD
# ==========================================

hazard_components = [
    "Natural",
    "Human"
]

hazard_results = []

for component in hazard_components:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    y = model_data["GHSI"].to_numpy(dtype=float)

    # WGI only
    X_wgi = sm.add_constant(
        model_data[
            ["WGI_SCORE"]
        ].to_numpy(dtype=float)
    )

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # WGI + hazard component
    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].to_numpy(dtype=float)
    )

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # Nested F-test
    f_test = full_model.compare_f_test(
        restricted_model
    )

    # VIF
    vif_data = model_data[
        ["WGI_SCORE", component]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    hazard_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Component_coef": full_model.params[2],
        "Component_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })

hazard_results_df = pd.DataFrame(
    hazard_results
)

hazard_results_df = (
    hazard_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

hazard_results_df.index = (
    hazard_results_df.index + 1
)

hazard_results_df.index.name = "Rank"

display(
    hazard_results_df.round(4)
)

In [ ]:
# ==========================================
# HAZARD & EXPOSURE DECOMPOSITION
# WGI + NATURAL / HUMAN HAZARD
# AFRICA ONLY
# ==========================================

hazard_results = []

for component in hazard_components:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    # Force numeric
    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    # NumPy arrays
    y = model_data["GHSI"].to_numpy(dtype=float)

    # WGI-only model
    X_wgi = sm.add_constant(
        model_data[
            ["WGI_SCORE"]
        ].to_numpy(dtype=float)
    )

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # WGI + Natural/Human
    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].to_numpy(dtype=float)
    )

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # Nested F-test
    f_test = full_model.compare_f_test(
        restricted_model
    )

    # VIF
    vif_data = model_data[
        ["WGI_SCORE", component]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    # Store results
    hazard_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Component_coef": full_model.params[2],
        "Component_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })


# ==========================================
# RESULTS TABLE
# ==========================================

hazard_results_df = pd.DataFrame(
    hazard_results
)

hazard_results_df = (
    hazard_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

# Rank from 1
hazard_results_df.index = (
    hazard_results_df.index + 1
)

hazard_results_df.index.name = "Rank"

display(
    hazard_results_df.round(4)
)

In [ ]:
# ==========================================
# HUMAN HAZARD DECOMPOSITION
# WGI + HUMAN HAZARD COMPONENTS
# ==========================================

human_components = [
    "Projected Conflict Probability",
    "Current Conflict Intensity"
]

human_results = []

for component in human_components:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    # Force numeric
    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    y = model_data["GHSI"].to_numpy(dtype=float)

    # WGI-only model
    X_wgi = sm.add_constant(
        model_data[
            ["WGI_SCORE"]
        ].to_numpy(dtype=float)
    )

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # WGI + human component
    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].to_numpy(dtype=float)
    )

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # Nested F-test
    f_test = full_model.compare_f_test(
        restricted_model
    )

    # VIF
    vif_data = model_data[
        ["WGI_SCORE", component]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    human_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Component_coef": full_model.params[2],
        "Component_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })

# ==========================================
# RESULTS
# ==========================================

human_results_df = pd.DataFrame(
    human_results
)

human_results_df = (
    human_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

human_results_df.index = (
    human_results_df.index + 1
)

human_results_df.index.name = "Rank"

display(
    human_results_df.round(4)
)

In [ ]:
# ==========================================
# ADD HUMAN HAZARD COMPONENTS
# ==========================================

human_components = [
    "Projected Conflict Probability",
    "Current Conflict Intensity"
]

human_data = inform_africa[
    ["ISO3"] + human_components
].copy()

for component in human_components:
    human_data[component] = pd.to_numeric(
        human_data[component],
        errors="coerce"
    )

df_analysis_inform = df_analysis_inform.drop(
    columns=human_components,
    errors="ignore"
)

df_analysis_inform = df_analysis_inform.merge(
    human_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

print("Merged N:", len(df_analysis_inform))

print("\nMissing values:")
print(
    df_analysis_inform[human_components].isna().sum()
)

In [ ]:
# ==========================================
# HUMAN HAZARD DECOMPOSITION
# WGI + HUMAN HAZARD COMPONENTS
# ==========================================

human_results = []

for component in human_components:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    y = model_data["GHSI"].to_numpy(dtype=float)

    # WGI-only model
    X_wgi = sm.add_constant(
        model_data[["WGI_SCORE"]].to_numpy(dtype=float)
    )

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # WGI + Human Hazard component
    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].to_numpy(dtype=float)
    )

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # Nested F-test
    f_test = full_model.compare_f_test(
        restricted_model
    )

    # VIF
    vif_data = model_data[
        ["WGI_SCORE", component]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    human_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Component_coef": full_model.params[2],
        "Component_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })

human_results_df = pd.DataFrame(
    human_results
)

human_results_df = (
    human_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

human_results_df.index = human_results_df.index + 1
human_results_df.index.name = "Rank"

display(
    human_results_df.round(4)
)

In [ ]:
# ==========================================
# ADD NATURAL HAZARD INDICATORS
# ==========================================

natural_components = [
    "Earthquake",
    "River Flood",
    "Tsunami",
    "Tropical Cyclone",
    "Coastal flood",
    "Drought",
    "Epidemic"
]

natural_data = inform_africa[
    ["ISO3"] + natural_components
].copy()

for component in natural_components:
    natural_data[component] = pd.to_numeric(
        natural_data[component],
        errors="coerce"
    )

df_analysis_inform = df_analysis_inform.drop(
    columns=natural_components,
    errors="ignore"
)

df_analysis_inform = df_analysis_inform.merge(
    natural_data,
    on="ISO3",
    how="left",
    validate="one_to_one"
)

print("Merged N:", len(df_analysis_inform))

print("\nMissing values:")
print(
    df_analysis_inform[natural_components].isna().sum()
)

In [ ]:
# ==========================================
# NATURAL HAZARD DECOMPOSITION
# WGI + INDIVIDUAL NATURAL HAZARDS
# ==========================================

natural_results = []

for component in natural_components:

    model_data = df_analysis_inform[
        ["GHSI", "WGI_SCORE", component]
    ].copy()

    # Force numeric
    for col in ["GHSI", "WGI_SCORE", component]:
        model_data[col] = pd.to_numeric(
            model_data[col],
            errors="coerce"
        )

    model_data = model_data.dropna()

    y = model_data["GHSI"].to_numpy(dtype=float)

    # --------------------------------------
    # WGI-only restricted model
    # --------------------------------------

    X_wgi = sm.add_constant(
        model_data[
            ["WGI_SCORE"]
        ].to_numpy(dtype=float)
    )

    restricted_model = sm.OLS(
        y,
        X_wgi
    ).fit()

    # --------------------------------------
    # WGI + Natural Hazard component
    # --------------------------------------

    X_full = sm.add_constant(
        model_data[
            ["WGI_SCORE", component]
        ].to_numpy(dtype=float)
    )

    full_model = sm.OLS(
        y,
        X_full
    ).fit()

    # --------------------------------------
    # Nested F-test
    # --------------------------------------

    f_test = full_model.compare_f_test(
        restricted_model
    )

    # --------------------------------------
    # VIF
    # --------------------------------------

    vif_data = model_data[
        ["WGI_SCORE", component]
    ].astype(float)

    vif_table = calculate_vif(
        vif_data,
        ["WGI_SCORE", component]
    )

    component_vif = vif_table.loc[
        vif_table["Variable"] == component,
        "VIF"
    ].iloc[0]

    # --------------------------------------
    # Store results
    # --------------------------------------

    natural_results.append({
        "Component": component,
        "N": int(full_model.nobs),
        "R_squared": full_model.rsquared,
        "Adj_R_squared": full_model.rsquared_adj,
        "WGI_coef": full_model.params[1],
        "WGI_p": full_model.pvalues[1],
        "Component_coef": full_model.params[2],
        "Component_p": full_model.pvalues[2],
        "Nested_F": f_test[0],
        "Nested_p": f_test[1],
        "Component_VIF": component_vif
    })


# ==========================================
# RESULTS TABLE
# ==========================================

natural_results_df = pd.DataFrame(
    natural_results
)

natural_results_df = (
    natural_results_df
    .sort_values("Nested_p")
    .reset_index(drop=True)
)

# Rank from 1
natural_results_df.index = (
    natural_results_df.index + 1
)

natural_results_df.index.name = "Rank"

display(
    natural_results_df.round(4)
)

In [ ]:
# ==========================================
# ROBUSTNESS: WGI + PROJECTED CONFLICT
# INFLUENCE DIAGNOSTICS
# ==========================================

component = "Projected Conflict Probability"

model_data = df_analysis_inform[
    ["COUNTRY_NAME", "GHSI", "WGI_SCORE", component]
].copy()

for col in ["GHSI", "WGI_SCORE", component]:
    model_data[col] = pd.to_numeric(
        model_data[col],
        errors="coerce"
    )

model_data = model_data.dropna().copy()

# ------------------------------------------
# Fit model
# ------------------------------------------

X = sm.add_constant(
    model_data[
        ["WGI_SCORE", component]
    ]
)

y = model_data["GHSI"]

model_pc = sm.OLS(
    y,
    X
).fit()

print(model_pc.summary())

# ------------------------------------------
# Influence diagnostics
# ------------------------------------------

influence = model_pc.get_influence()

# Cook's Distance
model_data["Cooks_Distance"] = (
    influence.cooks_distance[0]
)

# Leverage
model_data["Leverage"] = (
    influence.hat_matrix_diag
)

# Studentized residuals
model_data["Studentized_Residual"] = (
    influence.resid_studentized_external
)

# ------------------------------------------
# Display highest Cook's Distance
# ------------------------------------------

print("\n" + "="*70)
print("TOP COOK'S DISTANCE")
print("="*70)

display(
    model_data[
        [
            "COUNTRY_NAME",
            "Cooks_Distance"
        ]
    ]
    .sort_values(
        "Cooks_Distance",
        ascending=False
    )
    .head(15)
    .round(4)
)

# ------------------------------------------
# Display highest leverage
# ------------------------------------------

print("\n" + "="*70)
print("TOP LEVERAGE")
print("="*70)

display(
    model_data[
        [
            "COUNTRY_NAME",
            "Leverage"
        ]
    ]
    .sort_values(
        "Leverage",
        ascending=False
    )
    .head(15)
    .round(4)
)

In [ ]:
# ==========================================
# PROJECTED CONFLICT MODEL
# IDENTIFY POTENTIALLY INFLUENTIAL COUNTRIES
# ==========================================

n = len(model_data)

cook_threshold = 4 / n

print("N =", n)
print("Cook's Distance threshold =", round(cook_threshold, 4))

influential_pc = (
    model_data[
        model_data["Cooks_Distance"] > cook_threshold
    ]
    [
        [
            "COUNTRY_NAME",
            "Cooks_Distance",
            "Leverage",
            "Studentized_Residual"
        ]
    ]
    .sort_values(
        "Cooks_Distance",
        ascending=False
    )
)

print("\nPotentially influential observations:")
display(
    influential_pc.round(4)
)

In [ ]:
# ==========================================
# PROJECTED CONFLICT PROBABILITY
# SENSITIVITY ANALYSIS
# REMOVE INFLUENTIAL OBSERVATIONS
# ==========================================

exclude_pc = [
    "South Africa",
    "Liberia",
    "Somalia, Fed. Rep.",
    "Equatorial Guinea"
]

# ------------------------------------------
# Create sensitivity dataset
# ------------------------------------------

sensitivity_data_pc = df_analysis_inform[
    ~df_analysis_inform["COUNTRY_NAME"].isin(exclude_pc)
][
    ["GHSI", "WGI_SCORE", "Projected Conflict Probability"]
].copy()

# Convert to numeric
for col in [
    "GHSI",
    "WGI_SCORE",
    "Projected Conflict Probability"
]:
    sensitivity_data_pc[col] = pd.to_numeric(
        sensitivity_data_pc[col],
        errors="coerce"
    )

sensitivity_data_pc = (
    sensitivity_data_pc
    .dropna()
    .copy()
)

# ------------------------------------------
# Fit sensitivity model
# ------------------------------------------

X_sensitivity_pc = sm.add_constant(
    sensitivity_data_pc[
        [
            "WGI_SCORE",
            "Projected Conflict Probability"
        ]
    ]
)

y_sensitivity_pc = (
    sensitivity_data_pc["GHSI"]
)

model_pc_sensitivity = sm.OLS(
    y_sensitivity_pc,
    X_sensitivity_pc
).fit()

# ------------------------------------------
# Results
# ------------------------------------------

print("=" * 70)
print("WGI + PROJECTED CONFLICT PROBABILITY")
print("SENSITIVITY ANALYSIS")
print("=" * 70)

print("\nExcluded countries:")
for country in exclude_pc:
    print("-", country)

print("\nOriginal N:", len(model_data))
print(
    "Sensitivity N:",
    len(sensitivity_data_pc)
)

print("\n")
print(model_pc_sensitivity.summary())

In [ ]:
# ==========================================
# HETEROSKEDASTICITY TEST
# WGI + PROJECTED CONFLICT PROBABILITY
# SENSITIVITY MODEL
# ==========================================

from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(
    model_pc_sensitivity.resid,
    model_pc_sensitivity.model.exog
)

bp_results = pd.Series(
    bp_test[:4],
    index=[
        "LM Statistic",
        "LM p-value",
        "F Statistic",
        "F p-value"
    ]
)

print(bp_results.round(4))

### Sierra Leone Country-Level Analysis


## 7. Sierra Leone Baseline

The analysis then moves from the wider country dataset to Sierra Leone's position within the African comparison set.


In [ ]:
# ==========================================
# SIERRA LEONE: AFRICAN BENCHMARK POSITION
# ==========================================

sierra = df_analysis_inform[
    df_analysis_inform["COUNTRY_NAME"].str.strip().eq(
        "Sierra Leone"
    )
].copy()

benchmark_variables = [
    "GHSI",
    "WGI_SCORE",
    "INFORM RISK",
    "HAZARD & EXPOSURE",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY",
    "Human",
    "Projected Conflict Probability"
]

# Keep only variables that exist
benchmark_variables = [
    v for v in benchmark_variables
    if v in df_analysis_inform.columns
]

# ------------------------------------------
# Calculate African rank and percentile
# ------------------------------------------

benchmark_results = []

for variable in benchmark_variables:

    values = pd.to_numeric(
        df_analysis_inform[variable],
        errors="coerce"
    )

    sierra_value = pd.to_numeric(
        sierra[variable],
        errors="coerce"
    ).iloc[0]

    valid_values = values.dropna()

    # Rank: highest value = rank 1
    rank = (
        valid_values
        .rank(
            ascending=False,
            method="min"
        )
        .loc[
            sierra.index[0]
        ]
        if sierra.index[0] in valid_values.index
        else np.nan
    )

    # Percentile: proportion of countries at or below Sierra Leone
    percentile = (
        (valid_values <= sierra_value).sum()
        / len(valid_values)
    ) * 100

    benchmark_results.append({
        "Variable": variable,
        "Sierra_Leone_Value": sierra_value,
        "African_N": len(valid_values),
        "African_Rank": int(rank),
        "Percentile": percentile
    })

sierra_benchmark = pd.DataFrame(
    benchmark_results
)

display(
    sierra_benchmark.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE: AFRICAN BENCHMARK POSITION
# INCLUDING GINI
# ==========================================

sierra = df_analysis_inform[
    df_analysis_inform["COUNTRY_NAME"].str.strip().eq(
        "Sierra Leone"
    )
].copy()

benchmark_variables = [
    "GHSI",
    "WGI_SCORE",
    "INFORM RISK",
    "HAZARD & EXPOSURE",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY",
    "Human",
    "Projected Conflict Probability",
    "GINI"
]

# Keep only variables actually present
benchmark_variables = [
    v for v in benchmark_variables
    if v in df_analysis_inform.columns
]

benchmark_results = []

for variable in benchmark_variables:

    values = pd.to_numeric(
        df_analysis_inform[variable],
        errors="coerce"
    )

    sierra_value = pd.to_numeric(
        sierra[variable],
        errors="coerce"
    ).iloc[0]

    valid_values = values.dropna()

    # --------------------------------------
    # Rank
    # Highest value = Rank 1
    # --------------------------------------

    if sierra.index[0] in valid_values.index:

        rank = (
            valid_values
            .rank(
                ascending=False,
                method="min"
            )
            .loc[sierra.index[0]]
        )

    else:
        rank = np.nan

    # --------------------------------------
    # Percentile
    # --------------------------------------

    percentile = (
        (valid_values <= sierra_value).sum()
        / len(valid_values)
    ) * 100

    benchmark_results.append({
        "Variable": variable,
        "Sierra_Leone_Value": sierra_value,
        "African_N": len(valid_values),
        "African_Rank": (
            int(rank)
            if not pd.isna(rank)
            else np.nan
        ),
        "Percentile": percentile
    })

sierra_benchmark = pd.DataFrame(
    benchmark_results
)

display(
    sierra_benchmark.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE: OBSERVED VS PREDICTED GHSI
# FINAL ROBUST MODEL
# WGI + PROJECTED CONFLICT PROBABILITY
# ==========================================

# ------------------------------------------
# Use the original 54-country model
# ------------------------------------------

component = "Projected Conflict Probability"

model_data = df_analysis_inform[
    [
        "COUNTRY_NAME",
        "GHSI",
        "WGI_SCORE",
        component
    ]
].copy()

for col in ["GHSI", "WGI_SCORE", component]:
    model_data[col] = pd.to_numeric(
        model_data[col],
        errors="coerce"
    )

model_data = model_data.dropna().copy()

# ------------------------------------------
# Fit final model
# ------------------------------------------

X = sm.add_constant(
    model_data[
        ["WGI_SCORE", component]
    ]
)

y = model_data["GHSI"]

final_model = sm.OLS(
    y,
    X
).fit()

print(final_model.summary())

# ------------------------------------------
# Sierra Leone values
# ------------------------------------------

sierra = model_data[
    model_data["COUNTRY_NAME"].str.strip().eq(
        "Sierra Leone"
    )
].copy()

# ------------------------------------------
# Predicted GHSI
# ------------------------------------------

X_sierra = sm.add_constant(
    sierra[
        ["WGI_SCORE", component]
    ],
    has_constant="add"
)

sierra["Predicted_GHSI"] = (
    final_model.predict(X_sierra)
)

sierra["Observed_GHSI"] = (
    sierra["GHSI"]
)

sierra["GHSI_Gap"] = (
    sierra["Observed_GHSI"]
    - sierra["Predicted_GHSI"]
)

# ------------------------------------------
# Display
# ------------------------------------------

print("\n" + "=" * 70)
print("SIERRA LEONE: OBSERVED VS PREDICTED GHSI")
print("=" * 70)

display(
    sierra[
        [
            "COUNTRY_NAME",
            "WGI_SCORE",
            "Projected Conflict Probability",
            "Observed_GHSI",
            "Predicted_GHSI",
            "GHSI_Gap"
        ]
    ].round(4)
)

In [ ]:
# ==========================================
# SIERRA LEONE: INFORM DIMENSION PROFILE
# ==========================================

sierra = df_analysis_inform[
    df_analysis_inform["COUNTRY_NAME"].str.strip().eq(
        "Sierra Leone"
    )
].copy()

inform_dimensions = [
    "HAZARD & EXPOSURE",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY"
]

dimension_profile = []

for dimension in inform_dimensions:

    values = pd.to_numeric(
        df_analysis_inform[dimension],
        errors="coerce"
    ).dropna()

    sierra_value = pd.to_numeric(
        sierra[dimension],
        errors="coerce"
    ).iloc[0]

    # Higher INFORM dimension score = greater risk / poorer position
    rank = (
        values
        .rank(
            ascending=False,
            method="min"
        )
        .loc[sierra.index[0]]
    )

    percentile = (
        (values <= sierra_value).sum()
        / len(values)
    ) * 100

    african_mean = values.mean()

    dimension_profile.append({
        "Dimension": dimension,
        "Sierra_Leone": sierra_value,
        "African_Mean": african_mean,
        "Difference_from_Mean": (
            sierra_value - african_mean
        ),
        "African_Rank": int(rank),
        "Percentile": percentile,
        "N": len(values)
    })

dimension_profile_df = pd.DataFrame(
    dimension_profile
)

display(
    dimension_profile_df.round(3)
)

In [ ]:
# ==========================================
# CHECK INFORM COLUMN NAMES
# ==========================================

for col in df_analysis_inform.columns:
    if "INFORM" in str(col).upper():
        print(repr(col))

In [ ]:
# ==========================================
# SIERRA LEONE: INFORM RISK PROFILE
# ==========================================

inform_risk_variables = [
    "INFORM_RISK_SCORE",
    "HAZARD & EXPOSURE",
    "Natural",
    "Human",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY"
]

sierra = df_analysis_inform[
    df_analysis_inform["COUNTRY_NAME"].str.strip().eq(
        "Sierra Leone"
    )
].copy()

risk_profile = []

for variable in inform_risk_variables:

    values = pd.to_numeric(
        df_analysis_inform[variable],
        errors="coerce"
    ).dropna()

    sierra_value = pd.to_numeric(
        sierra[variable],
        errors="coerce"
    ).iloc[0]

    african_mean = values.mean()
    african_median = values.median()

    rank = (
        values
        .rank(
            ascending=False,
            method="min"
        )
        .loc[sierra.index[0]]
    )

    percentile = (
        (values <= sierra_value).sum()
        / len(values)
    ) * 100

    risk_profile.append({
        "Variable": variable,
        "Sierra_Leone": sierra_value,
        "African_Mean": african_mean,
        "African_Median": african_median,
        "Difference_from_Mean": (
            sierra_value - african_mean
        ),
        "African_Rank": int(rank),
        "Percentile": percentile,
        "N": len(values)
    })

risk_profile_df = pd.DataFrame(
    risk_profile
)

display(
    risk_profile_df.round(3)
)

### Sierra Leone Benchmark Summary


| Dimension                   | Sierra Leone | African mean | Difference |      Rank | Percentile |
| --------------------------- | -----------: | -----------: | ---------: | --------: | ---------: |
| INFORM Risk                 |      **5.1** |        5.376 | **−0.276** |     32/54 |      42.6% |
| Hazard & Exposure           |      **3.8** |        4.822 | **−1.022** |     38/54 |      31.5% |
| Natural                     |      **3.7** |        3.528 | **+0.172** |     24/54 |      57.4% |
| Human                       |      **3.8** |        5.498 | **−1.698** |     37/54 |      33.3% |
| Vulnerability               |      **5.1** |        5.515 | **−0.415** |     34/54 |      38.9% |
| **Lack of Coping Capacity** |      **7.0** |        6.235 | **+0.765** | **14/54** |  **75.9%** |


## 8. International INFORM Benchmarking


In [ ]:
# ==========================================
# INTERNATIONAL INFORM BENCHMARK COUNTRIES
# ==========================================

benchmark_countries = [
    "Sierra Leone",
    "Japan",
    "Singapore",
    "Switzerland",
    "China",
    "Ecuador",
    "New Zealand",
    "Netherlands",
    "South Korea",
    "Chile",
    "Rwanda",
    "Mauritius",
    "Australia"
]

# Load full INFORM dataset
inform_all = pd.read_excel(
    INFORM_FILE,
    sheet_name="INFORM Risk 2026 (a-z)",
    header=1
)

print("Full INFORM dataset:", inform_all.shape)

# Select benchmark countries
benchmark_data = inform_all[
    inform_all["COUNTRY"].isin(benchmark_countries)
].copy()

print("\nCountries found:")
print(benchmark_data["COUNTRY"].tolist())

print("\nCountries NOT found:")
print(
    [
        c for c in benchmark_countries
        if c not in benchmark_data["COUNTRY"].values
    ]
)

print("\nBenchmark N:", len(benchmark_data))

In [ ]:
print(benchmark_data["COUNTRY"].tolist())
print("Benchmark N:", len(benchmark_data))


In [ ]:
# ==========================================
# INTERNATIONAL INFORM BENCHMARK
# RANK FROM 1
# ==========================================

international_benchmark["Rank"] = (
    international_benchmark.index + 1
)

# Move Rank to first column
international_benchmark = international_benchmark[
    ["Rank"] + [
        col for col in international_benchmark.columns
        if col != "Rank"
    ]
]

display(
    international_benchmark.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE VS INTERNATIONAL BENCHMARKS
# GAP ANALYSIS
# ==========================================

# Sierra Leone reference values
sierra_values = (
    international_benchmark[
        international_benchmark["COUNTRY"] == "Sierra Leone"
    ]
    .iloc[0]
)

gap_results = []

for _, row in international_benchmark.iterrows():

    result = {
        "COUNTRY": row["COUNTRY"]
    }

    for variable in available_variables:

        sierra_value = pd.to_numeric(
            sierra_values[variable],
            errors="coerce"
        )

        country_value = pd.to_numeric(
            row[variable],
            errors="coerce"
        )

        # Positive = comparator has higher score than Sierra Leone
        result[
            variable + "_Gap_vs_SL"
        ] = country_value - sierra_value

    gap_results.append(result)

international_gaps = pd.DataFrame(
    gap_results
)

# Rank countries from 1
international_gaps.insert(
    0,
    "Rank",
    range(1, len(international_gaps) + 1)
)

display(
    international_gaps.round(2)
)

In [ ]:
# ==========================================
# INTERNATIONAL BENCHMARK RANKINGS
# SIERRA LEONE VS COMPARATOR COUNTRIES
# ==========================================

international_rankings = []

for variable in available_variables:

    values = pd.to_numeric(
        international_benchmark[variable],
        errors="coerce"
    )

    # Remove missing values
    valid = values.dropna()

    # Higher INFORM score = greater risk / poorer capacity
    ranks = valid.rank(
        ascending=False,
        method="min"
    )

    sierra_row = international_benchmark[
        international_benchmark["COUNTRY"] == "Sierra Leone"
    ].index[0]

    sierra_value = values.loc[sierra_row]

    sierra_rank = ranks.loc[sierra_row]

    percentile = (
        (valid <= sierra_value).sum()
        / len(valid)
    ) * 100

    international_rankings.append({
        "Variable": variable,
        "Sierra_Leone": sierra_value,
        "International_N": len(valid),
        "Sierra_Leone_Rank": int(sierra_rank),
        "Sierra_Leone_Percentile": percentile
    })

international_rankings_df = pd.DataFrame(
    international_rankings
)

display(
    international_rankings_df.round(2)
)

In [ ]:
# ==========================================
# INTERNATIONAL BENCHMARK RANKINGS
# SIERRA LEONE VS COMPARATOR COUNTRIES
# ==========================================

# Reset index first
international_benchmark = (
    international_benchmark
    .reset_index(drop=True)
)

international_rankings = []

for variable in available_variables:

    values = pd.to_numeric(
        international_benchmark[variable],
        errors="coerce"
    )

    valid = values.dropna()

    # Higher INFORM score = greater risk / poorer capacity
    ranks = valid.rank(
        ascending=False,
        method="min"
    )

    # Sierra Leone position
    sierra_position = international_benchmark.index[
        international_benchmark["COUNTRY"] == "Sierra Leone"
    ][0]

    sierra_value = values.iloc[sierra_position]

    sierra_rank = ranks.loc[sierra_position]

    percentile = (
        (valid <= sierra_value).sum()
        / len(valid)
    ) * 100

    international_rankings.append({
        "Variable": variable,
        "Sierra_Leone": sierra_value,
        "International_N": len(valid),
        "Sierra_Leone_Rank": int(sierra_rank),
        "Sierra_Leone_Percentile": percentile
    })

international_rankings_df = pd.DataFrame(
    international_rankings
)

display(
    international_rankings_df.round(2)
)

In [ ]:
# ==========================================
# INTERNATIONAL BENCHMARK RANKINGS
# SIERRA LEONE VS COMPARATOR COUNTRIES
# ==========================================

international_rankings = []

for variable in available_variables:

    values = pd.to_numeric(
        international_benchmark[variable],
        errors="coerce"
    )

    # Sierra Leone value
    sierra_value = pd.to_numeric(
        international_benchmark.loc[
            international_benchmark["COUNTRY"] == "Sierra Leone",
            variable
        ],
        errors="coerce"
    ).iloc[0]

    # Remove missing values
    valid = values.dropna()

    # Rank: highest INFORM score = Rank 1
    sierra_rank = (
        (valid > sierra_value).sum() + 1
    )

    # Percentile
    percentile = (
        (valid <= sierra_value).sum()
        / len(valid)
    ) * 100

    international_rankings.append({
        "Variable": variable,
        "Sierra_Leone": sierra_value,
        "International_N": len(valid),
        "Sierra_Leone_Rank": int(sierra_rank),
        "Sierra_Leone_Percentile": percentile
    })

international_rankings_df = pd.DataFrame(
    international_rankings
)

# Rank the rows of the results table from 1
international_rankings_df.index = (
    international_rankings_df.index + 1
)

international_rankings_df.index.name = "Rank"

display(
    international_rankings_df.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE: COPING CAPACITY
# TARGETED INTERNATIONAL COMPARISON
# ==========================================

comparison_countries = [
    "Sierra Leone",
    "Japan",
    "Ecuador",
    "Rwanda"
]

coping_components = [
    "Institutional",
    "Governance",
    "Infrastructure",
    "Communication",
    "Physical infrastructure",
    "Access to health care"
]

# ------------------------------------------
# Extract countries and components
# ------------------------------------------

coping_comparison = benchmark_data[
    benchmark_data["COUNTRY"].isin(
        comparison_countries
    )
][
    ["COUNTRY"] + coping_components
].copy()

# Convert to numeric
for component in coping_components:
    coping_comparison[component] = pd.to_numeric(
        coping_comparison[component],
        errors="coerce"
    )

# ------------------------------------------
# Put Sierra Leone first
# ------------------------------------------

coping_comparison["SL_first"] = (
    coping_comparison["COUNTRY"]
    .eq("Sierra Leone")
)

coping_comparison = (
    coping_comparison
    .sort_values(
        "SL_first",
        ascending=False
    )
    .drop(columns="SL_first")
    .reset_index(drop=True)
)

# Rank rows from 1
coping_comparison.insert(
    0,
    "Rank",
    range(1, len(coping_comparison) + 1)
)

display(
    coping_comparison.round(2)
)

## 9. Coping Capacity & Indicator-Level Analysis


In [ ]:
# ==========================================
# COPING CAPACITY GAP
# SIERRA LEONE VS JAPAN, ECUADOR, RWANDA
# ==========================================

sierra = coping_comparison[
    coping_comparison["COUNTRY"] == "Sierra Leone"
].iloc[0]

gap_results = []

for _, row in coping_comparison.iterrows():

    result = {
        "COUNTRY": row["COUNTRY"]
    }

    for component in coping_components:

        result[component + "_Gap_vs_SL"] = (
            row[component] - sierra[component]
        )

    gap_results.append(result)

coping_gaps = pd.DataFrame(gap_results)

# Remove Sierra Leone itself
coping_gaps = coping_gaps[
    coping_gaps["COUNTRY"] != "Sierra Leone"
].reset_index(drop=True)

# Rank comparators
coping_gaps.insert(
    0,
    "Rank",
    range(1, len(coping_gaps) + 1)
)

display(
    coping_gaps.round(2)
)

In [ ]:
# ==========================================
# INFORM INDICATOR-LEVEL ANALYSIS
# SIERRA LEONE VS BENCHMARKS
# ==========================================

# Load the Indicator Data sheet from the full INFORM workbook
indicator_data = pd.read_excel(
    INFORM_FILE,
    sheet_name="Indicator Data",
    header=None
)

print("Shape:", indicator_data.shape)

# Inspect first rows/columns
display(
    indicator_data.iloc[:10, :15]
)

In [ ]:
# ==========================================
# LOAD INFORM INDICATOR METADATA
# ==========================================

indicator_metadata = pd.read_excel(
    INFORM_FILE,
    sheet_name="Indicator Metadata",
    header=None
)

print("Shape:", indicator_metadata.shape)

display(
    indicator_metadata.iloc[:15, :20]
)

In [ ]:
# ==========================================
# IDENTIFY COPING-CAPACITY INDICATORS
# ==========================================

# Remove blank first row and use row 1 as headers
metadata = indicator_metadata.iloc[2:].copy()

metadata.columns = indicator_metadata.iloc[1].values

metadata = metadata.reset_index(drop=True)

# Show all indicators belonging to Lack of Coping Capacity
coping_metadata = metadata[
    metadata["Dimension"]
    .astype(str)
    .str.strip()
    .eq("Lack of Coping Capacity")
].copy()

print(
    "Number of coping-capacity indicators:",
    len(coping_metadata)
)

display(
    coping_metadata[
        [
            "Dimension",
            "Category",
            "Component",
            "Sub-Component",
            "INFORM Id",
            "Indicator Name"
        ]
    ]
)

In [ ]:
# ==========================================
# INDICATOR-LEVEL COPING CAPACITY COMPARISON
# SIERRA LEONE VS JAPAN, ECUADOR, RWANDA
# ==========================================

comparison_countries = [
    "Sierra Leone",
    "Japan",
    "Ecuador",
    "Rwanda"
]

# Get coping-capacity metadata
coping_metadata = coping_metadata.copy()

inform_ids = (
    coping_metadata["INFORM Id"]
    .dropna()
    .astype(str)
    .tolist()
)

indicator_names = (
    coping_metadata[
        ["INFORM Id", "Component", "Indicator Name"]
    ]
    .dropna(subset=["INFORM Id"])
    .copy()
)

# ------------------------------------------
# Extract Indicator Data
# ------------------------------------------

# First two columns are COUNTRY and ISO3
indicator_values = indicator_data.iloc[5:].copy()

indicator_values = indicator_values.rename(
    columns={
        1: "COUNTRY",
        2: "ISO3"
    }
)

# Find columns corresponding to INFORM IDs
id_row = indicator_data.iloc[3]

id_to_column = {}

for col in indicator_data.columns:
    value = str(id_row[col]).strip()

    if value in inform_ids:
        id_to_column[value] = col

print(
    "Coping indicators found:",
    len(id_to_column),
    "of",
    len(inform_ids)
)

# ------------------------------------------
# Build long-format comparison
# ------------------------------------------

rows = []

for country in comparison_countries:

    country_row = indicator_values[
        indicator_values["COUNTRY"].astype(str).str.strip()
        == country
    ]

    if country_row.empty:
        print("NOT FOUND:", country)
        continue

    country_row = country_row.iloc[0]

    for _, meta in indicator_names.iterrows():

        inform_id = str(
            meta["INFORM Id"]
        ).strip()

        if inform_id not in id_to_column:
            continue

        col = id_to_column[inform_id]

        value = pd.to_numeric(
            country_row[col],
            errors="coerce"
        )

        rows.append({
            "COUNTRY": country,
            "Component": meta["Component"],
            "Indicator": meta["Indicator Name"],
            "INFORM_ID": inform_id,
            "Value": value
        })

indicator_comparison = pd.DataFrame(rows)

# ------------------------------------------
# Display
# ------------------------------------------

display(
    indicator_comparison.round(3)
)

In [ ]:
# ==========================================
# INDICATOR-LEVEL COPING CAPACITY COMPARISON
# CORRECTED VERSION
# ==========================================

comparison_countries = {
    "Sierra Leone": "SLE",
    "Japan": "JPN",
    "Ecuador": "ECU",
    "Rwanda": "RWA"
}

# ------------------------------------------
# Coping-capacity INFORM IDs
# ------------------------------------------

inform_ids = (
    coping_metadata["INFORM Id"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

indicator_names = (
    coping_metadata[
        ["INFORM Id", "Component", "Indicator Name"]
    ]
    .dropna(subset=["INFORM Id"])
    .copy()
)

# ------------------------------------------
# Map INFORM ID -> Indicator Data column
# ------------------------------------------

id_row = indicator_data.iloc[3]

id_to_column = {}

for col in indicator_data.columns:

    inform_id = str(
        id_row[col]
    ).strip()

    if inform_id in inform_ids:
        id_to_column[inform_id] = col

print(
    "Coping indicators found:",
    len(id_to_column),
    "of",
    len(inform_ids)
)

print("\nMissing INFORM IDs:")

print(
    [
        x for x in inform_ids
        if x not in id_to_column
    ]
)

# ------------------------------------------
# Country data starts at row 5
# ------------------------------------------

indicator_values = indicator_data.iloc[5:].copy()

# Column 1 = COUNTRY
# Column 2 = ISO3

rows = []

for country, iso3 in comparison_countries.items():

    country_rows = indicator_values[
        indicator_values[2]
        .astype(str)
        .str.strip()
        .eq(iso3)
    ]

    if country_rows.empty:
        print(
            "NOT FOUND:",
            country,
            iso3
        )
        continue

    country_row = country_rows.iloc[0]

    for _, meta in indicator_names.iterrows():

        inform_id = str(
            meta["INFORM Id"]
        ).strip()

        if inform_id not in id_to_column:
            continue

        col = id_to_column[inform_id]

        value = pd.to_numeric(
            country_row[col],
            errors="coerce"
        )

        rows.append({
            "COUNTRY": country,
            "ISO3": iso3,
            "Component": meta["Component"],
            "Indicator": meta["Indicator Name"],
            "INFORM_ID": inform_id,
            "Value": value
        })

indicator_comparison = pd.DataFrame(rows)

print(
    "\nRows extracted:",
    len(indicator_comparison)
)

print(
    "Countries:",
    indicator_comparison["COUNTRY"].unique()
)

display(
    indicator_comparison.round(3)
)

In [ ]:
# ==========================================
# CORRECTED INDICATOR-LEVEL EXTRACTION
# ==========================================

comparison_countries = {
    "Sierra Leone": "SLE",
    "Japan": "JPN",
    "Ecuador": "ECU",
    "Rwanda": "RWA"
}

# Indicator IDs from coping metadata
inform_ids = (
    coping_metadata["INFORM Id"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

indicator_names = (
    coping_metadata[
        ["INFORM Id", "Component", "Indicator Name"]
    ]
    .dropna(subset=["INFORM Id"])
    .copy()
)

# ------------------------------------------
# Map INFORM ID -> Indicator Data column
# ------------------------------------------

id_row = indicator_data.iloc[3]

id_to_column = {}

for col in indicator_data.columns:

    inform_id = str(
        id_row[col]
    ).strip()

    if inform_id in inform_ids:
        id_to_column[inform_id] = col

print(
    "Coping indicators found:",
    len(id_to_column),
    "of",
    len(inform_ids)
)

# ------------------------------------------
# Country data
# Column 0 = COUNTRY
# Column 1 = ISO3
# ------------------------------------------

indicator_values = indicator_data.iloc[5:].copy()

rows = []

for country, iso3 in comparison_countries.items():

    country_rows = indicator_values[
        indicator_values[1]
        .astype(str)
        .str.strip()
        .eq(iso3)
    ]

    if country_rows.empty:
        print("NOT FOUND:", country, iso3)
        continue

    country_row = country_rows.iloc[0]

    for _, meta in indicator_names.iterrows():

        inform_id = str(
            meta["INFORM Id"]
        ).strip()

        if inform_id not in id_to_column:
            continue

        col = id_to_column[inform_id]

        value = pd.to_numeric(
            country_row[col],
            errors="coerce"
        )

        rows.append({
            "COUNTRY": country,
            "ISO3": iso3,
            "Component": meta["Component"],
            "Indicator": meta["Indicator Name"],
            "INFORM_ID": inform_id,
            "Value": value
        })

indicator_comparison = pd.DataFrame(rows)

print(
    "\nRows extracted:",
    len(indicator_comparison)
)

print(
    "Countries:",
    indicator_comparison["COUNTRY"].unique()
)

display(
    indicator_comparison.round(3)
)

In [ ]:
# ==========================================
# MAP INFORM COPING INDICATORS
# METADATA ID -> INDICATOR DATA ID
# ==========================================

coping_id_map = {
    # Institutional / Governance
    "CC.INS.GOV.GE": "GovernmentEffectiveness",
    "CC.INS.GOV.CPI": "CPI",

    # Institutional / DRR
    "CC.INS.DRR.SG_DSR_LGRGSR": "SG_DSR_LGRGSR",
    "CC.INS.DRR.SG_DSR_LGRGSR_FREQ": "SG_DSR_LGRGSR_FREQ",

    # Communication
    "CC.INF.COM.LITR": "SE.ADT.LITR.ZS",
    "CC.INF.COM.ELACCS": "EG.ELC.ACCS.ZS",
    "CC.INF.COM.NETUS": "IT.NET.USER.P2",
    "CC.INF.COM.CEL": "IT.CEL.SETS.P2",

    # Physical connectivity
    "CC.INF.PHY.STA": "SH.STA.BASS.ZS",
    "CC.INF.PHY.H2O": "SH.H2O.SAFE.ZS",
    "CC.INF.PHY.ROD": "IS.ROD.TOTL.KM",

    # These two already match
    "SH.STA.BASS.ZS": "SH.STA.BASS.ZS",
    "SH.H2O.SAFE.ZS": "SH.H2O.SAFE.ZS",

    # Access to health care
    "CC.INF.AHC.HEALTH_EXP": "HEALTH_EXP",
    "CC.INF.AHC.IMM.DTP3": "SH_ACS_DTP3",
    "CC.INF.AHC.IMM.MCV2": "SH_ACS_MCV2",
    "CC.INF.AHC.IMM.PCV3": "SH_ACS_PCV3",
    "CC.INF.AHC.PHYS": "PHIS",
    "CC.INF.AHC.MMR": "MMR"
}

print(
    "Mapped indicators:",
    len(coping_id_map)
)

In [ ]:
# ==========================================
# EXTRACT ALL COPING-CAPACITY INDICATORS
# ==========================================

comparison_countries = {
    "Sierra Leone": "SLE",
    "Japan": "JPN",
    "Ecuador": "ECU",
    "Rwanda": "RWA"
}

indicator_names = (
    coping_metadata[
        ["INFORM Id", "Component", "Indicator Name"]
    ]
    .dropna(subset=["INFORM Id"])
    .copy()
)

indicator_values = indicator_data.iloc[5:].copy()

rows = []

for country, iso3 in comparison_countries.items():

    country_rows = indicator_values[
        indicator_values[1]
        .astype(str)
        .str.strip()
        .eq(iso3)
    ]

    if country_rows.empty:
        print("NOT FOUND:", country, iso3)
        continue

    country_row = country_rows.iloc[0]

    for _, meta in indicator_names.iterrows():

        metadata_id = str(
            meta["INFORM Id"]
        ).strip()

        data_id = coping_id_map.get(
            metadata_id
        )

        if data_id is None:
            continue

        # Find actual Indicator Data column
        matching_columns = [
            col for col in indicator_data.columns
            if str(
                indicator_data.iloc[3, col]
            ).strip() == data_id
        ]

        if not matching_columns:
            print(
                "DATA ID NOT FOUND:",
                data_id
            )
            continue

        col = matching_columns[0]

        value = pd.to_numeric(
            country_row[col],
            errors="coerce"
        )

        rows.append({
            "COUNTRY": country,
            "ISO3": iso3,
            "Component": meta["Component"],
            "Indicator": meta["Indicator Name"],
            "INFORM_ID": metadata_id,
            "Data_ID": data_id,
            "Value": value
        })

indicator_comparison = pd.DataFrame(rows)

print(
    "Rows extracted:",
    len(indicator_comparison)
)

print(
    "Countries:",
    indicator_comparison["COUNTRY"].unique()
)

display(
    indicator_comparison.round(3)
)

In [ ]:
# ==========================================
# STANDARDISE INDICATOR DIRECTION
# HIGHER = GREATER COPING CAPACITY
# ==========================================

higher_is_better = [
    "Government effectiveness",
    "Corruption Perception Index",
    "Sendai Framework Monitor indicator E-1",
    "Reporting frequecncy of Sendai Framework Monitor indicator E-1 in last 10 years",
    "Adult literacy rate",
    "Access to electricity",
    "Internet Users",
    "Mobile celluar subscriptions",
    "Improved sanitation facilities",
    "Improved water source",
    "Road density",
    "People using at least basic sanitation services (% of population)",
    "People using at least basic drinking water services (% of population)",
    "Current health expenditure per capita",
    "Coverage of DTP3 vaccine",
    "Coverage of measles-containing vaccine",
    "Coverage of pneumococcal conjugate vaccine",
    "Physicians density"
]

lower_is_better = [
    "Maternal Mortality Ratio"
]

# Check every indicator is classified
all_indicators = set(
    indicator_comparison["Indicator"]
)

classified = set(
    higher_is_better + lower_is_better
)

print("Total indicators:", len(all_indicators))
print("Classified indicators:", len(classified))

print("\nUnclassified:")
print(
    all_indicators - classified
)

# ------------------------------------------
# Create directional score
# ------------------------------------------

indicator_comparison["Directional_Value"] = (
    indicator_comparison["Value"]
)

indicator_comparison.loc[
    indicator_comparison["Indicator"].isin(
        lower_is_better
    ),
    "Directional_Value"
] = (
    -indicator_comparison.loc[
        indicator_comparison["Indicator"].isin(
            lower_is_better
        ),
        "Value"
    ]
)

display(
    indicator_comparison.round(3)
)

In [ ]:
# ==========================================
# NORMALISE INDICATORS
# BENCHMARK SET: SIERRA LEONE + JAPAN +
# ECUADOR + RWANDA
# HIGHER = BETTER
# ==========================================

normalised = indicator_comparison.copy()

normalised["Percentile"] = np.nan

for indicator in normalised["Indicator"].unique():

    mask = (
        normalised["Indicator"]
        == indicator
    )

    values = normalised.loc[
        mask,
        "Directional_Value"
    ]

    valid = values.dropna()

    if len(valid) < 2:
        continue

    # Percentile within the benchmark countries
    normalised.loc[
        mask & normalised["Directional_Value"].notna(),
        "Percentile"
    ] = (
        normalised.loc[
            mask & normalised["Directional_Value"].notna(),
            "Directional_Value"
        ].rank(
            method="average",
            pct=True
        ) * 100
    )

display(
    normalised.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE INDICATOR GAPS
# COMPARED WITH BEST BENCHMARK PERFORMANCE
# ==========================================

sl_indicators = normalised[
    normalised["COUNTRY"] == "Sierra Leone"
][
    [
        "Component",
        "Indicator",
        "Value",
        "Percentile"
    ]
].copy()

# Best benchmark percentile for each indicator
best_benchmark = (
    normalised
    .groupby(
        ["Component", "Indicator"],
        as_index=False
    )["Percentile"]
    .max()
    .rename(
        columns={
            "Percentile": "Best_Benchmark_Percentile"
        }
    )
)

sl_indicators = sl_indicators.merge(
    best_benchmark,
    on=["Component", "Indicator"],
    how="left"
)

sl_indicators["Gap_to_Best"] = (
    sl_indicators["Best_Benchmark_Percentile"]
    - sl_indicators["Percentile"]
)

sl_indicators = (
    sl_indicators
    .sort_values(
        "Gap_to_Best",
        ascending=False
    )
    .reset_index(drop=True)
)

# Rank from 1 = largest gap
sl_indicators.index = (
    sl_indicators.index + 1
)

sl_indicators.index.name = "Rank"

display(
    sl_indicators.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE COMPONENT BENCHMARK SCORE
# ==========================================

component_scores = (
    normalised
    .groupby(
        ["COUNTRY", "Component"],
        as_index=False
    )
    .agg(
        Mean_Percentile=("Percentile", "mean"),
        Indicators_Available=("Percentile", "count"),
        Indicators_Total=("Indicator", "count")
    )
)

# Sierra Leone only
sl_component_scores = component_scores[
    component_scores["COUNTRY"] == "Sierra Leone"
].copy()

# Rank: 1 = weakest component
sl_component_scores["Weakness_Rank"] = (
    sl_component_scores["Mean_Percentile"]
    .rank(
        ascending=True,
        method="min"
    )
)

sl_component_scores = (
    sl_component_scores
    .sort_values("Weakness_Rank")
    .reset_index(drop=True)
)

sl_component_scores.index = (
    sl_component_scores.index + 1
)

sl_component_scores.index.name = "Rank"

display(
    sl_component_scores.round(2)
)

In [ ]:
# ==========================================
# COMPONENT BENCHMARK:
# SIERRA LEONE VS JAPAN, ECUADOR, RWANDA
# ==========================================

component_comparison = (
    component_scores[
        component_scores["Component"] != "DRR implementation"
    ]
    .pivot(
        index="Component",
        columns="COUNTRY",
        values="Mean_Percentile"
    )
    .reset_index()
)

# Calculate Sierra Leone's gap versus each benchmark
for country in ["Japan", "Ecuador", "Rwanda"]:

    component_comparison[
        f"Gap_vs_{country.replace(' ', '_')}"
    ] = (
        component_comparison[country]
        - component_comparison["Sierra Leone"]
    )

# Average benchmark performance
component_comparison["Benchmark_Mean"] = (
    component_comparison[
        ["Japan", "Ecuador", "Rwanda"]
    ].mean(axis=1)
)

# Gap between Sierra Leone and average benchmark
component_comparison["Gap_vs_Benchmark_Mean"] = (
    component_comparison["Benchmark_Mean"]
    - component_comparison["Sierra Leone"]
)

# Largest gap first
component_comparison = (
    component_comparison
    .sort_values(
        "Gap_vs_Benchmark_Mean",
        ascending=False
    )
    .reset_index(drop=True)
)

component_comparison.index = (
    component_comparison.index + 1
)

component_comparison.index.name = "Rank"

display(
    component_comparison.round(2)
)

In [ ]:
# ==========================================
# SIERRA LEONE VS AVERAGE OF
# JAPAN + ECUADOR + RWANDA
# INDICATOR-LEVEL GAP
# ==========================================

benchmark_countries = [
    "Japan",
    "Ecuador",
    "Rwanda"
]

# Calculate benchmark mean for every indicator
indicator_benchmark = (
    normalised[
        normalised["COUNTRY"].isin(
            benchmark_countries
        )
    ]
    .groupby(
        ["Component", "Indicator"],
        as_index=False
    )
    .agg(
        Benchmark_Mean=(
            "Percentile",
            "mean"
        ),
        Benchmark_N=(
            "Percentile",
            "count"
        )
    )
)

# Sierra Leone
sl_indicator = normalised[
    normalised["COUNTRY"] == "Sierra Leone"
][
    [
        "Component",
        "Indicator",
        "Value",
        "Percentile"
    ]
].copy()

# Merge
indicator_priority = sl_indicator.merge(
    indicator_benchmark,
    on=["Component", "Indicator"],
    how="left"
)

# Gap: positive = Sierra Leone behind benchmark
indicator_priority["Gap_vs_Benchmark"] = (
    indicator_priority["Benchmark_Mean"]
    - indicator_priority["Percentile"]
)

# Sort largest gap first
indicator_priority = (
    indicator_priority
    .sort_values(
        "Gap_vs_Benchmark",
        ascending=False
    )
    .reset_index(drop=True)
)

indicator_priority.index = (
    indicator_priority.index + 1
)

indicator_priority.index.name = "Rank"

display(
    indicator_priority.round(2)
)

In [ ]:
# ==========================================
# COMPONENT PRIORITY EVIDENCE TABLE
# ==========================================

# INFORM component scores for Sierra Leone
inform_component_scores = pd.DataFrame({
    "Component": [
        "Governance",
        "Communication",
        "Physical Connectivity",
        "Access to health care"
    ],
    "INFORM_Score": [
        7.0,
        6.9,
        7.0,
        6.4
    ]
})

# Benchmark component gaps from previous analysis
benchmark_component_gaps = component_comparison[
    [
        "Component",
        "Sierra Leone",
        "Benchmark_Mean",
        "Gap_vs_Benchmark_Mean"
    ]
].copy()

benchmark_component_gaps = (
    benchmark_component_gaps
    .rename(
        columns={
            "Sierra Leone": "Benchmark_Percentile",
            "Benchmark_Mean": "Benchmark_Mean_Percentile"
        }
    )
)

# Merge
component_evidence = inform_component_scores.merge(
    benchmark_component_gaps,
    on="Component",
    how="left"
)

# Sort by benchmark gap
component_evidence = (
    component_evidence
    .sort_values(
        "Gap_vs_Benchmark_Mean",
        ascending=False
    )
    .reset_index(drop=True)
)

component_evidence.index = (
    component_evidence.index + 1
)

component_evidence.index.name = "Priority_Rank"

display(
    component_evidence.round(2)
)

In [ ]:
# ==========================================
# COPING-CAPACITY REGRESSION EVIDENCE
# ==========================================

regression_evidence = pd.DataFrame({
    "Component": [
        "Governance",
        "Institutional",
        "Physical infrastructure",
        "Infrastructure",
        "Access to health care",
        "Communication",
        "DRR"
    ],
    "R_squared": [
        0.3709,
        0.3634,
        0.3554,
        0.3532,
        0.3524,
        0.3515,
        0.2868
    ],
    "Component_p": [
        0.2138,
        0.3303,
        0.5709,
        0.7040,
        0.7775,
        0.9016,
        0.9516
    ],
    "Nested_p": [
        0.2138,
        0.3303,
        0.5709,
        0.7040,
        0.7775,
        0.9016,
        0.9516
    ]
})

display(
    regression_evidence.round(4)
)

In [ ]:
# ==========================================
# FINAL EVIDENCE MATRIX
# BUILD DIRECTLY FROM EXISTING RESULTS
# ==========================================

final_evidence = component_evidence.merge(
    regression_evidence,
    on="Component",
    how="left"
)

# Keep the four comparable components
final_evidence = final_evidence[
    final_evidence["Component"].isin([
        "Governance",
        "Communication",
        "Physical Connectivity",
        "Access to health care"
    ])
].copy()

# ------------------------------------------
# Evidence flags
# ------------------------------------------

final_evidence["High_INFORM_Weakness"] = (
    final_evidence["INFORM_Score"] >= 6.5
)

final_evidence["Large_Benchmark_Gap"] = (
    final_evidence["Gap_vs_Benchmark_Mean"] >= 30
)

final_evidence["Statistically_Significant"] = (
    final_evidence["Nested_p"] < 0.05
)

# ------------------------------------------
# Priority classification
# ------------------------------------------

final_evidence["Evidence_Category"] = "Lower priority"

final_evidence.loc[
    (
        final_evidence["High_INFORM_Weakness"]
        & final_evidence["Large_Benchmark_Gap"]
    ),
    "Evidence_Category"
] = "High structural priority"

# ------------------------------------------
# Sort by benchmark gap
# ------------------------------------------

final_evidence = (
    final_evidence
    .sort_values(
        "Gap_vs_Benchmark_Mean",
        ascending=False
    )
    .reset_index(drop=True)
)

final_evidence.index = (
    final_evidence.index + 1
)

final_evidence.index.name = "Rank"

display(
    final_evidence.round(4)
)

In [ ]:
# ==========================================
# MAP REGRESSION COMPONENTS TO
# BENCHMARK COMPONENTS
# ==========================================

regression_mapping = {
    "Physical infrastructure": "Physical Connectivity",
    "Infrastructure": "Physical Connectivity",
    "Governance": "Governance",
    "Communication": "Communication",
    "Access to health care": "Access to health care"
}

regression_mapped = regression_evidence.copy()

regression_mapped["Benchmark_Component"] = (
    regression_mapped["Component"]
    .map(regression_mapping)
)

# Remove components that don't map directly
regression_mapped = regression_mapped[
    regression_mapped["Benchmark_Component"].notna()
].copy()

# Display mapping
display(
    regression_mapped.round(4)
)

In [ ]:
# ==========================================
# FINAL COMPONENT EVIDENCE MATRIX
# ==========================================

# Map regression components to benchmark components
regression_mapping = {
    "Governance": "Governance",
    "Communication": "Communication",
    "Access to health care": "Access to health care",
    "Physical infrastructure": "Physical Connectivity",
    "Infrastructure": "Physical Connectivity"
}

regression_mapped = regression_evidence.copy()

regression_mapped["Benchmark_Component"] = (
    regression_mapped["Component"]
    .map(regression_mapping)
)

# Keep only components that map to our benchmark framework
regression_mapped = regression_mapped[
    regression_mapped["Benchmark_Component"].notna()
].copy()

# Summarise regression evidence
regression_summary = (
    regression_mapped
    .groupby(
        "Benchmark_Component",
        as_index=False
    )
    .agg(
        Mean_R_squared=("R_squared", "mean"),
        Best_Component_p=("Component_p", "min"),
        Best_Nested_p=("Nested_p", "min")
    )
    .rename(
        columns={
            "Benchmark_Component": "Component"
        }
    )
)

# Merge with structural + benchmark evidence
final_evidence_v2 = component_evidence.merge(
    regression_summary,
    on="Component",
    how="left"
)

# Statistical evidence flag
final_evidence_v2["Regression_Evidence"] = np.where(
    final_evidence_v2["Best_Nested_p"] < 0.05,
    "Significant incremental association",
    "No significant incremental association"
)

# Sort by benchmark gap
final_evidence_v2 = (
    final_evidence_v2
    .sort_values(
        "Gap_vs_Benchmark_Mean",
        ascending=False
    )
    .reset_index(drop=True)
)

# Rank from 1
final_evidence_v2.index = (
    final_evidence_v2.index + 1
)

final_evidence_v2.index.name = "Rank"

display(
    final_evidence_v2.round(4)
)

## 10. Investment Prioritisation


In [ ]:
# ==========================================
# SIERRA LEONE INVESTMENT PRIORITISATION
# ==========================================

investment_matrix = final_evidence_v2[
    [
        "Component",
        "INFORM_Score",
        "Gap_vs_Benchmark_Mean",
        "Mean_R_squared",
        "Best_Nested_p",
        "Regression_Evidence"
    ]
].copy()

# ------------------------------------------
# Structural weakness rank
# 1 = weakest
# ------------------------------------------

investment_matrix["INFORM_Weakness_Rank"] = (
    investment_matrix["INFORM_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# ------------------------------------------
# Benchmark gap rank
# 1 = largest gap
# ------------------------------------------

investment_matrix["Benchmark_Gap_Rank"] = (
    investment_matrix["Gap_vs_Benchmark_Mean"]
    .rank(
        ascending=False,
        method="min"
    )
)

# ------------------------------------------
# Combined structural priority
# Average of the two ranks
# ------------------------------------------

investment_matrix["Structural_Priority_Score"] = (
    investment_matrix["INFORM_Weakness_Rank"]
    + investment_matrix["Benchmark_Gap_Rank"]
) / 2

# Lower score = higher priority
investment_matrix = (
    investment_matrix
    .sort_values(
        "Structural_Priority_Score"
    )
    .reset_index(drop=True)
)

# Rank from 1
investment_matrix.index = (
    investment_matrix.index + 1
)

investment_matrix.index.name = "Priority"

display(
    investment_matrix.round(3)
)

In [ ]:
# ==========================================
# TOP INDICATORS WITHIN PRIORITY AREAS
# ==========================================

top_components = [
    "Governance",
    "Communication",
    "Physical Connectivity"
]

top_indicator_gaps = (
    indicator_priority[
        indicator_priority["Component"].isin(
            top_components
        )
    ]
    [
        [
            "Component",
            "Indicator",
            "Value",
            "Percentile",
            "Benchmark_Mean",
            "Gap_vs_Benchmark"
        ]
    ]
    .sort_values(
        ["Component", "Gap_vs_Benchmark"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(
    top_indicator_gaps.round(2)
)

In [ ]:
# ==========================================
# CONSOLIDATE OVERLAPPING INDICATORS
# ==========================================

indicator_priority_clean = indicator_priority.copy()

indicator_priority_clean["Evidence_Group"] = (
    indicator_priority_clean["Indicator"]
)

# Consolidate near-duplicate indicators
indicator_priority_clean.loc[
    indicator_priority_clean["Indicator"].isin([
        "Improved sanitation facilities",
        "People using at least basic sanitation services (% of population)"
    ]),
    "Evidence_Group"
] = "Sanitation access"

indicator_priority_clean.loc[
    indicator_priority_clean["Indicator"].isin([
        "Improved water source",
        "People using at least basic drinking water services (% of population)"
    ]),
    "Evidence_Group"
] = "Water access"

# Aggregate duplicate indicators
consolidated_priority = (
    indicator_priority_clean
    .groupby(
        ["Component", "Evidence_Group"],
        as_index=False
    )
    .agg(
        Sierra_Leone_Value=("Value", "mean"),
        Sierra_Leone_Percentile=("Percentile", "mean"),
        Benchmark_Mean=("Benchmark_Mean", "mean"),
        Gap_vs_Benchmark=("Gap_vs_Benchmark", "mean"),
        Indicators_Combined=("Indicator", "count")
    )
)

# Largest gap first
consolidated_priority = (
    consolidated_priority
    .sort_values(
        "Gap_vs_Benchmark",
        ascending=False
    )
    .reset_index(drop=True)
)

# Rank from 1
consolidated_priority.index = (
    consolidated_priority.index + 1
)

consolidated_priority.index.name = "Rank"

display(
    consolidated_priority.round(2)
)

In [ ]:
# ==========================================
# COMPONENT-LEVEL INVESTMENT EVIDENCE
# FROM CONSOLIDATED INDICATORS
# ==========================================

component_investment = (
    consolidated_priority[
        consolidated_priority["Component"] != "DRR implementation"
    ]
    .groupby(
        "Component",
        as_index=False
    )
    .agg(
        Mean_Gap=("Gap_vs_Benchmark", "mean"),
        Max_Gap=("Gap_vs_Benchmark", "max"),
        Indicators=("Evidence_Group", "count"),
        Severe_Gaps=(
            "Gap_vs_Benchmark",
            lambda x: (x >= 30).sum()
        )
    )
)

# Add original INFORM component score
component_investment = component_investment.merge(
    component_evidence[
        [
            "Component",
            "INFORM_Score",
            "Benchmark_Percentile"
        ]
    ],
    on="Component",
    how="left"
)

# Rank by number of severe gaps, then mean gap
component_investment = (
    component_investment
    .sort_values(
        ["Severe_Gaps", "Mean_Gap"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Rank from 1
component_investment.index = (
    component_investment.index + 1
)

component_investment.index.name = "Priority"

display(
    component_investment.round(2)
)

In [ ]:
# ==========================================
# NORMALISED COMPONENT SEVERITY
# ==========================================

component_investment["Severe_Gap_Rate"] = (
    component_investment["Severe_Gaps"]
    / component_investment["Indicators"]
)

# Rank by proportion of indicators showing
# a large benchmark gap, then mean gap
component_investment = (
    component_investment
    .sort_values(
        ["Severe_Gap_Rate", "Mean_Gap"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Rank from 1
component_investment.index = (
    component_investment.index + 1
)

component_investment.index.name = "Priority"

display(
    component_investment.round(3)
)

In [ ]:
# ==========================================
# FINAL STRUCTURAL PRIORITY SCORE
# ==========================================

final_priority = component_investment.copy()

# ------------------------------------------
# Convert each measure to 0-1
# Higher = greater structural priority
# ------------------------------------------

final_priority["INFORM_Priority"] = (
    final_priority["INFORM_Score"]
    / final_priority["INFORM_Score"].max()
)

final_priority["Benchmark_Priority"] = (
    final_priority["Mean_Gap"]
    / final_priority["Mean_Gap"].max()
)

final_priority["Severe_Gap_Priority"] = (
    final_priority["Severe_Gap_Rate"]
    / final_priority["Severe_Gap_Rate"].max()
)

# ------------------------------------------
# Equal-weight structural score
# ------------------------------------------

final_priority["Structural_Priority_Score"] = (
    final_priority["INFORM_Priority"]
    + final_priority["Benchmark_Priority"]
    + final_priority["Severe_Gap_Priority"]
) / 3

# ------------------------------------------
# Rank from 1 = highest priority
# ------------------------------------------

final_priority = (
    final_priority
    .sort_values(
        "Structural_Priority_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_priority.index = (
    final_priority.index + 1
)

final_priority.index.name = "Priority"

display(
    final_priority[
        [
            "Component",
            "INFORM_Score",
            "Mean_Gap",
            "Severe_Gap_Rate",
            "INFORM_Priority",
            "Benchmark_Priority",
            "Severe_Gap_Priority",
            "Structural_Priority_Score"
        ]
    ].round(3)
)

## 11. Hazard Prioritisation


In [ ]:
# ==========================================
# SIERRA LEONE NATURAL HAZARD PROFILE
# ==========================================

natural_hazard_variables = [
    "River Flood",
    "Drought",
    "Tropical Cyclone",
    "Epidemic",
    "Tsunami",
    "Earthquake",
    "Coastal flood"
]

hazard_profile = []

for hazard in natural_hazard_variables:

    values = pd.to_numeric(
        df_analysis_inform[hazard],
        errors="coerce"
    )

    sl_value = values.loc[
        df_analysis_inform["COUNTRY_NAME"] == "Sierra Leone"
    ].iloc[0]

    valid = values.dropna()

    percentile = (
        (valid <= sl_value).sum()
        / len(valid)
    ) * 100

    hazard_profile.append({
        "Hazard": hazard,
        "Sierra_Leone_Value": sl_value,
        "Africa_Percentile": percentile,
        "N": len(valid)
    })

hazard_profile = pd.DataFrame(
    hazard_profile
)

# Rank from highest exposure
hazard_profile = (
    hazard_profile
    .sort_values(
        "Sierra_Leone_Value",
        ascending=False
    )
    .reset_index(drop=True)
)

hazard_profile.index = (
    hazard_profile.index + 1
)

hazard_profile.index.name = "Rank"

display(
    hazard_profile.round(3)
)

In [ ]:
# ==========================================
# SIERRA LEONE HAZARD PRIORITY FRAMEWORK
# ==========================================

hazard_priority = hazard_profile.copy()

# Add Sierra Leone's INFORM dimensions
hazard_priority["Vulnerability"] = 5.1
hazard_priority["Lack_of_Coping_Capacity"] = 7.0

# ------------------------------------------
# Normalise hazard percentile
# ------------------------------------------

hazard_priority["Hazard_Pressure"] = (
    hazard_priority["Africa_Percentile"] / 100
)

# Higher vulnerability = greater concern
hazard_priority["Vulnerability_Pressure"] = (
    hazard_priority["Vulnerability"] / 10
)

# Higher lack of coping capacity = greater concern
hazard_priority["Coping_Pressure"] = (
    hazard_priority["Lack_of_Coping_Capacity"] / 10
)

# ------------------------------------------
# DO NOT combine into one score yet
# ------------------------------------------
# Instead create a transparent profile.

hazard_priority["Hazard_Category"] = pd.cut(
    hazard_priority["Africa_Percentile"],
    bins=[-1, 33.33, 66.67, 100],
    labels=[
        "Lower relative exposure",
        "Moderate relative exposure",
        "Higher relative exposure"
    ]
)

display(
    hazard_priority[
        [
            "Hazard",
            "Sierra_Leone_Value",
            "Africa_Percentile",
            "Hazard_Category",
            "Vulnerability",
            "Lack_of_Coping_Capacity"
        ]
    ].round(2)
)


In [ ]:
# ==========================================
# STAKEHOLDER HAZARD PRIORITY MATRIX
# ==========================================

stakeholder_hazards = hazard_priority.copy()

# ------------------------------------------
# Relative hazard priority
# ------------------------------------------

def classify_hazard(percentile):
    if percentile >= 75:
        return "High relative exposure"
    elif percentile >= 50:
        return "Moderate relative exposure"
    else:
        return "Lower relative exposure"

stakeholder_hazards["Relative_Hazard_Priority"] = (
    stakeholder_hazards["Africa_Percentile"]
    .apply(classify_hazard)
)

# ------------------------------------------
# Preparedness context
# ------------------------------------------

stakeholder_hazards["Coping_Context"] = np.where(
    stakeholder_hazards["Lack_of_Coping_Capacity"] >= 6.5,
    "High coping-capacity constraint",
    "Moderate coping-capacity constraint"
)

stakeholder_hazards["Vulnerability_Context"] = np.where(
    stakeholder_hazards["Vulnerability"] >= 5,
    "High vulnerability",
    "Moderate vulnerability"
)

# ------------------------------------------
# Stakeholder action category
# ------------------------------------------

stakeholder_hazards["Stakeholder_Action"] = np.select(
    [
        stakeholder_hazards["Africa_Percentile"] >= 75,
        stakeholder_hazards["Africa_Percentile"] >= 50
    ],
    [
        "Prioritise preparedness planning",
        "Maintain preparedness and contingency planning"
    ],
    default="Lower relative priority"
)

# ------------------------------------------
# Validation flag
# ------------------------------------------

stakeholder_hazards["Data_Validation_Flag"] = ""

stakeholder_hazards.loc[
    stakeholder_hazards["Hazard"] == "Tropical Cyclone",
    "Data_Validation_Flag"
] = "Validate indicator direction before interpretation"

# ------------------------------------------
# Display
# ------------------------------------------

display(
    stakeholder_hazards[
        [
            "Hazard",
            "Sierra_Leone_Value",
            "Africa_Percentile",
            "Relative_Hazard_Priority",
            "Vulnerability_Context",
            "Coping_Context",
            "Stakeholder_Action",
            "Data_Validation_Flag"
        ]
    ].round(2)
)

In [ ]:
# ==========================================
# HAZARD × CAPACITY INTERVENTION MATRIX
# ==========================================

hazard_capacity_map = {
    "Epidemic": [
        "Access to health care",
        "Communication",
        "Governance"
    ],
    "River Flood": [
        "Physical Connectivity",
        "Communication",
        "Governance",
        "Access to health care"
    ],
    "Coastal flood": [
        "Physical Connectivity",
        "Communication",
        "Governance"
    ],
    "Tsunami": [
        "Communication",
        "Physical Connectivity",
        "Governance"
    ],
    "Drought": [
        "Physical Connectivity",
        "Access to health care",
        "Governance"
    ],
    "Earthquake": [
        "Physical Connectivity",
        "Communication",
        "Governance"
    ],
    "Tropical Cyclone": [
        "Communication",
        "Physical Connectivity",
        "Governance"
    ]
}

hazard_capacity_rows = []

for _, row in stakeholder_hazards.iterrows():

    hazard = row["Hazard"]

    for component in hazard_capacity_map.get(
        hazard,
        []
    ):

        component_row = component_investment[
            component_investment["Component"] == component
        ]

        if component_row.empty:
            continue

        component_row = component_row.iloc[0]

        hazard_capacity_rows.append({
            "Hazard": hazard,
            "Hazard_Percentile": row["Africa_Percentile"],
            "Hazard_Priority": row[
                "Relative_Hazard_Priority"
            ],
            "Capacity_Component": component,
            "INFORM_Score": component_row[
                "INFORM_Score"
            ],
            "Benchmark_Gap": component_row[
                "Mean_Gap"
            ],
            "Severe_Gap_Rate": component_row[
                "Severe_Gap_Rate"
            ]
        })

hazard_capacity_matrix = pd.DataFrame(
    hazard_capacity_rows
)

display(
    hazard_capacity_matrix.round(2)
)

In [ ]:
# ==========================================
# HAZARD-LEVEL PREPAREDNESS PRIORITY
# ==========================================

hazard_priority_summary = (
    hazard_capacity_matrix
    .groupby(
        ["Hazard", "Hazard_Percentile", "Hazard_Priority"],
        as_index=False
    )
    .agg(
        Mean_Capacity_Gap=("Benchmark_Gap", "mean"),
        Max_Capacity_Gap=("Benchmark_Gap", "max"),
        Mean_Severe_Gap_Rate=("Severe_Gap_Rate", "mean"),
        High_Priority_Capacity_Gaps=(
            "Benchmark_Gap",
            lambda x: (x >= 30).sum()
        ),
        Capacity_Areas=("Capacity_Component", "count")
    )
)

# ------------------------------------------
# Normalised hazard pressure
# ------------------------------------------

hazard_priority_summary["Hazard_Pressure"] = (
    hazard_priority_summary["Hazard_Percentile"] / 100
)

# ------------------------------------------
# Normalised capacity weakness
# ------------------------------------------

hazard_priority_summary["Capacity_Weakness"] = (
    hazard_priority_summary["Mean_Capacity_Gap"]
    / hazard_priority_summary["Mean_Capacity_Gap"].max()
)

# ------------------------------------------
# Normalised severe-gap pressure
# ------------------------------------------

hazard_priority_summary["Severe_Gap_Pressure"] = (
    hazard_priority_summary["Mean_Severe_Gap_Rate"]
    / hazard_priority_summary["Mean_Severe_Gap_Rate"].max()
)

# ------------------------------------------
# Equal-weight preparedness priority
# ------------------------------------------

hazard_priority_summary["Preparedness_Priority_Score"] = (
    hazard_priority_summary["Hazard_Pressure"]
    + hazard_priority_summary["Capacity_Weakness"]
    + hazard_priority_summary["Severe_Gap_Pressure"]
) / 3

# ------------------------------------------
# Rank
# ------------------------------------------

hazard_priority_summary = (
    hazard_priority_summary
    .sort_values(
        "Preparedness_Priority_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

hazard_priority_summary.index = (
    hazard_priority_summary.index + 1
)

hazard_priority_summary.index.name = "Priority"

display(
    hazard_priority_summary[
        [
            "Hazard",
            "Hazard_Percentile",
            "Hazard_Priority",
            "Mean_Capacity_Gap",
            "Max_Capacity_Gap",
            "Mean_Severe_Gap_Rate",
            "High_Priority_Capacity_Gaps",
            "Preparedness_Priority_Score"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# HAZARD-SPECIFIC CAPACITY PRIORITIES
# ==========================================

hazard_capacity_detail = (
    hazard_capacity_matrix
    .copy()
)

# Score each capacity component within each hazard
hazard_capacity_detail["Capacity_Priority"] = (
    0.5 * (
        hazard_capacity_detail["Benchmark_Gap"]
        / hazard_capacity_detail["Benchmark_Gap"].max()
    )
    +
    0.5 * hazard_capacity_detail["Severe_Gap_Rate"]
)

# Rank capacity weaknesses within each hazard
hazard_capacity_detail["Within_Hazard_Rank"] = (
    hazard_capacity_detail
    .groupby("Hazard")["Capacity_Priority"]
    .rank(
        ascending=False,
        method="min"
    )
)

hazard_capacity_detail = (
    hazard_capacity_detail
    .sort_values(
        ["Hazard", "Within_Hazard_Rank"]
    )
    .reset_index(drop=True)
)

display(
    hazard_capacity_detail[
        [
            "Hazard",
            "Hazard_Percentile",
            "Capacity_Component",
            "INFORM_Score",
            "Benchmark_Gap",
            "Severe_Gap_Rate",
            "Capacity_Priority",
            "Within_Hazard_Rank"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# INTERVENTION-LEVEL EVIDENCE MATRIX
# ==========================================

intervention_matrix = consolidated_priority.copy()

# ------------------------------------------
# Define hazard-relevant intervention areas
# ------------------------------------------

hazard_intervention_map = {
    "Epidemic": [
        "Current health expenditure per capita",
        "Physicians density",
        "Coverage of DTP3 vaccine",
        "Coverage of measles-containing vaccine",
        "Coverage of pneumococcal conjugate vaccine",
        "Internet Users",
        "Adult literacy rate",
        "Government effectiveness"
    ],

    "River Flood": [
        "Sanitation access",
        "Water access",
        "Road density",
        "Internet Users",
        "Access to electricity",
        "Government effectiveness",
        "Current health expenditure per capita"
    ],

    "Coastal flood": [
        "Sanitation access",
        "Water access",
        "Road density",
        "Internet Users",
        "Access to electricity",
        "Government effectiveness"
    ],

    "Tsunami": [
        "Internet Users",
        "Access to electricity",
        "Adult literacy rate",
        "Road density",
        "Government effectiveness"
    ],

    "Earthquake": [
        "Road density",
        "Internet Users",
        "Access to electricity",
        "Government effectiveness",
        "Sanitation access"
    ],

    "Tropical Cyclone": [
        "Road density",
        "Internet Users",
        "Access to electricity",
        "Government effectiveness",
        "Sanitation access"
    ],

    "Drought": [
        "Water access",
        "Sanitation access",
        "Road density",
        "Current health expenditure per capita",
        "Government effectiveness"
    ]
}

# ------------------------------------------
# Build matrix
# ------------------------------------------

rows = []

for hazard, indicators in hazard_intervention_map.items():

    for indicator in indicators:

        match = intervention_matrix[
            intervention_matrix["Evidence_Group"] == indicator
        ]

        if match.empty:
            continue

        row = match.iloc[0]

        rows.append({
            "Hazard": hazard,
            "Intervention_Area": indicator,
            "Sierra_Leone_Value": row[
                "Sierra_Leone_Value"
            ],
            "Percentile": row[
                "Sierra_Leone_Percentile"
            ],
            "Benchmark_Mean": row[
                "Benchmark_Mean"
            ],
            "Gap_vs_Benchmark": row[
                "Gap_vs_Benchmark"
            ],
            "Indicators_Combined": row[
                "Indicators_Combined"
            ]
        })

intervention_matrix_final = pd.DataFrame(rows)

# ------------------------------------------
# Rank intervention gaps within each hazard
# ------------------------------------------

intervention_matrix_final[
    "Within_Hazard_Rank"
] = (
    intervention_matrix_final
    .groupby("Hazard")[
        "Gap_vs_Benchmark"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

intervention_matrix_final = (
    intervention_matrix_final
    .sort_values(
        ["Hazard", "Within_Hazard_Rank"]
    )
    .reset_index(drop=True)
)

display(
    intervention_matrix_final.round(2)
)

In [ ]:
# ==========================================
# TOP INTERVENTION GAPS BY HAZARD
# ==========================================

top_interventions = (
    intervention_matrix_final[
        intervention_matrix_final["Gap_vs_Benchmark"] >= 30
    ]
    .copy()
)

# Add hazard percentile from the original hazard profile
hazard_percentile_lookup = (
    stakeholder_hazards[
        ["Hazard", "Africa_Percentile"]
    ]
    .drop_duplicates("Hazard")
)

top_interventions = top_interventions.merge(
    hazard_percentile_lookup,
    on="Hazard",
    how="left"
)

top_interventions = (
    top_interventions
    .sort_values(
        [
            "Africa_Percentile",
            "Gap_vs_Benchmark"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

top_interventions = top_interventions[
    [
        "Hazard",
        "Africa_Percentile",
        "Intervention_Area",
        "Sierra_Leone_Value",
        "Percentile",
        "Benchmark_Mean",
        "Gap_vs_Benchmark",
        "Indicators_Combined"
    ]
]

display(
    top_interventions.round(2)
)

## 12. Structural Similarity & Machine Learning


In [ ]:
# ==========================================
# ML DATASET — INITIAL AUDIT
# ==========================================

ml_variables = [
    "COUNTRY_NAME",
    "ISO3",
    "GHSI",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "Natural",
    "Human",
    "VULNERABILITY",
    "LACK OF COPING CAPACITY",
    "GINI",
    "GDPpp",
    "Population"
]

available_ml_variables = [
    col for col in ml_variables
    if col in df_analysis_inform.columns
]

missing_ml_variables = [
    col for col in ml_variables
    if col not in df_analysis_inform.columns
]

print("AVAILABLE:")
print(available_ml_variables)

print("\nMISSING:")
print(missing_ml_variables)

In [ ]:
# ==========================================
# LOAD GLOBAL MASTER SHEET
# ==========================================

global_master = pd.read_excel(
    MASTER_DATA_FILE,
    sheet_name="GLOBAL MASTER SHEET"
)

print("Shape:", global_master.shape)
print("\nColumns:")
print(global_master.columns.tolist())

In [ ]:
# ==========================================
# LOAD GLOBAL MASTER SHEET
# ==========================================

global_master = pd.read_excel(
    MASTER_DATA_FILE,
    sheet_name="GLOBAL MASTER SHEET"
)

print("Shape:", global_master.shape)
print("\nColumns:")
print(global_master.columns.tolist())

display(global_master.head())

In [ ]:
# ==========================================
# GLOBAL ML FEATURE MATRIX
# ==========================================

cluster_variables = [
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION ",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "GINI"
]

# Add hazard/capacity variables if they exist
print("Columns available:")
print(global_master.columns.tolist())

# Check required variables
missing = [
    col for col in cluster_variables
    if col not in global_master.columns
]

print("\nMissing clustering variables:", missing)

In [ ]:
# Clean column names
global_master.columns = (
    global_master.columns
    .astype(str)
    .str.strip()
)

print(global_master.columns.tolist())

In [ ]:
# Check required variables
cluster_variables = [
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "WGI_SCORE",
    "INFORM_RISK_SCORE",
    "GINI"
]

missing = [
    col for col in cluster_variables
    if col not in global_master.columns
]

print("Missing clustering variables:", missing)

In [ ]:
# ==========================================
# GLOBAL ML DATASET
# ==========================================

global_ml = global_master[
    [
        "ISO3",
        "REGION",
        "COUNTRY_NAME",
        "GHSI",
        "INFORM_RISK_SCORE",
        "WGI_SCORE",
        "GDP_PER_CAPITA",
        "TOTAL_POPULATION",
        "GINI",
        "DISASTER COUNT",
        "DEATHS FROM DISASTER COUNT",
        "TOTAL PEOPLE AFFECTED BY DISASTER"
    ]
].copy()

# Convert quantitative variables to numeric
numeric_variables = [
    "GHSI",
    "INFORM_RISK_SCORE",
    "WGI_SCORE",
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "GINI",
    "DISASTER COUNT",
    "DEATHS FROM DISASTER COUNT",
    "TOTAL PEOPLE AFFECTED BY DISASTER"
]

for col in numeric_variables:
    global_ml[col] = pd.to_numeric(
        global_ml[col],
        errors="coerce"
    )

print("Global ML dataset shape:", global_ml.shape)

display(global_ml.head())

In [ ]:
# ==========================================
# GLOBAL ML SAMPLE AUDIT
# ==========================================

cluster_features = [
    "INFORM_RISK_SCORE",
    "WGI_SCORE",
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "GINI"
]

ml_sample = global_ml.dropna(
    subset=cluster_features
).copy()

print("Total countries:", len(global_ml))
print("Complete ML sample:", len(ml_sample))
print("Countries excluded:", len(global_ml) - len(ml_sample))

print("\nRegions represented:")
print(ml_sample["REGION"].value_counts())

display(
    ml_sample[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION"
        ] + cluster_features
    ].head(20)
)

In [ ]:
# ==========================================
# CHECK KEY BENCHMARK COUNTRIES
# ==========================================

benchmark_countries = [
    "Sierra Leone",
    "Japan",
    "Switzerland",
    "Ecuador",
    "Singapore",
    "China",
    "Australia",
    "Chile",
    "Mauritius",
    "Netherlands",
    "New Zealand",
    "Rwanda"
]

benchmark_check = ml_sample[
    ml_sample["COUNTRY_NAME"].isin(benchmark_countries)
][
    ["COUNTRY_NAME"] + cluster_features
]

display(benchmark_check)

In [ ]:
# ==========================================
# STANDARDISE GLOBAL CLUSTER FEATURES
# ==========================================

from sklearn.preprocessing import StandardScaler

cluster_features = [
    "INFORM_RISK_SCORE",
    "WGI_SCORE",
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "GINI"
]

X_cluster = ml_sample[cluster_features].copy()

# Standardise
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_cluster)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=cluster_features,
    index=ml_sample.index
)

print("Scaled feature matrix:", X_scaled.shape)

display(X_scaled.head())

In [ ]:
# ==========================================
# PCA — GLOBAL STRUCTURE
# ==========================================

from sklearn.decomposition import PCA
import numpy as np

pca = PCA()

X_pca = pca.fit_transform(X_scaled)

explained_variance = pd.DataFrame({
    "Component": [
        f"PC{i+1}"
        for i in range(len(pca.explained_variance_ratio_))
    ],
    "Explained_Variance_%": (
        pca.explained_variance_ratio_ * 100
    ),
    "Cumulative_Variance_%": (
        np.cumsum(pca.explained_variance_ratio_) * 100
    )
})

display(
    explained_variance.round(2)
)

In [ ]:
# ==========================================
# PCA LOADINGS
# ==========================================

display(
    loadings.round(3)
)

In [ ]:
# ==========================================
# PCA LOADINGS
# ==========================================

loadings = pd.DataFrame(
    pca.components_.T,
    index=cluster_features,
    columns=[
        f"PC{i+1}"
        for i in range(len(cluster_features))
    ]
)

display(
    loadings.round(3)
)

In [ ]:
# ==========================================
# RETAIN 3 PCA COMPONENTS
# ==========================================

X_pca_3 = pd.DataFrame(
    X_pca[:, :3],
    columns=["PC1", "PC2", "PC3"],
    index=ml_sample.index
)

# Attach country information
pca_countries = pd.concat(
    [
        ml_sample[
            [
                "ISO3",
                "COUNTRY_NAME",
                "REGION",
                "GHSI"
            ]
        ],
        X_pca_3
    ],
    axis=1
)

print("PCA dataset:", pca_countries.shape)

display(
    pca_countries.head()
)

In [ ]:
# ==========================================
# CREATE 3-COMPONENT PCA DATASET
# ==========================================

X_pca_3 = pd.DataFrame(
    X_pca[:, :3],
    columns=["PC1", "PC2", "PC3"],
    index=ml_sample.index
)

pca_countries = pd.concat(
    [
        ml_sample[
            ["ISO3", "COUNTRY_NAME", "REGION", "GHSI"]
        ],
        X_pca_3
    ],
    axis=1
)

print("PCA dataset shape:", pca_countries.shape)

display(
    pca_countries.head()
)

In [ ]:
# ==========================================
# GLOBAL HDBSCAN CLUSTERING
# ==========================================

from sklearn.cluster import HDBSCAN

cluster_input = pca_countries[
    ["PC1", "PC2", "PC3"]
].copy()

hdbscan_model = HDBSCAN(
    min_cluster_size=8,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom"
)

pca_countries["Cluster"] = hdbscan_model.fit_predict(
    cluster_input
)

print("Clusters found:")
print(
    pca_countries["Cluster"]
    .value_counts()
    .sort_index()
)

display(
    pca_countries[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "PC1",
            "PC2",
            "PC3",
            "Cluster"
        ]
    ]
    .sort_values(["Cluster", "COUNTRY_NAME"])
    .head(30)
)

In [ ]:
# ==========================================
# FIND OPTIMAL NUMBER OF GLOBAL CLUSTERS
# ==========================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_input = pca_countries[
    ["PC1", "PC2", "PC3"]
].copy()

results = []

for k in range(2, 9):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(cluster_input)

    silhouette = silhouette_score(
        cluster_input,
        labels
    )

    results.append({
        "K": k,
        "Silhouette_Score": silhouette
    })

cluster_scores = pd.DataFrame(results)

display(
    cluster_scores.round(3)
)

In [ ]:
# ==========================================
# FINAL GLOBAL K-MEANS CLUSTERING
# ==========================================

from sklearn.cluster import KMeans

kmeans_final = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=50
)

pca_countries["Cluster"] = kmeans_final.fit_predict(
    cluster_input
)

print("Cluster sizes:")
print(
    pca_countries["Cluster"]
    .value_counts()
    .sort_index()
)

display(
    pca_countries[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "PC1",
            "PC2",
            "PC3",
            "Cluster"
        ]
    ]
    .sort_values(["Cluster", "COUNTRY_NAME"])
)

In [ ]:
# ==========================================
# PROFILE THE 3 GLOBAL STRUCTURAL CLUSTERS
# ==========================================

cluster_profile = (
    pca_countries
    .groupby("Cluster")
    .agg(
        Countries=("COUNTRY_NAME", "count"),
        Mean_GHSI=("GHSI", "mean"),
        Mean_PC1=("PC1", "mean"),
        Mean_PC2=("PC2", "mean"),
        Mean_PC3=("PC3", "mean")
    )
    .round(3)
)

display(cluster_profile)

In [ ]:
# ==========================================
# CLUSTER PROFILE — ORIGINAL VARIABLES
# ==========================================

cluster_profile_original = (
    ml_sample
    .copy()
    .join(
        pca_countries["Cluster"],
        how="inner"
    )
    .groupby("Cluster")[
        [
            "INFORM_RISK_SCORE",
            "WGI_SCORE",
            "GDP_PER_CAPITA",
            "TOTAL_POPULATION",
            "GINI",
            "GHSI"
        ]
    ]
    .mean()
    .round(2)
)

display(cluster_profile_original)

## 13. Structural Peer & Benchmark Analysis


In [ ]:
# ==========================================
# SIERRA LEONE'S STRUCTURAL PEERS
# ==========================================

sierra_cluster = pca_countries.loc[
    pca_countries["COUNTRY_NAME"] == "Sierra Leone",
    "Cluster"
].iloc[0]

sierra_peers = pca_countries[
    pca_countries["Cluster"] == sierra_cluster
].copy()

print("Sierra Leone cluster:", sierra_cluster)
print("Number of countries:", len(sierra_peers))

display(
    sierra_peers[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "PC1",
            "PC2",
            "PC3",
            "Cluster"
        ]
    ]
    .sort_values("GHSI")
    .round(3)
)

In [ ]:
# ==========================================
# KNN — SIERRA LEONE'S CLOSEST STRUCTURAL PEERS
# ==========================================

from sklearn.neighbors import NearestNeighbors

# Original standardised feature space
knn_features = X_scaled.copy()

# Sierra Leone row
sierra_idx = ml_sample.index[
    ml_sample["COUNTRY_NAME"] == "Sierra Leone"
][0]

# Fit nearest-neighbour model
knn = NearestNeighbors(
    n_neighbors=11,
    metric="euclidean"
)

knn.fit(knn_features)

# Find Sierra Leone + 10 nearest countries
distances, indices = knn.kneighbors(
    knn_features.loc[[sierra_idx]]
)

nearest_indices = indices[0]
nearest_distances = distances[0]

knn_results = ml_sample.loc[
    nearest_indices,
    [
        "ISO3",
        "COUNTRY_NAME",
        "REGION",
        "GHSI",
        "INFORM_RISK_SCORE",
        "WGI_SCORE",
        "GDP_PER_CAPITA",
        "TOTAL_POPULATION",
        "GINI"
    ]
].copy()

knn_results["Distance_from_Sierra_Leone"] = nearest_distances

# Remove Sierra Leone itself
knn_results = knn_results[
    knn_results["COUNTRY_NAME"] != "Sierra Leone"
].copy()

knn_results = (
    knn_results
    .sort_values("Distance_from_Sierra_Leone")
    .reset_index(drop=True)
)

knn_results.index = knn_results.index + 1

display(
    knn_results.round(3)
)

In [ ]:
# ==========================================
# KNN — SIERRA LEONE'S CLOSEST STRUCTURAL PEERS
# ==========================================

from sklearn.neighbors import NearestNeighbors

# Features already standardised
knn_features = X_scaled.copy()

# Find Sierra Leone's POSITION in the ML dataset
sierra_position = np.where(
    ml_sample["COUNTRY_NAME"].values == "Sierra Leone"
)[0][0]

# Fit KNN
knn = NearestNeighbors(
    n_neighbors=11,
    metric="euclidean"
)

knn.fit(knn_features)

# Find Sierra Leone + 10 nearest countries
distances, indices = knn.kneighbors(
    knn_features[sierra_position].reshape(1, -1)
)

nearest_positions = indices[0]
nearest_distances = distances[0]

# IMPORTANT:
# indices are positional -> use iloc, NOT loc
knn_results = ml_sample.iloc[
    nearest_positions
].copy()

# Add distance
knn_results["Distance_from_Sierra_Leone"] = (
    nearest_distances
)

# Remove Sierra Leone itself
knn_results = knn_results[
    knn_results["COUNTRY_NAME"] != "Sierra Leone"
].copy()

# Keep relevant variables
knn_results = knn_results[
    [
        "ISO3",
        "COUNTRY_NAME",
        "REGION",
        "GHSI",
        "INFORM_RISK_SCORE",
        "WGI_SCORE",
        "GDP_PER_CAPITA",
        "TOTAL_POPULATION",
        "GINI",
        "Distance_from_Sierra_Leone"
    ]
]

knn_results = (
    knn_results
    .sort_values("Distance_from_Sierra_Leone")
    .reset_index(drop=True)
)

knn_results.index = knn_results.index + 1

display(
    knn_results.round(3)
)

In [ ]:
# ==========================================
# CLEAN KNN — SIERRA LEONE STRUCTURAL PEERS
# ==========================================

import numpy as np
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------
# 1. Convert feature matrix to NumPy
# ------------------------------------------

X_knn = np.asarray(X_scaled)

print("X_knn shape:", X_knn.shape)


# ------------------------------------------
# 2. Find Sierra Leone's row POSITION
# ------------------------------------------

country_array = ml_sample["COUNTRY_NAME"].astype(str).to_numpy()

sierra_positions = np.where(
    country_array == "Sierra Leone"
)[0]

if len(sierra_positions) == 0:
    raise ValueError("Sierra Leone not found in ml_sample.")

sierra_position = int(sierra_positions[0])

print("Sierra Leone position:", sierra_position)


# ------------------------------------------
# 3. Check that dimensions match
# ------------------------------------------

if len(ml_sample) != X_knn.shape[0]:
    raise ValueError(
        f"Row mismatch: ml_sample has {len(ml_sample)} rows "
        f"but X_scaled has {X_knn.shape[0]} rows."
    )


# ------------------------------------------
# 4. Sierra Leone feature vector
# ------------------------------------------

sierra_vector = X_knn[sierra_position].reshape(1, -1)

print(
    "Sierra Leone vector shape:",
    sierra_vector.shape
)


# ------------------------------------------
# 5. Fit KNN
# ------------------------------------------

knn = NearestNeighbors(
    n_neighbors=min(11, len(ml_sample)),
    metric="euclidean"
)

knn.fit(X_knn)


# ------------------------------------------
# 6. Find nearest countries
# ------------------------------------------

distances, indices = knn.kneighbors(
    sierra_vector
)

nearest_positions = indices[0]
nearest_distances = distances[0]


# ------------------------------------------
# 7. Extract countries
# ------------------------------------------

knn_results = ml_sample.iloc[
    nearest_positions
].copy()

knn_results["Distance_from_Sierra_Leone"] = (
    nearest_distances
)


# ------------------------------------------
# 8. Remove Sierra Leone
# ------------------------------------------

knn_results = knn_results[
    knn_results["COUNTRY_NAME"] != "Sierra Leone"
].copy()


# ------------------------------------------
# 9. Keep relevant variables
# ------------------------------------------

wanted_columns = [
    "ISO3",
    "COUNTRY_NAME",
    "REGION",
    "GHSI",
    "INFORM_RISK_SCORE",
    "WGI_SCORE",
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "GINI",
    "Distance_from_Sierra_Leone"
]

available_columns = [
    col for col in wanted_columns
    if col in knn_results.columns
]

knn_results = knn_results[
    available_columns
]


# ------------------------------------------
# 10. Rank nearest peers
# ------------------------------------------

knn_results = (
    knn_results
    .sort_values(
        "Distance_from_Sierra_Leone"
    )
    .reset_index(drop=True)
)

knn_results.insert(
    0,
    "Peer_Rank",
    np.arange(1, len(knn_results) + 1)
)


# ------------------------------------------
# FINAL RESULT
# ------------------------------------------

display(
    knn_results.round(3)
)

In [ ]:
# ==========================================
# SIERRA LEONE BASELINE
# ==========================================

sierra_leone = ml_sample[
    ml_sample["COUNTRY_NAME"] == "Sierra Leone"
].copy()

display(
    sierra_leone[
        [
            "ISO3",
            "COUNTRY_NAME",
            "GHSI",
            "INFORM_RISK_SCORE",
            "WGI_SCORE",
            "GDP_PER_CAPITA",
            "TOTAL_POPULATION",
            "GINI"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# PEER PERFORMANCE GAP
# ==========================================

# Get Sierra Leone's GHSI
sl_ghsi = float(
    sierra_leone["GHSI"].iloc[0]
)

# Compare every KNN peer against Sierra Leone
knn_results["GHSI_Gap_vs_Sierra_Leone"] = (
    knn_results["GHSI"] - sl_ghsi
)

# Classify benchmark potential
knn_results["Benchmark_Type"] = np.select(
    [
        knn_results["GHSI_Gap_vs_Sierra_Leone"] > 5,
        knn_results["GHSI_Gap_vs_Sierra_Leone"] >= 0,
        knn_results["GHSI_Gap_vs_Sierra_Leone"] < 0
    ],
    [
        "Potential positive benchmark",
        "Comparable / modest positive benchmark",
        "Learning / risk peer"
    ],
    default="Unknown"
)

display(
    knn_results[
        [
            "Peer_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone",
            "INFORM_RISK_SCORE",
            "WGI_SCORE",
            "Benchmark_Type"
        ]
    ].round(3)
)

## 14. Positive Deviance & Benchmark Selection


In [ ]:
# ==========================================
# GLOBAL POSITIVE BENCHMARK SEARCH
# ==========================================

# Sierra Leone GHSI
sl_ghsi = float(
    sierra_leone["GHSI"].iloc[0]
)

# Calculate distance from Sierra Leone
all_distances = np.linalg.norm(
    X_knn - X_knn[sierra_position],
    axis=1
)

benchmark_search = ml_sample.copy()

benchmark_search[
    "Distance_from_Sierra_Leone"
] = all_distances

benchmark_search[
    "GHSI_Gap_vs_Sierra_Leone"
] = (
    benchmark_search["GHSI"] - sl_ghsi
)

# Remove Sierra Leone
benchmark_search = benchmark_search[
    benchmark_search["COUNTRY_NAME"] != "Sierra Leone"
].copy()

# Only countries that outperform Sierra Leone
positive_benchmarks = benchmark_search[
    benchmark_search["GHSI_Gap_vs_Sierra_Leone"] > 0
].copy()

# Rank by structural similarity first
positive_benchmarks = (
    positive_benchmarks
    .sort_values(
        [
            "Distance_from_Sierra_Leone",
            "GHSI_Gap_vs_Sierra_Leone"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

positive_benchmarks.insert(
    0,
    "Benchmark_Rank",
    np.arange(
        1,
        len(positive_benchmarks) + 1
    )
)

display(
    positive_benchmarks[
        [
            "Benchmark_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone",
            "INFORM_RISK_SCORE",
            "WGI_SCORE",
            "GDP_PER_CAPITA",
            "TOTAL_POPULATION",
            "GINI"
        ]
    ].head(20).round(3)
)

In [ ]:
# ==========================================
# BENCHMARK SUITABILITY SCORE
# ==========================================

benchmark_df = positive_benchmarks.copy()

# ------------------------------------------
# 1. Similarity score
# ------------------------------------------
# Higher = more structurally similar

benchmark_df["Similarity_Score"] = (
    1 / (1 + benchmark_df["Distance_from_Sierra_Leone"])
)


# ------------------------------------------
# 2. Performance improvement
# ------------------------------------------
# Cap extreme values so one country does not
# dominate purely because of a huge GHSI gap.

benchmark_df["Performance_Score"] = (
    benchmark_df["GHSI_Gap_vs_Sierra_Leone"]
    .clip(lower=0, upper=20)
    / 20
)


# ------------------------------------------
# 3. African relevance
# ------------------------------------------

benchmark_df["African_Relevance"] = np.where(
    benchmark_df["REGION"].eq("Africa"),
    1.0,
    0.0
)


# ------------------------------------------
# 4. Combine dimensions
# ------------------------------------------
#
# Transparent weights:
#
# 50% structural similarity
# 30% performance advantage
# 20% African relevance
#
# These are analytical weights, not empirical
# estimates.

benchmark_df["Benchmark_Suitability"] = (
    0.50 * benchmark_df["Similarity_Score"]
    + 0.30 * benchmark_df["Performance_Score"]
    + 0.20 * benchmark_df["African_Relevance"]
)


# ------------------------------------------
# 5. Rank
# ------------------------------------------

benchmark_df = (
    benchmark_df
    .sort_values(
        "Benchmark_Suitability",
        ascending=False
    )
    .reset_index(drop=True)
)

benchmark_df.insert(
    0,
    "Benchmark_Priority",
    np.arange(
        1,
        len(benchmark_df) + 1
    )
)


# ------------------------------------------
# 6. Display
# ------------------------------------------

display(
    benchmark_df[
        [
            "Benchmark_Priority",
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone",
            "Similarity_Score",
            "Performance_Score",
            "African_Relevance",
            "Benchmark_Suitability"
        ]
    ].head(20).round(3)
)

In [ ]:
# ==========================================
# BENCHMARK LENSES
# ==========================================

benchmark_lenses = positive_benchmarks.copy()

# ------------------------------------------
# 1. STRUCTURAL PEERS
# ------------------------------------------

structural_peers = (
    benchmark_lenses
    .sort_values(
        "Distance_from_Sierra_Leone"
    )
    .reset_index(drop=True)
)

structural_peers.insert(
    0,
    "Structural_Rank",
    np.arange(1, len(structural_peers) + 1)
)


# ------------------------------------------
# 2. PERFORMANCE BENCHMARKS
# ------------------------------------------

performance_benchmarks = (
    benchmark_lenses
    .sort_values(
        "GHSI_Gap_vs_Sierra_Leone",
        ascending=False
    )
    .reset_index(drop=True)
)

performance_benchmarks.insert(
    0,
    "Performance_Rank",
    np.arange(
        1,
        len(performance_benchmarks) + 1
    )
)


# ------------------------------------------
# 3. AFRICAN BENCHMARKS
# ------------------------------------------

african_benchmarks = benchmark_lenses[
    benchmark_lenses["REGION"] == "Africa"
].copy()

african_benchmarks = (
    african_benchmarks
    .sort_values(
        [
            "Distance_from_Sierra_Leone",
            "GHSI_Gap_vs_Sierra_Leone"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

african_benchmarks.insert(
    0,
    "African_Benchmark_Rank",
    np.arange(
        1,
        len(african_benchmarks) + 1
    )
)


# ------------------------------------------
# DISPLAY
# ------------------------------------------

print("STRUCTURAL PEERS")
display(
    structural_peers[
        [
            "Structural_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone"
        ]
    ].head(15).round(3)
)

print("\nAFRICAN BENCHMARKS")
display(
    african_benchmarks[
        [
            "African_Benchmark_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone"
        ]
    ].head(15).round(3)
)

print("\nPERFORMANCE BENCHMARKS")
display(
    performance_benchmarks[
        [
            "Performance_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone"
        ]
    ].head(15).round(3)
)

In [ ]:
# ==========================================
# SIERRA LEONE BENCHMARK LADDER
# ==========================================

benchmark_ladder = positive_benchmarks.copy()

# Structural similarity categories
benchmark_ladder["Structural_Category"] = np.select(
    [
        benchmark_ladder["Distance_from_Sierra_Leone"] <= 1.0,
        benchmark_ladder["Distance_from_Sierra_Leone"] <= 1.5,
        benchmark_ladder["Distance_from_Sierra_Leone"] <= 2.5
    ],
    [
        "Very close structural peer",
        "Close structural peer",
        "Moderate structural similarity"
    ],
    default="Distant structural comparator"
)

# Benchmark role
benchmark_ladder["Benchmark_Role"] = np.select(
    [
        (
            (benchmark_ladder["Distance_from_Sierra_Leone"] <= 1.5)
            &
            (benchmark_ladder["GHSI_Gap_vs_Sierra_Leone"] > 0)
        ),
        (
            (benchmark_ladder["REGION"] == "Africa")
            &
            (benchmark_ladder["GHSI_Gap_vs_Sierra_Leone"] > 0)
        ),
        (
            benchmark_ladder["GHSI_Gap_vs_Sierra_Leone"] >= 15
        ),
    ],
    [
        "Direct positive peer",
        "African positive benchmark",
        "Aspirational benchmark"
    ],
    default="International comparator"
)

# Structural peers that underperform Sierra Leone
all_peer_data = benchmark_search.copy()

risk_peers = all_peer_data[
    (
        all_peer_data["Distance_from_Sierra_Leone"] <= 1.5
    )
    &
    (
        all_peer_data["GHSI_Gap_vs_Sierra_Leone"] < 0
    )
].copy()

risk_peers["Benchmark_Role"] = "Structural learning / risk peer"

# Display positive benchmark ladder
display(
    benchmark_ladder[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone",
            "Structural_Category",
            "Benchmark_Role"
        ]
    ]
    .sort_values(
        [
            "Distance_from_Sierra_Leone",
            "GHSI_Gap_vs_Sierra_Leone"
        ],
        ascending=[True, False]
    )
    .head(30)
    .round(3)
)

print("\nSTRUCTURAL LEARNING / RISK PEERS")

display(
    risk_peers[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "Distance_from_Sierra_Leone",
            "GHSI",
            "GHSI_Gap_vs_Sierra_Leone",
            "Benchmark_Role"
        ]
    ]
    .sort_values("Distance_from_Sierra_Leone")
    .head(15)
    .round(3)
)

In [ ]:
# ==========================================
# POSITIVE DEVIANCE ANALYSIS
# ==========================================

import statsmodels.api as sm
import numpy as np
import pandas as pd

# ------------------------------------------
# 1. Variables
# ------------------------------------------

pd_features = [
    "INFORM_RISK_SCORE",
    "WGI_SCORE",
    "GDP_PER_CAPITA",
    "TOTAL_POPULATION",
    "GINI"
]

pd_data = ml_sample[
    ["ISO3", "COUNTRY_NAME", "REGION", "GHSI"] + pd_features
].copy()

# ------------------------------------------
# 2. Log-transform skewed variables
# ------------------------------------------

pd_data["LOG_GDP"] = np.log1p(
    pd_data["GDP_PER_CAPITA"]
)

pd_data["LOG_POPULATION"] = np.log1p(
    pd_data["TOTAL_POPULATION"]
)

model_features = [
    "INFORM_RISK_SCORE",
    "WGI_SCORE",
    "LOG_GDP",
    "LOG_POPULATION",
    "GINI"
]

pd_model = pd_data.dropna(
    subset=["GHSI"] + model_features
).copy()

# ------------------------------------------
# 3. OLS model
# ------------------------------------------

X = pd_model[model_features]

X = sm.add_constant(X)

y = pd_model["GHSI"]

model = sm.OLS(
    y,
    X
).fit()

print(model.summary())

# ------------------------------------------
# 4. Predicted GHSI
# ------------------------------------------

pd_model["Predicted_GHSI"] = model.predict(X)

# ------------------------------------------
# 5. Positive-deviance residual
# ------------------------------------------

pd_model["GHSI_Residual"] = (
    pd_model["GHSI"]
    - pd_model["Predicted_GHSI"]
)

# ------------------------------------------
# 6. Standardised residual
# ------------------------------------------

pd_model["Residual_Z"] = (
    pd_model["GHSI_Residual"]
    - pd_model["GHSI_Residual"].mean()
) / pd_model["GHSI_Residual"].std()

# ------------------------------------------
# 7. Classify
# ------------------------------------------

pd_model["Performance_Type"] = np.select(
    [
        pd_model["Residual_Z"] >= 1.0,
        pd_model["Residual_Z"] <= -1.0
    ],
    [
        "Positive deviant",
        "Negative deviant"
    ],
    default="Expected-range performance"
)

# ------------------------------------------
# 8. Rank positive deviants
# ------------------------------------------

positive_deviants = (
    pd_model[
        pd_model["Performance_Type"]
        == "Positive deviant"
    ]
    .sort_values(
        "GHSI_Residual",
        ascending=False
    )
    .reset_index(drop=True)
)

positive_deviants.insert(
    0,
    "Positive_Deviance_Rank",
    np.arange(
        1,
        len(positive_deviants) + 1
    )
)

# ------------------------------------------
# 9. Display
# ------------------------------------------

display(
    positive_deviants[
        [
            "Positive_Deviance_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "Predicted_GHSI",
            "GHSI_Residual",
            "Residual_Z",
            "INFORM_RISK_SCORE",
            "WGI_SCORE",
            "GDP_PER_CAPITA",
            "Performance_Type"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# SIERRA LEONE — POSITIVE DEVIANCE POSITION
# ==========================================

display(
    pd_model[
        pd_model["COUNTRY_NAME"]
        == "Sierra Leone"
    ][
        [
            "COUNTRY_NAME",
            "GHSI",
            "Predicted_GHSI",
            "GHSI_Residual",
            "Residual_Z",
            "Performance_Type"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# POSITIVE DEVIANCE + STRUCTURAL SIMILARITY
# ==========================================

# Start with the positive-deviance model results
pd_benchmark = pd_model.copy()

# Merge / attach KNN distance information
pd_benchmark = pd_benchmark.merge(
    knn_results[
        [
            "ISO3",
            "Distance_from_Sierra_Leone"
        ]
    ],
    on="ISO3",
    how="left"
)

# African relevance
pd_benchmark["African_Relevance"] = np.where(
    pd_benchmark["REGION"] == "Africa",
    1,
    0
)

# Structural similarity score
pd_benchmark["Structural_Similarity"] = (
    1 /
    (1 + pd_benchmark["Distance_from_Sierra_Leone"])
)

# Positive-deviance score
pd_benchmark["Positive_Deviance_Score"] = (
    pd_benchmark["GHSI_Residual"]
    .clip(lower=0)
)

# Combined MEL benchmark score
pd_benchmark["MEL_Benchmark_Score"] = (
    0.40 * pd_benchmark["Structural_Similarity"]
    +
    0.40 * (
        pd_benchmark["Positive_Deviance_Score"]
        /
        pd_benchmark["Positive_Deviance_Score"].max()
    )
    +
    0.20 * pd_benchmark["African_Relevance"]
)

# Keep countries that actually outperform expectation
mel_candidates = pd_benchmark[
    pd_benchmark["GHSI_Residual"] > 0
].copy()

# Rank
mel_candidates = (
    mel_candidates
    .sort_values(
        "MEL_Benchmark_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

mel_candidates.insert(
    0,
    "MEL_Benchmark_Rank",
    np.arange(
        1,
        len(mel_candidates) + 1
    )
)

display(
    mel_candidates[
        [
            "MEL_Benchmark_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "GHSI_Residual",
            "Residual_Z",
            "Distance_from_Sierra_Leone",
            "Structural_Similarity",
            "African_Relevance",
            "MEL_Benchmark_Score"
        ]
    ]
    .head(30)
    .round(3)
)

In [ ]:
# ==========================================
# GLOBAL STRUCTURAL DISTANCE
# SIERRA LEONE → EVERY COUNTRY
# ==========================================

from sklearn.neighbors import NearestNeighbors

# Use the same PCA features used for clustering
distance_features = ["PC1", "PC2", "PC3"]

distance_data = cluster_data[
    ["ISO3", "COUNTRY_NAME", "REGION", "GHSI"] + distance_features
].dropna(
    subset=distance_features
).copy()

# Reset index so sklearn positions match dataframe rows
distance_data = distance_data.reset_index(drop=True)

# Sierra Leone position
sierra_position = distance_data.index[
    distance_data["ISO3"] == "SLE"
][0]

# Feature matrix
X_distance = distance_data[distance_features].values

# Euclidean distance from Sierra Leone
sierra_vector = X_distance[sierra_position]

distance_data["Distance_from_Sierra_Leone"] = np.linalg.norm(
    X_distance - sierra_vector,
    axis=1
)

# Remove Sierra Leone itself
global_distance = distance_data[
    distance_data["ISO3"] != "SLE"
].copy()

# Structural similarity
global_distance["Structural_Similarity"] = (
    1 /
    (1 + global_distance["Distance_from_Sierra_Leone"])
)

global_distance = global_distance.sort_values(
    "Distance_from_Sierra_Leone"
).reset_index(drop=True)

display(
    global_distance[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "Distance_from_Sierra_Leone",
            "Structural_Similarity"
        ]
    ].head(20).round(3)
)


In [ ]:
# ==========================================
# GLOBAL STRUCTURAL DISTANCE
# SIERRA LEONE → EVERY COUNTRY
# ==========================================

distance_features = ["PC1", "PC2", "PC3"]

distance_data = pca_countries[
    [
        "ISO3",
        "COUNTRY_NAME",
        "REGION",
        "GHSI"
    ] + distance_features
].dropna(
    subset=distance_features
).copy()

# Reset index so positions match the numpy array
distance_data = distance_data.reset_index(drop=True)

# Find Sierra Leone
sierra_position = distance_data.index[
    distance_data["ISO3"].eq("SLE")
][0]

# PCA coordinates
X_distance = distance_data[
    distance_features
].to_numpy()

# Sierra Leone's coordinates
sierra_vector = X_distance[sierra_position]

# Euclidean distance from Sierra Leone
distance_data["Distance_from_Sierra_Leone"] = np.linalg.norm(
    X_distance - sierra_vector,
    axis=1
)

# Convert distance to a 0–1 similarity measure
distance_data["Structural_Similarity"] = (
    1 / (
        1 +
        distance_data["Distance_from_Sierra_Leone"]
    )
)

# Remove Sierra Leone itself
global_distance = distance_data[
    distance_data["ISO3"] != "SLE"
].copy()

# Rank closest structural peers
global_distance = (
    global_distance
    .sort_values("Distance_from_Sierra_Leone")
    .reset_index(drop=True)
)

display(
    global_distance[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "Distance_from_Sierra_Leone",
            "Structural_Similarity"
        ]
    ]
    .head(20)
    .round(3)
)

## 15. MEL Benchmarking


In [ ]:
# ==========================================
# GLOBAL MEL BENCHMARK MATRIX
# STRUCTURAL SIMILARITY + POSITIVE DEVIANCE
# ==========================================

# Merge full structural distances with positive-deviance results
mel_matrix = pd_model.merge(
    global_distance[
        [
            "ISO3",
            "Distance_from_Sierra_Leone",
            "Structural_Similarity"
        ]
    ],
    on="ISO3",
    how="inner"
)

# African relevance
mel_matrix["African_Relevance"] = (
    mel_matrix["REGION"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("africa")
    .astype(int)
)

# Only countries with positive residuals
mel_matrix = mel_matrix[
    mel_matrix["GHSI_Residual"] > 0
].copy()

# Performance score
max_positive_residual = mel_matrix[
    "GHSI_Residual"
].max()

mel_matrix["Performance_Score"] = (
    mel_matrix["GHSI_Residual"] /
    max_positive_residual
)

# ------------------------------------------
# GLOBAL SCORE
# ------------------------------------------
mel_matrix["Global_MEL_Score"] = (
    0.50 * mel_matrix["Structural_Similarity"]
    +
    0.50 * mel_matrix["Performance_Score"]
)

# ------------------------------------------
# AFRICAN SCORE
# ------------------------------------------
african_matrix = mel_matrix[
    mel_matrix["African_Relevance"] == 1
].copy()

african_matrix["African_MEL_Score"] = (
    0.50 * african_matrix["Structural_Similarity"]
    +
    0.50 * african_matrix["Performance_Score"]
)

# ------------------------------------------
# GLOBAL RANKING
# ------------------------------------------
global_benchmarks = (
    mel_matrix
    .sort_values(
        "Global_MEL_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

global_benchmarks.insert(
    0,
    "Global_MEL_Rank",
    np.arange(
        1,
        len(global_benchmarks) + 1
    )
)

# ------------------------------------------
# AFRICAN RANKING
# ------------------------------------------
african_benchmarks = (
    african_matrix
    .sort_values(
        "African_MEL_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

african_benchmarks.insert(
    0,
    "African_MEL_Rank",
    np.arange(
        1,
        len(african_benchmarks) + 1
    )
)

# ------------------------------------------
# DISPLAY
# ------------------------------------------

print("GLOBAL MEL BENCHMARKS")
display(
    global_benchmarks[
        [
            "Global_MEL_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "GHSI",
            "GHSI_Residual",
            "Residual_Z",
            "Distance_from_Sierra_Leone",
            "Structural_Similarity",
            "Performance_Score",
            "Global_MEL_Score"
        ]
    ]
    .head(20)
    .round(3)
)

print("\nAFRICAN MEL BENCHMARKS")
display(
    african_benchmarks[
        [
            "African_MEL_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "GHSI",
            "GHSI_Residual",
            "Residual_Z",
            "Distance_from_Sierra_Leone",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score"
        ]
    ]
    .head(20)
    .round(3)
)

In [ ]:
# ==========================================
# MEL BENCHMARK SENSITIVITY ANALYSIS
# ==========================================

sensitivity = african_matrix.copy()

# Scenario 1: Structural similarity dominant
sensitivity["Score_Structure_Heavy"] = (
    0.70 * sensitivity["Structural_Similarity"]
    +
    0.30 * sensitivity["Performance_Score"]
)

# Scenario 2: Balanced
sensitivity["Score_Balanced"] = (
    0.50 * sensitivity["Structural_Similarity"]
    +
    0.50 * sensitivity["Performance_Score"]
)

# Scenario 3: Performance dominant
sensitivity["Score_Performance_Heavy"] = (
    0.30 * sensitivity["Structural_Similarity"]
    +
    0.70 * sensitivity["Performance_Score"]
)

# Rank under each scenario
sensitivity["Rank_Structure"] = (
    sensitivity["Score_Structure_Heavy"]
    .rank(method="min", ascending=False)
)

sensitivity["Rank_Balanced"] = (
    sensitivity["Score_Balanced"]
    .rank(method="min", ascending=False)
)

sensitivity["Rank_Performance"] = (
    sensitivity["Score_Performance_Heavy"]
    .rank(method="min", ascending=False)
)

# Average rank across scenarios
sensitivity["Mean_Rank"] = (
    sensitivity[
        [
            "Rank_Structure",
            "Rank_Balanced",
            "Rank_Performance"
        ]
    ].mean(axis=1)
)

# Final robust ranking
sensitivity = (
    sensitivity
    .sort_values("Mean_Rank")
    .reset_index(drop=True)
)

sensitivity.insert(
    0,
    "Robust_Rank",
    np.arange(1, len(sensitivity) + 1)
)

display(
    sensitivity[
        [
            "Robust_Rank",
            "ISO3",
            "COUNTRY_NAME",
            "GHSI",
            "GHSI_Residual",
            "Structural_Similarity",
            "Performance_Score",
            "Rank_Structure",
            "Rank_Balanced",
            "Rank_Performance",
            "Mean_Rank"
        ]
    ]
    .head(20)
    .round(3)
)

In [ ]:
# ==========================================
# INDICATOR-LEVEL MEL BENCHMARK ANALYSIS
# ==========================================

# Use the cleaned indicator priority table
indicator_benchmarks = indicator_priority_clean.copy()

# Check structure
print("Shape:", indicator_benchmarks.shape)
print("\nComponents:")
print(indicator_benchmarks["Component"].value_counts())

print("\nIndicators:")
display(
    indicator_benchmarks[
        [
            "Component",
            "Indicator",
            "Value",
            "Percentile",
            "Benchmark_Mean",
            "Gap_vs_Benchmark",
            "Evidence_Group"
        ]
    ].sort_values(
        ["Component", "Gap_vs_Benchmark"],
        ascending=[True, False]
    ).round(2)
)

In [ ]:
# ==========================================
# INDICATOR-LEVEL MEL BENCHMARK MATRIX
# ==========================================

# 1. Start with the global indicator dataset
indicator_data = normalised.copy()

# 2. Keep the fields needed
indicator_data = indicator_data[
    [
        "COUNTRY",
        "ISO3",
        "Component",
        "Indicator",
        "Value",
        "Directional_Value",
        "Percentile"
    ]
].copy()

# Clean country identifiers
indicator_data["ISO3"] = indicator_data["ISO3"].astype(str).str.strip()
indicator_data["COUNTRY"] = indicator_data["COUNTRY"].astype(str).str.strip()

# 3. Sierra Leone's indicator values
sl_indicator_data = indicator_data[
    indicator_data["ISO3"].eq("SLE")
].copy()

sl_indicator_data = sl_indicator_data.rename(
    columns={
        "COUNTRY": "Sierra_Leone",
        "Value": "Sierra_Leone_Value",
        "Directional_Value": "Sierra_Leone_Directional_Value",
        "Percentile": "Sierra_Leone_Percentile"
    }
)

sl_indicator_data = sl_indicator_data[
    [
        "Component",
        "Indicator",
        "Sierra_Leone_Value",
        "Sierra_Leone_Directional_Value",
        "Sierra_Leone_Percentile"
    ]
]

print("Sierra Leone indicators:", len(sl_indicator_data))

display(
    sl_indicator_data.sort_values(
        ["Component", "Sierra_Leone_Percentile"]
    ).round(2)
)

In [ ]:
# ==========================================
# CONNECT ML BENCHMARK COUNTRIES
# ==========================================

# Use your robust African benchmark results
benchmark_countries = sensitivity[
    [
        "ISO3",
        "COUNTRY_NAME",
        "REGION",
        "GHSI",
        "GHSI_Residual",
        "Structural_Similarity",
        "Performance_Score",
        "African_MEL_Score",
        "Mean_Rank"
    ]
].copy()

# Only countries with usable benchmark information
benchmark_countries = benchmark_countries.dropna(
    subset=["ISO3"]
)

benchmark_countries["ISO3"] = (
    benchmark_countries["ISO3"]
    .astype(str)
    .str.strip()
)

print("Benchmark candidates:", len(benchmark_countries))

display(
    benchmark_countries.sort_values("Mean_Rank").head(20)
)

In [ ]:
# ==========================================
# BENCHMARK COUNTRY × INDICATOR PERFORMANCE
# ==========================================

candidate_indicator_data = indicator_data.merge(
    benchmark_countries,
    on="ISO3",
    how="inner"
)

print(
    "Candidate country-indicator observations:",
    candidate_indicator_data.shape
)

display(
    candidate_indicator_data[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "Component",
            "Indicator",
            "Value",
            "Percentile",
            "GHSI",
            "GHSI_Residual",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score",
            "Mean_Rank"
        ]
    ].head(20)
)

In [ ]:
# ==========================================
# INDICATOR-LEVEL BENCHMARK GAP
# ==========================================

# Sierra Leone indicator values
sl = indicator_data[
    indicator_data["ISO3"] == "SLE"
][
    ["Component", "Indicator", "Value", "Percentile"]
].copy()

sl = sl.rename(
    columns={
        "Value": "Sierra_Leone_Value",
        "Percentile": "Sierra_Leone_Percentile"
    }
)

# Remove Sierra Leone from benchmark candidates
candidates = candidate_indicator_data[
    candidate_indicator_data["ISO3"] != "SLE"
].copy()

# Merge Sierra Leone's indicator performance onto every candidate
indicator_benchmark_matrix = candidates.merge(
    sl,
    on=["Component", "Indicator"],
    how="inner"
)

# Calculate performance advantage over Sierra Leone
indicator_benchmark_matrix["Indicator_Gap"] = (
    indicator_benchmark_matrix["Percentile"]
    - indicator_benchmark_matrix["Sierra_Leone_Percentile"]
)

print("Rows:", indicator_benchmark_matrix.shape)

display(
    indicator_benchmark_matrix[
        [
            "ISO3",
            "COUNTRY_NAME",
            "REGION",
            "Component",
            "Indicator",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "GHSI_Residual",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score",
            "Mean_Rank"
        ]
    ]
    .sort_values(
        ["Component", "Indicator", "Indicator_Gap"],
        ascending=[True, True, False]
    )
    .head(50)
    .round(3)
)

In [ ]:
# ==========================================
# BEST BENCHMARK PER INDICATOR
# ==========================================

# Prefer countries that:
# 1. materially outperform Sierra Leone on the indicator
# 2. have strong overall MEL benchmark standing
# 3. have structural similarity
#
# African_MEL_Score is used as the main tie-breaking criterion.

indicator_benchmark_matrix["Benchmark_Score"] = (
    0.50 * indicator_benchmark_matrix["Indicator_Gap"].clip(lower=0)
    + 0.25 * indicator_benchmark_matrix["Structural_Similarity"].fillna(0)
    + 0.25 * indicator_benchmark_matrix["African_MEL_Score"].fillna(0)
)

best_indicator_benchmarks = (
    indicator_benchmark_matrix
    .sort_values(
        ["Component", "Indicator", "Benchmark_Score"],
        ascending=[True, True, False]
    )
    .groupby(
        ["Component", "Indicator"],
        as_index=False
    )
    .first()
)

print(
    "Indicator benchmark pairs:",
    len(best_indicator_benchmarks)
)

display(
    best_indicator_benchmarks[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Structural_Similarity",
            "African_MEL_Score",
            "Benchmark_Score"
        ]
    ].sort_values(
        "Indicator_Gap",
        ascending=False
    ).round(3)
)

In [ ]:
# ==========================================
# VALIDATE INDICATOR BENCHMARKS
# ==========================================

# Show the strongest benchmark for every Sierra Leone indicator
validation = best_indicator_benchmarks[
    [
        "Component",
        "Indicator",
        "COUNTRY_NAME",
        "REGION",
        "Sierra_Leone_Value",
        "Sierra_Leone_Percentile",
        "Value",
        "Percentile",
        "Indicator_Gap",
        "Structural_Similarity",
        "Performance_Score",
        "African_MEL_Score",
        "Benchmark_Score"
    ]
].copy()

validation = validation.sort_values(
    ["Component", "Indicator"]
).reset_index(drop=True)

display(validation.round(3))

In [ ]:
# ==========================================
# FINAL MEL BENCHMARK SELECTION
# ==========================================

mel_candidates = validation.copy()

# Keep only benchmarks that:
# 1. outperform Sierra Leone on the indicator
# 2. have reasonable structural similarity
# 3. have positive overall benchmark performance

mel_candidates = mel_candidates[
    (mel_candidates["Indicator_Gap"] > 0) &
    (mel_candidates["Structural_Similarity"] >= 0.50) &
    (mel_candidates["Performance_Score"] > 0)
].copy()

# Rank candidates within each indicator
mel_candidates["Indicator_Benchmark_Rank"] = (
    mel_candidates
    .groupby(["Component", "Indicator"])["Benchmark_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Keep top 3 candidates per indicator
top_mel_candidates = mel_candidates[
    mel_candidates["Indicator_Benchmark_Rank"] <= 3
].copy()

top_mel_candidates = top_mel_candidates.sort_values(
    [
        "Component",
        "Indicator",
        "Indicator_Benchmark_Rank"
    ]
)

display(
    top_mel_candidates[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score",
            "Benchmark_Score",
            "Indicator_Benchmark_Rank"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# BENCHMARK ROLE
# ==========================================

def benchmark_role(row):

    if (
        row["REGION"] == "Africa"
        and row["Structural_Similarity"] >= 0.60
        and row["Performance_Score"] > 0
    ):
        return "African structural-performance benchmark"

    elif (
        row["REGION"] == "Africa"
        and row["Performance_Score"] > 0
    ):
        return "African performance benchmark"

    elif row["Structural_Similarity"] >= 0.60:
        return "Structural learning benchmark"

    else:
        return "Global performance benchmark"


top_mel_candidates["Benchmark_Role"] = (
    top_mel_candidates.apply(
        benchmark_role,
        axis=1
    )
)

display(
    top_mel_candidates[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "Benchmark_Role",
            "Indicator_Gap",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# FINAL MEL BENCHMARK MATRIX
# ==========================================

mel_benchmark_matrix = top_mel_candidates.copy()

# Priority within each indicator
mel_benchmark_matrix["MEL_Priority"] = (
    mel_benchmark_matrix
    .groupby(["Component", "Indicator"])["Benchmark_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Benchmark interpretation
def assign_mel_action(row):

    if row["Indicator_Gap"] >= 25:
        return "High-priority improvement opportunity"
    elif row["Indicator_Gap"] >= 15:
        return "Priority improvement opportunity"
    elif row["Indicator_Gap"] > 0:
        return "Monitor and improve"
    else:
        return "No immediate improvement gap"


mel_benchmark_matrix["MEL_Action"] = (
    mel_benchmark_matrix.apply(assign_mel_action, axis=1)
)

# Monitoring direction
mel_benchmark_matrix["Monitoring_Direction"] = (
    "Track progress toward benchmark percentile"
)

# Final columns
mel_benchmark_matrix = mel_benchmark_matrix[
    [
        "Component",
        "Indicator",
        "COUNTRY_NAME",
        "REGION",
        "Sierra_Leone_Value",
        "Sierra_Leone_Percentile",
        "Value",
        "Percentile",
        "Indicator_Gap",
        "Structural_Similarity",
        "Performance_Score",
        "African_MEL_Score",
        "Benchmark_Score",
        "Benchmark_Role",
        "MEL_Priority",
        "MEL_Action",
        "Monitoring_Direction"
    ]
].sort_values(
    [
        "Component",
        "Indicator",
        "MEL_Priority"
    ]
)

display(mel_benchmark_matrix.round(3))

In [ ]:
# ==========================================
# FINAL MEL BENCHMARK MATRIX
# ==========================================

mel_benchmark_matrix = top_mel_candidates.copy()

# Priority within each indicator
mel_benchmark_matrix["MEL_Priority"] = (
    mel_benchmark_matrix
    .groupby(["Component", "Indicator"])["Benchmark_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Benchmark interpretation
def assign_mel_action(row):

    if row["Indicator_Gap"] >= 25:
        return "High-priority improvement opportunity"
    elif row["Indicator_Gap"] >= 15:
        return "Priority improvement opportunity"
    elif row["Indicator_Gap"] > 0:
        return "Monitor and improve"
    else:
        return "No immediate improvement gap"


mel_benchmark_matrix["MEL_Action"] = (
    mel_benchmark_matrix.apply(assign_mel_action, axis=1)
)

# Monitoring direction
mel_benchmark_matrix["Monitoring_Direction"] = (
    "Track progress toward benchmark percentile"
)

# Final columns
mel_benchmark_matrix = mel_benchmark_matrix[
    [
        "Component",
        "Indicator",
        "COUNTRY_NAME",
        "REGION",
        "Sierra_Leone_Value",
        "Sierra_Leone_Percentile",
        "Value",
        "Percentile",
        "Indicator_Gap",
        "Structural_Similarity",
        "Performance_Score",
        "African_MEL_Score",
        "Benchmark_Score",
        "Benchmark_Role",
        "MEL_Priority",
        "MEL_Action",
        "Monitoring_Direction"
    ]
].sort_values(
    [
        "Component",
        "Indicator",
        "MEL_Priority"
    ]
)

display(mel_benchmark_matrix.round(3))

In [ ]:
# ==========================================
# EXPORT FINAL MEL MATRIX
# ==========================================

mel_benchmark_matrix.to_csv(
    OUTPUT_DIR / "Sierra_Leone_MEL_Benchmark_Matrix.csv",
    index=False
)

print(
    "Saved:",
    "Sierra_Leone_MEL_Benchmark_Matrix.csv"
)

print(
    "Rows:",
    len(mel_benchmark_matrix)
)

In [ ]:
# ==========================================
# BENCHMARK CONFIDENCE SCORE
# ==========================================

confidence = mel_benchmark_matrix.copy()

# Normalise the indicator gap so large gaps do not dominate
gap_min = confidence["Indicator_Gap"].min()
gap_max = confidence["Indicator_Gap"].max()

if gap_max > gap_min:
    confidence["Gap_Score"] = (
        (confidence["Indicator_Gap"] - gap_min)
        / (gap_max - gap_min)
    )
else:
    confidence["Gap_Score"] = 0.0

# Benchmark Confidence Score
confidence["Benchmark_Confidence_Score"] = (
    0.35 * confidence["Structural_Similarity"]
    + 0.30 * confidence["Performance_Score"]
    + 0.20 * confidence["African_MEL_Score"]
    + 0.15 * confidence["Gap_Score"]
)

# Rank within each indicator
confidence["Confidence_Rank"] = (
    confidence
    .groupby(["Component", "Indicator"])
    ["Benchmark_Confidence_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Confidence category
def confidence_category(score):
    if score >= 0.70:
        return "High confidence"
    elif score >= 0.50:
        return "Moderate confidence"
    elif score >= 0.30:
        return "Limited confidence"
    else:
        return "Low confidence"

confidence["Benchmark_Confidence"] = (
    confidence["Benchmark_Confidence_Score"]
    .apply(confidence_category)
)

display(
    confidence[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Indicator_Gap",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score",
            "Benchmark_Confidence_Score",
            "Benchmark_Confidence",
            "Confidence_Rank"
        ]
    ]
    .sort_values(
        ["Component", "Indicator", "Confidence_Rank"]
    )
    .round(3)
)

In [ ]:
# ==========================================
# FINAL BENCHMARK SELECTION
# ==========================================

final_mel_benchmarks = confidence[
    confidence["Confidence_Rank"] == 1
].copy()

final_mel_benchmarks = final_mel_benchmarks[
    final_mel_benchmarks["Benchmark_Confidence"].isin(
        ["High confidence", "Moderate confidence"]
    )
].copy()

final_mel_benchmarks = final_mel_benchmarks.sort_values(
    ["Component", "Benchmark_Confidence_Score"],
    ascending=[True, False]
)

display(
    final_mel_benchmarks[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Structural_Similarity",
            "Performance_Score",
            "African_MEL_Score",
            "Benchmark_Confidence_Score",
            "Benchmark_Confidence"
        ]
    ].round(3)
)

## 16. Hazard → Capacity → MEL Framework


In [ ]:
# ==========================================
# HAZARD → CAPACITY → MEL BENCHMARK MATRIX
# ==========================================

hazard_capacity = hazard_priority_matrix.copy()

# Keep only the major hazards
hazard_capacity = hazard_capacity[
    hazard_capacity["Hazard_Priority"].isin(
        [
            "High relative exposure",
            "Moderate relative exposure",
            "Lower relative exposure"
        ]
    )
].copy()

# Merge component-level priorities
hazard_capacity = hazard_capacity.merge(
    final_priority[
        [
            "Component",
            "Structural_Priority_Score"
        ]
    ],
    on="Component",
    how="left"
)

# Merge final indicator benchmarks
hazard_mel = hazard_capacity.merge(
    final_mel_benchmarks[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Benchmark_Confidence_Score",
            "Benchmark_Confidence"
        ]
    ],
    on="Component",
    how="left"
)

# Combined decision score
hazard_mel["Decision_Score"] = (
    0.40 * hazard_mel["Hazard_Percentile"] / 100
    + 0.35 * hazard_mel["Structural_Priority_Score"]
    + 0.25 * hazard_mel["Benchmark_Confidence_Score"]
)

# Rank within hazard
hazard_mel["Decision_Rank"] = (
    hazard_mel
    .groupby("Hazard")["Decision_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Keep the strongest recommendations
hazard_mel_final = hazard_mel[
    hazard_mel["Decision_Rank"] <= 5
].copy()

hazard_mel_final = hazard_mel_final.sort_values(
    ["Decision_Score"],
    ascending=False
)

display(
    hazard_mel_final[
        [
            "Hazard",
            "Hazard_Percentile",
            "Hazard_Priority",
            "Component",
            "Structural_Priority_Score",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Benchmark_Confidence_Score",
            "Benchmark_Confidence",
            "Decision_Score",
            "Decision_Rank"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# HAZARD → CAPACITY → MEL BENCHMARK MATRIX
# ==========================================

hazard_capacity = hazard_capacity_detail.copy()

# Merge the final indicator-level MEL benchmarks
hazard_mel = hazard_capacity.merge(
    final_mel_benchmarks[
        [
            "Component",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Benchmark_Confidence_Score",
            "Benchmark_Confidence"
        ]
    ],
    left_on="Capacity_Component",
    right_on="Component",
    how="left"
)

# Remove duplicate component column
hazard_mel = hazard_mel.drop(
    columns=["Component"]
)

# Decision score:
# 40% hazard exposure
# 35% capacity weakness
# 25% benchmark confidence

hazard_mel["Decision_Score"] = (
    0.40 * (hazard_mel["Hazard_Percentile"] / 100)
    + 0.35 * hazard_mel["Benchmark_Gap"] / 100
    + 0.25 * hazard_mel["Benchmark_Confidence_Score"]
)

# Rank recommendations within each hazard
hazard_mel["Decision_Rank"] = (
    hazard_mel
    .groupby("Hazard")["Decision_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Keep top 5 recommendations per hazard
hazard_mel_final = hazard_mel[
    hazard_mel["Decision_Rank"] <= 5
].copy()

hazard_mel_final = hazard_mel_final.sort_values(
    ["Hazard", "Decision_Rank"]
)

display(
    hazard_mel_final[
        [
            "Hazard",
            "Hazard_Percentile",
            "Hazard_Priority",
            "Capacity_Component",
            "Capacity_Priority",
            "Within_Hazard_Rank",
            "Indicator",
            "COUNTRY_NAME",
            "REGION",
            "Sierra_Leone_Value",
            "Sierra_Leone_Percentile",
            "Value",
            "Percentile",
            "Indicator_Gap",
            "Benchmark_Confidence_Score",
            "Benchmark_Confidence",
            "Decision_Score",
            "Decision_Rank"
        ]
    ].round(3)
)

In [ ]:
# ==========================================
# FINAL STAKEHOLDER MEL MATRIX
# ==========================================

stakeholder_mel = hazard_mel_final.copy()

# Benchmark target = benchmark's observed percentile
stakeholder_mel["MEL_Target_Percentile"] = (
    stakeholder_mel["Percentile"]
)

# Size of improvement required
stakeholder_mel["Required_Improvement"] = (
    stakeholder_mel["Percentile"]
    - stakeholder_mel["Sierra_Leone_Percentile"]
)

# Priority classification
def mel_priority(row):

    if row["Decision_Score"] >= 0.70:
        return "Critical"

    elif row["Decision_Score"] >= 0.55:
        return "High"

    elif row["Decision_Score"] >= 0.40:
        return "Moderate"

    else:
        return "Lower"


stakeholder_mel["MEL_Priority"] = (
    stakeholder_mel.apply(mel_priority, axis=1)
)

# Recommended MEL action
def mel_action(row):

    if row["Required_Improvement"] >= 30:
        return "Prioritise system strengthening and track progress against benchmark"

    elif row["Required_Improvement"] >= 15:
        return "Targeted improvement and routine monitoring"

    elif row["Required_Improvement"] > 0:
        return "Monitor progress toward benchmark"

    else:
        return "Maintain performance and monitor"

stakeholder_mel["Recommended_MEL_Action"] = (
    stakeholder_mel.apply(mel_action, axis=1)
)

# Final stakeholder-facing columns
stakeholder_mel = stakeholder_mel[
    [
        "Hazard",
        "Hazard_Percentile",
        "Hazard_Priority",
        "Capacity_Component",
        "Capacity_Priority",
        "Indicator",
        "COUNTRY_NAME",
        "REGION",
        "Sierra_Leone_Value",
        "Sierra_Leone_Percentile",
        "Value",
        "Percentile",
        "Indicator_Gap",
        "Required_Improvement",
        "Benchmark_Confidence_Score",
        "Benchmark_Confidence",
        "Decision_Score",
        "MEL_Priority",
        "Recommended_MEL_Action"
    ]
].sort_values(
    ["MEL_Priority", "Decision_Score"],
    ascending=[True, False]
)

display(
    stakeholder_mel.round(3)
)


## 17. Executive MEL Outputs


In [ ]:
# ==========================================
# EXECUTIVE MEL PRIORITIES
# ==========================================

executive_mel = stakeholder_mel[
    stakeholder_mel["MEL_Priority"].isin(
        ["Critical", "High"]
    )
].copy()

display(
    executive_mel[
        [
            "Hazard",
            "Hazard_Priority",
            "Capacity_Component",
            "Indicator",
            "COUNTRY_NAME",
            "Sierra_Leone_Percentile",
            "Percentile",
            "Required_Improvement",
            "Benchmark_Confidence",
            "MEL_Priority",
            "Recommended_MEL_Action"
        ]
    ].round(2)
)

## 20. Exporting Results


In [ ]:
# ==========================================
# EXPORT COMPLETE MEL ANALYTICAL WORKBOOK
# ==========================================

import pandas as pd

output_file = OUTPUT_DIR / "SIERRA_LEONE_MEL_ANALYTICAL_OUTPUT.xlsx"

output_tables = {
    "Hazard_Profile": hazard_profile,
    "Hazard_Priorities": stakeholder_hazards,
    "Hazard_Capacity": hazard_capacity_detail,
    "Hazard_Priority_Summary": hazard_priority_summary,
    "Indicator_Priorities": indicator_priority_clean,
    "Component_Priorities": final_priority,
    "PCA_Variance": explained_variance,
    "Structural_Peers": sierra_peers,
    "African_MEL_Benchmarks": final_mel_benchmarks,
    "Hazard_MEL_Matrix": stakeholder_mel,
    "Executive_MEL": executive_mel
}

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, df in output_tables.items():

        # Excel sheet names cannot exceed 31 characters
        safe_name = sheet_name[:31]

        df.to_excel(
            writer,
            sheet_name=safe_name,
            index=False
        )

print(f"Saved: {output_file}")

print("\nSheets created:")
for sheet in output_tables:
    print(" -", sheet[:31])

In [ ]:
# ==========================================
# FINAL QA — MEL ANALYTICAL OUTPUT
# ==========================================

print("=== DATASET SIZES ===")
print("Hazards:", len(hazard_profile))
print("Hazard-capacity links:", len(hazard_capacity_detail))
print("Final MEL benchmarks:", len(final_mel_benchmarks))
print("Hazard-MEL rows:", len(stakeholder_mel))
print("Executive priorities:", len(executive_mel))

print("\n=== BENCHMARK CONFIDENCE ===")
print(
    final_mel_benchmarks["Benchmark_Confidence"]
    .value_counts(dropna=False)
)

print("\n=== MEL PRIORITY ===")
print(
    stakeholder_mel["MEL_Priority"]
    .value_counts(dropna=False)
)

print("\n=== HAZARDS ===")
print(
    stakeholder_mel[
        ["Hazard", "Hazard_Priority"]
    ].drop_duplicates()
)

print("\n=== COMPONENTS ===")
print(
    stakeholder_mel[
        ["Capacity_Component"]
    ].drop_duplicates()
)

print("\n=== MISSING VALUES ===")
print(
    stakeholder_mel[
        [
            "Indicator",
            "COUNTRY_NAME",
            "Sierra_Leone_Percentile",
            "Percentile",
            "Benchmark_Confidence_Score"
        ]
    ].isna().sum()
)

In [ ]:
# ==========================================
# CONSISTENCY CHECK
# ==========================================

checks = {
    "Sierra Leone appears in benchmark data":
        "Sierra Leone" in stakeholder_mel["COUNTRY_NAME"].astype(str).values,

    "No negative indicator gaps":
        (stakeholder_mel["Indicator_Gap"] >= 0).all(),

    "Benchmark confidence scores valid":
        stakeholder_mel["Benchmark_Confidence_Score"].between(0, 1).all(),

    "Decision scores valid":
        stakeholder_mel["Decision_Score"].between(0, 1).all(),

    "All hazards have recommendations":
        stakeholder_mel["Hazard"].nunique()
        == hazard_profile["Hazard"].nunique()
}

for check, result in checks.items():
    print(f"{'PASS' if result else 'CHECK'} — {check}")

## 18. Quality Assurance & Robustness Checks


In [ ]:
# ==========================================
# CLUSTERING SENSITIVITY ANALYSIS
# ==========================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

# Use the same PCA representation used for the main clustering
X_cluster = pca_countries[["PC1", "PC2", "PC3"]].dropna().copy()

sensitivity_results = []

for k in range(2, 9):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(X_cluster)

    score = silhouette_score(
        X_cluster,
        labels
    )

    sensitivity_results.append({
        "K": k,
        "Silhouette_Score": score
    })

sensitivity_results = pd.DataFrame(
    sensitivity_results
)

display(
    sensitivity_results.round(3)
)

In [ ]:
# ==========================================
# DBSCAN SENSITIVITY CHECK
# ==========================================

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

dbscan_results = []

for eps in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:

    model = DBSCAN(
        eps=eps,
        min_samples=5
    )

    labels = model.fit_predict(X_cluster)

    n_clusters = len(
        set(labels)
        - {-1}
    )

    n_noise = sum(labels == -1)

    if n_clusters >= 2:

        mask = labels != -1

        score = silhouette_score(
            X_cluster.loc[mask],
            labels[mask]
        )

    else:
        score = None

    dbscan_results.append({
        "eps": eps,
        "Clusters": n_clusters,
        "Noise_Points": n_noise,
        "Silhouette_Score": score
    })

dbscan_results = pd.DataFrame(
    dbscan_results
)

display(
    dbscan_results.round(3)
)

## 19. Portfolio Figures


In [ ]:
# ==========================================
# FIGURE 1 — GLOBAL PCA STRUCTURE
# ==========================================

import matplotlib.pyplot as plt

plot_data = pca_countries.dropna(
    subset=["PC1", "PC2", "Cluster"]
).copy()

plt.figure(figsize=(11, 8))

for cluster in sorted(plot_data["Cluster"].unique()):

    subset = plot_data[
        plot_data["Cluster"] == cluster
    ]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

# Highlight Sierra Leone
sl = plot_data[
    plot_data["ISO3"] == "SLE"
]

if len(sl) > 0:

    plt.scatter(
        sl["PC1"],
        sl["PC2"],
        s=120,
        marker="*",
        label="Sierra Leone"
    )

    plt.annotate(
        "Sierra Leone",
        (
            sl["PC1"].iloc[0],
            sl["PC2"].iloc[0]
        ),
        xytext=(8, 8),
        textcoords="offset points"
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Global Disaster Preparedness Structure")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# FIGURE 2 — AFRICAN MEL BENCHMARKS
# ==========================================

plot_data = robust_rank.head(10).copy()

plt.figure(figsize=(10, 6))

plt.barh(
    plot_data["COUNTRY_NAME"].iloc[::-1],
    plot_data["Performance_Score"].iloc[::-1]
)

plt.xlabel("Performance Score")
plt.ylabel("")
plt.title(
    "Top African Countries for MEL Benchmarking"
)

plt.tight_layout()
plt.show()

In [ ]:
robust_rank = african_mel_rank.copy()

In [ ]:
# ==========================================
# FIND AFRICAN MEL DATAFRAME
# ==========================================

required_cols = {
    "ISO3",
    "COUNTRY_NAME",
    "GHSI",
    "GHSI_Residual",
    "Structural_Similarity",
    "Performance_Score",
    "African_MEL_Score"
}

matches = []

for name, obj in list(globals().items()):
    if hasattr(obj, "columns"):
        if required_cols.issubset(set(obj.columns)):
            matches.append(name)

print("Matching dataframes:")
print(matches)

In [ ]:
# ==========================================
# FIGURE 2 — AFRICAN MEL BENCHMARKS
# ==========================================

import matplotlib.pyplot as plt

plot_data = (
    african_benchmarks
    .sort_values(
        "African_MEL_Score",
        ascending=True
    )
    .copy()
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["COUNTRY_NAME"],
    plot_data["African_MEL_Score"]
)

plt.xlabel("African MEL Benchmark Score")
plt.ylabel("")
plt.title(
    "African MEL Benchmark Suitability for Sierra Leone"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# FIGURE 3 — STRUCTURAL SIMILARITY VS PERFORMANCE
# ==========================================

plt.figure(figsize=(10, 7))

plt.scatter(
    african_benchmarks["Structural_Similarity"],
    african_benchmarks["Performance_Score"],
    s=70,
    alpha=0.75
)

# Label countries
for _, row in african_benchmarks.iterrows():

    plt.annotate(
        row["ISO3"],
        (
            row["Structural_Similarity"],
            row["Performance_Score"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Structural Similarity to Sierra Leone")
plt.ylabel("Performance Score")
plt.title(
    "African Benchmark Countries: Structural Similarity vs Performance"
)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# FIGURE 4 — ROBUSTNESS OF MEL RANKING
# ==========================================

plot_data = (
    sensitivity
    .sort_values("Mean_Rank", ascending=True)
    .head(10)
    .copy()
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["COUNTRY_NAME"],
    plot_data["Mean_Rank"]
)

plt.xlabel("Mean Rank Across Weighting Scenarios")
plt.ylabel("")
plt.title(
    "Robustness of African MEL Benchmark Rankings"
)

plt.gca().invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# BUILD INDICATOR-LEVEL MEL BENCHMARKS
# ==========================================

# Sierra Leone indicator data
sl = indicator_comparison[
    indicator_comparison["ISO3"] == "SLE"
].copy()

# Benchmark-country indicator data
bench = indicator_comparison[
    indicator_comparison["ISO3"].isin(benchmark_iso3)
].copy()

# Only indicators where Sierra Leone has a usable value
sl = sl.dropna(
    subset=["Directional_Value"]
)

# Only benchmark observations with usable values
bench = bench.dropna(
    subset=["Directional_Value"]
)

# Merge benchmark observations to Sierra Leone
indicator_benchmarks = bench.merge(
    sl[
        [
            "Component",
            "Indicator",
            "Directional_Value"
        ]
    ].rename(
        columns={
            "Directional_Value":
            "Sierra_Leone_Directional_Value"
        }
    ),
    on=["Component", "Indicator"],
    how="inner"
)

# Calculate benchmark gap
indicator_benchmarks["Indicator_Gap"] = (
    indicator_benchmarks["Directional_Value"]
    - indicator_benchmarks["Sierra_Leone_Directional_Value"]
)

print(
    "Indicator-country benchmark observations:",
    len(indicator_benchmarks)
)

print(
    "Indicators with benchmark data:",
    indicator_benchmarks["Indicator"].nunique()
)

display(
    indicator_benchmarks.head(20)
)

In [ ]:
# ==========================================
# BEST BENCHMARK BY INDICATOR
# ==========================================

best_indicator_benchmarks = (
    indicator_benchmarks
    .sort_values(
        ["Component", "Indicator", "Directional_Value"],
        ascending=[True, True, False]
    )
    .groupby(
        ["Component", "Indicator"],
        as_index=False
    )
    .first()
)

best_indicator_benchmarks = best_indicator_benchmarks[
    [
        "Component",
        "Indicator",
        "COUNTRY",
        "ISO3",
        "Value",
        "Directional_Value",
        "Sierra_Leone_Directional_Value",
        "Indicator_Gap"
    ]
].copy()

display(
    best_indicator_benchmarks
)

In [ ]:
# ==========================================
# FINAL INDICATOR-LEVEL MEL BENCHMARK SCORE
# ==========================================

final_indicator_benchmarks = best_indicator_benchmarks.copy()

# Absolute size of the improvement gap
final_indicator_benchmarks["Absolute_Gap"] = (
    final_indicator_benchmarks["Indicator_Gap"].abs()
)

# Rank benchmark countries within each indicator
final_indicator_benchmarks["Benchmark_Rank"] = (
    final_indicator_benchmarks
    .groupby(["Component", "Indicator"])["Directional_Value"]
    .rank(
        ascending=False,
        method="min"
    )
)

# Keep the strongest benchmark for each indicator
final_indicator_benchmarks = (
    final_indicator_benchmarks
    .sort_values(
        ["Component", "Indicator", "Benchmark_Rank"]
    )
    .reset_index(drop=True)
)

display(final_indicator_benchmarks)

## 21. Final Analytical Outputs


In [ ]:
# ==========================================
# FINAL MEL INTERVENTION TABLE
# ==========================================

mel_intervention_table = final_indicator_benchmarks[
    [
        "Component",
        "Indicator",
        "COUNTRY",
        "ISO3",
        "Value",
        "Sierra_Leone_Directional_Value",
        "Indicator_Gap",
        "Absolute_Gap"
    ]
].copy()

mel_intervention_table = (
    mel_intervention_table
    .sort_values(
        "Absolute_Gap",
        ascending=False
    )
    .reset_index(drop=True)
)

display(mel_intervention_table)

In [ ]:
output_path = OUTPUT_DIR / "SIERRA_LEONE_MEL_ANALYTICAL_OUTPUT.xlsx"

with pd.ExcelWriter(
    output_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    mel_intervention_table.to_excel(
        writer,
        sheet_name="Indicator_MEL_Benchmarks",
        index=False
    )

print("Saved:", output_path)

In [ ]:
# ==========================================
# FINAL ANALYSIS QA — OBJECT CHECK
# ==========================================

qa_objects = [
    "african_benchmarks",
    "sensitivity",
    "final_priority",
    "intervention_matrix_final",
    "hazard_priority_summary",
    "hazard_capacity_detail",
    "best_indicator_benchmarks",
]

for name in qa_objects:
    if name in globals():
        df = globals()[name]
        print(f"✓ {name:<30} {df.shape}")
    else:
        print(f"✗ MISSING: {name}")

In [ ]:
# ============================================================
# EXPORT ALL ANALYTICAL RESULTS FROM THE NOTEBOOK
# ============================================================

import pandas as pd
import os
import re

output_path = OUTPUT_DIR / "SIERRA_LEONE_COMPLETE_ANALYTICAL_RESULTS.xlsx"

# ------------------------------------------------------------
# 1. Identify DataFrames currently stored in the notebook
# ------------------------------------------------------------

dataframes = {}

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        # Ignore empty dataframes
        if obj.empty:
            continue

        # Ignore obvious temporary/internal objects
        if name.startswith("_"):
            continue

        dataframes[name] = obj.copy()


# ------------------------------------------------------------
# 2. Clean Excel sheet names
# ------------------------------------------------------------

def clean_sheet_name(name):

    # Excel does not allow these characters
    name = re.sub(r'[\[\]\:\*\?\/\\]', '_', str(name))

    # Maximum Excel sheet-name length = 31
    name = name[:31]

    if not name:
        name = "Sheet"

    return name


# ------------------------------------------------------------
# 3. Make sheet names unique
# ------------------------------------------------------------

used_names = set()
sheet_mapping = {}

for name in dataframes:

    base = clean_sheet_name(name)
    sheet = base
    counter = 1

    while sheet in used_names:

        suffix = f"_{counter}"
        sheet = base[:31-len(suffix)] + suffix
        counter += 1

    used_names.add(sheet)
    sheet_mapping[name] = sheet


# ------------------------------------------------------------
# 4. Export everything
# ------------------------------------------------------------

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    for name, df in dataframes.items():

        sheet_name = sheet_mapping[name]

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )


# ------------------------------------------------------------
# 5. Report what was exported
# ------------------------------------------------------------

print("=" * 60)
print("COMPLETE ANALYTICAL EXPORT")
print("=" * 60)

print(f"\nWorkbook:")
print(os.path.abspath(output_path))

print(f"\nDataFrames exported: {len(dataframes)}")

print("\nSheets:")
for name, df in dataframes.items():
    print(
        f"  {sheet_mapping[name]:<31} "
        f"{df.shape[0]:>5} rows × {df.shape[1]:>3} columns"
    )

print("\nEmpty DataFrames excluded:")
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and obj.empty:
        print(f"  - {name}")

print("\nDONE.")

## 24. Interpretation & Limitations

The outputs from this notebook are intended to support structured decision-making within the wider S-DRIF framework.

Key limitations to keep visible when presenting the analysis:

- The principal datasets are country-level and therefore cannot establish individual-level relationships.
- Cross-sectional associations should not be interpreted as causal effects.
- Missingness, indicator construction and differences in measurement across international datasets can affect comparability.
- Benchmark and clustering results are sensitive to variable selection, scaling and the comparison set; sensitivity analyses are therefore retained as part of the evidence base.
- Results should be interpreted alongside qualitative evidence and stakeholder consultation before policy or investment decisions are made.

## 25. Reproducibility

To reproduce the notebook:

1. Place the required source workbooks in the local `data/` directory.
2. Install the Python dependencies used in the notebook.
3. Run the notebook from the project root.
4. Review the QA and sensitivity-analysis sections before using the outputs.
